In [ ]:
# --- Environment setup (consolidated from the original scattered install cells) ---
# Heavyweight deps for the LLM / embedding cells below. See requirements.txt at the repo root.
%pip install -q -U transformers accelerate bitsandbytes mistral_common setfit

In [ ]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


# A1 & A2 vs Gold standard

In [ ]:
# ============================================================
# INTER-ANNOTATOR AGREEMENT vs GOLD STANDARD
#
# Gold standard:
#   - If A1 == A2 → gold = their common answer
#   - If A1 != A2 → gold = A3 (tie-breaker)
#
# Compute:
#   - Agreement A1 vs Gold
#   - Agreement A2 vs Gold
#   - Agreement A1 vs A2 (for reference)
#
# Metrics:
#   - Accuracy (% agreement)
#   - Cohen's Kappa
#   - F1, Precision, Recall per class
#   - Confusion matrices
# ============================================================

import os, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "inter_annotator_agreement")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("INTER-ANNOTATOR AGREEMENT ANALYSIS")
print("=" * 80)

df0 = pd.read_csv(PATH)
print(f"Total rows loaded: {len(df0)}")

# ============================================================
# EXTRACT ANNOTATIONS
# ============================================================
def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan

df0["A1"] = df0["eval_A1"].apply(extract_oui_non)
df0["A2"] = df0["eval_A2"].apply(extract_oui_non)
df0["A3"] = df0["eval_A3"].apply(extract_oui_non)

# ============================================================
# COMPUTE GOLD STANDARD
# ============================================================
def compute_gold(row):
    """
    Gold standard:
    - If A1 and A2 agree → their answer
    - If they disagree → A3's answer
    """
    a, t, s = row["A1"], row["A2"], row["A3"]

    # Both must have annotated
    if pd.isna(a) or pd.isna(t):
        return np.nan

    # Agreement
    if a == t:
        return a

    # Disagreement → A3 decides
    if pd.notna(s):
        return s

    return np.nan

df0["gold"] = df0.apply(compute_gold, axis=1)

# ============================================================
# FILTER VALID DATA
# ============================================================
# For A1 vs Gold: need A1 and gold
df_A1_gold = df0[df0["A1"].notna() & df0["gold"].notna()].copy()

# For A2 vs Gold: need A2 and gold
df_A2_gold = df0[df0["A2"].notna() & df0["gold"].notna()].copy()

# For A1 vs A2: need both
df_A1_A2 = df0[df0["A1"].notna() & df0["A2"].notna()].copy()

print(f"\nDataset sizes:")
print(f"  A1 vs Gold:   {len(df_A1_gold)} examples")
print(f"  A2 vs Gold:   {len(df_A2_gold)} examples")
print(f"  A1 vs A2: {len(df_A1_A2)} examples")

# ============================================================
# AGREEMENT STATISTICS
# ============================================================
# Agreement vs disagreement cases
n_accord = (df_A1_A2["A1"] == df_A1_A2["A2"]).sum()
n_desaccord = (df_A1_A2["A1"] != df_A1_A2["A2"]).sum()

print(f"\nAgreement/Disagreement between A1 and A2:")
print(f"  Agreement: {n_accord} ({100*n_accord/len(df_A1_A2):.1f}%)")
print(f"  Disagreement: {n_desaccord} ({100*n_desaccord/len(df_A1_A2):.1f}%)")

# Gold distribution
print(f"\nDistribution of the Gold Standard:")
gold_counts = df0["gold"].value_counts()
for val, count in gold_counts.items():
    print(f"  {val}: {count} ({100*count/df0['gold'].notna().sum():.1f}%)")

# ============================================================
# COMPUTE METRICS FUNCTION
# ============================================================
def compute_agreement_metrics(y_true, y_pred, name_true="Gold", name_pred="Annotator"):
    """Compute all agreement metrics between two annotation sets."""

    # Convert to binary for sklearn
    label_map = {"oui": 1, "non": 0}
    y_true_bin = np.array([label_map[y] for y in y_true])
    y_pred_bin = np.array([label_map[y] for y in y_pred])

    # Basic metrics
    accuracy = accuracy_score(y_true_bin, y_pred_bin)
    kappa = cohen_kappa_score(y_true_bin, y_pred_bin)

    # Per-class metrics
    f1_oui = f1_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true_bin, y_pred_bin, average="macro", zero_division=0)

    precision_oui = precision_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0)
    precision_non = precision_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0)

    recall_oui = recall_score(y_true_bin, y_pred_bin, pos_label=1, zero_division=0)
    recall_non = recall_score(y_true_bin, y_pred_bin, pos_label=0, zero_division=0)

    # Confusion matrix
    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])

    return {
        "n_samples": len(y_true),
        "accuracy": accuracy,
        "kappa": kappa,
        "f1_macro": f1_macro,
        "f1_oui": f1_oui,
        "f1_non": f1_non,
        "precision_oui": precision_oui,
        "precision_non": precision_non,
        "recall_oui": recall_oui,
        "recall_non": recall_non,
        "confusion_matrix": cm,
    }

# ============================================================
# COMPUTE ALL AGREEMENTS
# ============================================================
print("\n" + "=" * 80)
print("AGREEMENT METRICS")
print("=" * 80)

results = {}

# 1. A1 vs Gold
metrics_A1 = compute_agreement_metrics(
    df_A1_gold["gold"].values,
    df_A1_gold["A1"].values,
    "Gold", "A1"
)
results["A1 vs Gold"] = metrics_A1

# 2. A2 vs Gold
metrics_A2 = compute_agreement_metrics(
    df_A2_gold["gold"].values,
    df_A2_gold["A2"].values,
    "Gold", "A2"
)
results["A2 vs Gold"] = metrics_A2

# 3. A1 vs A2 (for reference)
metrics_A1_A2 = compute_agreement_metrics(
    df_A1_A2["A2"].values,  # Take A2 as the "reference"
    df_A1_A2["A1"].values,
    "A2", "A1"
)
results["A1 vs A2"] = metrics_A1_A2

# ============================================================
# PRINT RESULTS
# ============================================================
for name, m in results.items():
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(f"  N samples:     {m['n_samples']}")
    print(f"  Accuracy:      {m['accuracy']:.4f} ({100*m['accuracy']:.2f}%)")
    print(f"  Cohen's Kappa: {m['kappa']:.4f}")
    print(f"  F1 Macro:      {m['f1_macro']:.4f}")
    print(f"  ")
    print(f"  Per-class metrics:")
    print(f"    OUI:  P={m['precision_oui']:.3f}  R={m['recall_oui']:.3f}  F1={m['f1_oui']:.3f}")
    print(f"    NON:  P={m['precision_non']:.3f}  R={m['recall_non']:.3f}  F1={m['f1_non']:.3f}")
    print(f"  ")
    print(f"  Confusion Matrix (rows=Gold/Ref, cols=Annotator):")
    print(f"              Pred NON  Pred OUI")
    print(f"    True NON     {m['confusion_matrix'][0,0]:5d}     {m['confusion_matrix'][0,1]:5d}")
    print(f"    True OUI     {m['confusion_matrix'][1,0]:5d}     {m['confusion_matrix'][1,1]:5d}")

# ============================================================
# ANALYSIS: WHO AGREES MORE WITH GOLD?
# ============================================================
print("\n" + "=" * 80)
print("COMPARISON: WHO AGREES MORE WITH GOLD?")
print("=" * 80)

print(f"\n{'Metric':<20} {'A1':>12} {'A2':>12} {'Diff (A1-A2)':>12}")
print("─" * 60)

metrics_to_compare = [
    ("Accuracy", "accuracy"),
    ("Cohen's Kappa", "kappa"),
    ("F1 Macro", "f1_macro"),
    ("F1 OUI", "f1_oui"),
    ("F1 NON", "f1_non"),
]

for name, key in metrics_to_compare:
    val_a = results["A1 vs Gold"][key]
    val_t = results["A2 vs Gold"][key]
    diff = val_a - val_t
    winner = "←" if diff > 0.001 else ("→" if diff < -0.001 else "=")
    print(f"{name:<20} {val_a:>12.4f} {val_t:>12.4f} {diff:>+12.4f} {winner}")

# ============================================================
# DETAILED DISAGREEMENT ANALYSIS
# ============================================================
print("\n" + "=" * 80)
print("DISAGREEMENT ANALYSIS")
print("=" * 80)

# Cases where A1 != Gold
df_A1_wrong = df_A1_gold[df_A1_gold["A1"] != df_A1_gold["gold"]].copy()
print(f"\nA1 disagrees with Gold: {len(df_A1_wrong)} cases")

# Breakdown
A1_oui_gold_non = ((df_A1_wrong["A1"] == "oui") & (df_A1_wrong["gold"] == "non")).sum()
A1_non_gold_oui = ((df_A1_wrong["A1"] == "non") & (df_A1_wrong["gold"] == "oui")).sum()
print(f"  A1=OUI, Gold=NON: {A1_oui_gold_non}")
print(f"  A1=NON, Gold=OUI: {A1_non_gold_oui}")

# Cases where A2 != Gold
df_A2_wrong = df_A2_gold[df_A2_gold["A2"] != df_A2_gold["gold"]].copy()
print(f"\nA2 disagrees with Gold: {len(df_A2_wrong)} cases")

A2_oui_gold_non = ((df_A2_wrong["A2"] == "oui") & (df_A2_wrong["gold"] == "non")).sum()
A2_non_gold_oui = ((df_A2_wrong["A2"] == "non") & (df_A2_wrong["gold"] == "oui")).sum()
print(f"  A2=OUI, Gold=NON: {A2_oui_gold_non}")
print(f"  A2=NON, Gold=OUI: {A2_non_gold_oui}")

# ============================================================
# WHEN DISAGREEMENT: WHO DOES A3 AGREE WITH?
# ============================================================
print("\n" + "=" * 80)
print("TIE-BREAKER ANALYSIS: WHO DOES A3 SIDE WITH?")
print("=" * 80)

df_desaccord = df_A1_A2[df_A1_A2["A1"] != df_A1_A2["A2"]].copy()
df_desaccord = df_desaccord[df_desaccord["A3"].notna()].copy()

A3_with_A1 = (df_desaccord["A3"] == df_desaccord["A1"]).sum()
A3_with_A2 = (df_desaccord["A3"] == df_desaccord["A2"]).sum()

print(f"\nWhen A1 and A2 disagree (N={len(df_desaccord)}):")
print(f"  A3 agrees with A1: {A3_with_A1} ({100*A3_with_A1/len(df_desaccord):.1f}%)")
print(f"  A3 agrees with A2: {A3_with_A2} ({100*A3_with_A2/len(df_desaccord):.1f}%)")

# Breakdown by disagreement type
df_A1_oui_A2_non = df_desaccord[(df_desaccord["A1"] == "oui") & (df_desaccord["A2"] == "non")]
df_A1_non_A2_oui = df_desaccord[(df_desaccord["A1"] == "non") & (df_desaccord["A2"] == "oui")]

print(f"\nDetail by disagreement type:")
print(f"  A1=OUI, A2=NON: {len(df_A1_oui_A2_non)} cases")
if len(df_A1_oui_A2_non) > 0:
    sol_oui = (df_A1_oui_A2_non["A3"] == "oui").sum()
    sol_non = (df_A1_oui_A2_non["A3"] == "non").sum()
    print(f"    → A3 says OUI (with A1): {sol_oui} ({100*sol_oui/len(df_A1_oui_A2_non):.1f}%)")
    print(f"    → A3 says NON (with A2): {sol_non} ({100*sol_non/len(df_A1_oui_A2_non):.1f}%)")

print(f"\n  A1=NON, A2=OUI: {len(df_A1_non_A2_oui)} cases")
if len(df_A1_non_A2_oui) > 0:
    sol_oui = (df_A1_non_A2_oui["A3"] == "oui").sum()
    sol_non = (df_A1_non_A2_oui["A3"] == "non").sum()
    print(f"    → A3 says OUI (with A2): {sol_oui} ({100*sol_oui/len(df_A1_non_A2_oui):.1f}%)")
    print(f"    → A3 says NON (with A1): {sol_non} ({100*sol_non/len(df_A1_non_A2_oui):.1f}%)")

# ============================================================
# SAVE RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Summary DataFrame
summary_data = []
for name, m in results.items():
    summary_data.append({
        "Comparison": name,
        "N_samples": m["n_samples"],
        "Accuracy": m["accuracy"],
        "Cohen_Kappa": m["kappa"],
        "F1_Macro": m["f1_macro"],
        "F1_OUI": m["f1_oui"],
        "F1_NON": m["f1_non"],
        "Precision_OUI": m["precision_oui"],
        "Precision_NON": m["precision_non"],
        "Recall_OUI": m["recall_oui"],
        "Recall_NON": m["recall_non"],
        "TN": m["confusion_matrix"][0, 0],
        "FP": m["confusion_matrix"][0, 1],
        "FN": m["confusion_matrix"][1, 0],
        "TP": m["confusion_matrix"][1, 1],
    })

df_summary = pd.DataFrame(summary_data)

# Save Excel
out_xlsx = os.path.join(OUTPUT_PATH, "inter_annotator_agreement.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_summary.to_excel(writer, index=False, sheet_name="Summary")

    # Disagreement cases
    df_A1_wrong[["decision_id", "chunk_id", "A1", "A2", "A3", "gold"]].to_excel(
        writer, index=False, sheet_name="A1_vs_Gold_Errors"
    )
    df_A2_wrong[["decision_id", "chunk_id", "A1", "A2", "A3", "gold"]].to_excel(
        writer, index=False, sheet_name="A2_vs_Gold_Errors"
    )

    # All disagreements between A and T
    df_desaccord[["decision_id", "chunk_id", "A1", "A2", "A3", "gold"]].to_excel(
        writer, index=False, sheet_name="A1_A2_Disagreements"
    )

print(f"✅ Saved: {out_xlsx}")

# Save CSV
out_csv = os.path.join(OUTPUT_PATH, "inter_annotator_agreement_summary.csv")
df_summary.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# ============================================================
# INTERPRETATION GUIDE
# ============================================================
print("\n" + "=" * 80)
print("INTERPRETATION GUIDE")
print("=" * 80)

print("""
Cohen's Kappa interpretation:
  κ < 0.00    Poor agreement
  0.00-0.20   Slight agreement
  0.21-0.40   Fair agreement
  0.41-0.60   Moderate agreement
  0.61-0.80   Substantial agreement
  0.81-1.00   Almost perfect agreement

Note: the gold standard includes:
  - Cases where A1 and A2 agree (their common answer)
  - Cases where they disagree (A3's answer as tie-breaker)

So when comparing A1 vs Gold:
  - On agreement cases A1-A2: A1 == Gold by definition (100% accuracy)
  - On disagreement cases: A1 == Gold iff A3 agrees with A1

Same logic for A2 vs Gold.
""")

print(f"\n✅ DONE — Results saved in: {OUTPUT_PATH}")

# Saul & Lama

# A1

In [ ]:
# ============================================================
# 2-STEP PIPELINE (SAUL=LR, LLaMA=MLP1)
# A) Separately:
#    - SAUL: embeddings -> LR OOF -> thr=0.5 + best thr (MCC)
#    - LLaMA: embeddings -> MLP1 OOF -> thr=0.5 + best thr (MCC)
#    - 3 text configs x pooling/layer variants
#    -> keep BEST variant per model
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A1"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"outputs_saul_lr_llama_mlp1_3cfg_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"

SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16

# MLP1 (1 hidden layer)
MLP_HIDDEN = (256,)
MLP_ALPHA = 1e-3
MLP_LR_INIT = 5e-4
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)  # step ~0.005

# Ensemble (after selecting best per model)
W_GRID = np.linspace(0.0, 1.0, 41)       # step 0.025

# -------------------------
# Helpers: grouped folds
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

# -------------------------
# Metrics
# -------------------------
def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,   # (a.k.a. BF1)
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba — LR / MLP1
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="lr"):
    """
    Return OOF proba (p(oui)) for each row of df_cv.
    - model_kind="lr": LogisticRegression
    - model_kind="mlp1": MLPClassifier (1 hidden layer) + early stopping
    """
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]

        if model_kind == "lr":
            clf = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                C=1.0,
                random_state=seed
            )
        elif model_kind == "mlp1":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP_HIDDEN,
                alpha=MLP_ALPHA,
                learning_rate_init=MLP_LR_INIT,
                max_iter=MLP_MAX_ITER,
                early_stopping=MLP_EARLY_STOPPING,
                n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                random_state=seed
            )
        else:
            raise ValueError("model_kind must be 'lr' or 'mlp1'")

        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings encoder (pooling/layers) + cache
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    # "stable-ish" hash without blowing up runtime: take n examples + total len
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        if strategy.startswith("concat_last_k:"):
            k = int(strategy.split(":")[1])
            return torch.cat(hidden_states[-k:], dim=-1)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:
                mask = attn.unsqueeze(-1).to(x.dtype)
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)

    del model, tok
    _clear_cuda()
    return X

# -------------------------
# 3 text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD + PREP DATA (labels)
# ============================================================
print("="*80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("="*80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

# Keep only rows with a valid label
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# STEP A — separate evaluation per model
# ============================================================
def run_model_search(short_name, hf_model, batch_size, classifier_kind):
    """
    classifier_kind:
      - "lr"  for SAUL
      - "mlp1" for LLaMA
    """
    rows = []
    best = None

    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{short_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print("\n" + "="*90)
            print(f"[{short_name}] {run_id}  (clf={classifier_kind}) | GOLD={GOLD_ANNOTATOR}")
            print("="*90)

            X = encode_transformer_pool(
                texts=texts,
                model_name=hf_model,
                batch_size=batch_size,
                max_len=LLM_MAX_LEN,
                hf_token=HF_TOKEN,
                use_bf16=USE_BF16,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )
            print(f"✓ Embeddings: {X.shape}")

            proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=classifier_kind)

            # threshold 0.5
            pred_05 = (proba_oof >= 0.5).astype(int)
            met_05 = compute_metrics_from_pred(y_true, pred_05)

            # best MCC threshold
            best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
            pred_best = (proba_oof >= best_thr).astype(int)
            met_best = compute_metrics_from_pred(y_true, pred_best)

            row = {
                "Model": short_name,
                "HF_model": hf_model,
                "Classifier": classifier_kind,
                "Gold_Annotator": GOLD_ANNOTATOR,
                "TextCfg": cfg_id,
                "pooling": pooling,
                "layer_strategy": layer_strategy,

                "Thr(best)": best_thr,
                "MCC(best)": met_best["MCC"],
                "Acc(best)": met_best["Accuracy"],
                "BAcc(best)": met_best["Balanced Acc"],
                "BF1(best)": met_best["Balanced F1"],
                "F1-oui(best)": met_best["F1-oui"],
                "F1-non(best)": met_best["F1-non"],

                "MCC@0.5": met_05["MCC"],
                "Acc@0.5": met_05["Accuracy"],
                "BAcc@0.5": met_05["Balanced Acc"],
                "BF1@0.5": met_05["Balanced F1"],
                "F1-oui@0.5": met_05["F1-oui"],
                "F1-non@0.5": met_05["F1-non"],

                "ΔMCC": met_best["MCC"] - met_05["MCC"],
                "proba_oof_path": None,  # filled in after saving
            }

            # save proba_oof
            proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id.replace('|','__')}.npy")
            np.save(proba_path, proba_oof)
            row["proba_oof_path"] = proba_path

            rows.append(row)

            if (best is None) or (row["MCC(best)"] > best["MCC(best)"]):
                best = row

            print(f"@0.5  MCC={row['MCC@0.5']:.4f} | best thr={row['Thr(best)']:.3f} MCC={row['MCC(best)']:.4f} | Δ={row['ΔMCC']:.4f}")

    df_rows = pd.DataFrame(rows).sort_values(["MCC(best)", "BAcc(best)", "BF1(best)"], ascending=False).reset_index(drop=True)
    return df_rows, best

# --- SAUL (LR)
df_saul, best_saul = run_model_search(
    short_name="SAUL-7B",
    hf_model=SAUL_MODEL,
    batch_size=SAUL_BATCH_SIZE,
    classifier_kind="lr"
)

# --- LLaMA (MLP1)
df_llama, best_llama = run_model_search(
    short_name="LLaMA-3.1-8B",
    hf_model=LLAMA_MODEL,
    batch_size=LLAMA_BATCH_SIZE,
    classifier_kind="mlp1"
)

# save recap
out_a = os.path.join(OUTPUT_PATH, f"STEP_A__saul_lr__llama_mlp1__all_variants__gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_a) as w:
    df_saul.to_excel(w, index=False, sheet_name="SAUL_all")
    df_llama.to_excel(w, index=False, sheet_name="LLaMA_all")
print(f"\n✅ STEP A saved: {out_a}")

best_df = pd.DataFrame([best_saul, best_llama])
out_best = os.path.join(OUTPUT_PATH, f"STEP_A__BEST_PER_MODEL__gold_{GOLD_ANNOTATOR}.xlsx")
best_df.to_excel(out_best, index=False)
print(f"✅ BEST per model: {out_best}")

print("\n" + "="*100)
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("="*100)
print("BEST SAUL:")
print(pd.Series(best_saul)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("\nBEST LLaMA:")
print(pd.Series(best_llama)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("="*100)

# A2

In [ ]:
# ============================================================
# 2-STEP PIPELINE (SAUL=LR, LLaMA=MLP1)
# A) Separately:
#    - SAUL: embeddings -> LR OOF -> thr=0.5 + best thr (MCC)
#    - LLaMA: embeddings -> MLP1 OOF -> thr=0.5 + best thr (MCC)
#    - 3 text configs x pooling/layer variants
#    -> keep BEST variant per model
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A2"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"outputs_saul_lr_llama_mlp1_3cfg_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"

SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16

# MLP1 (1 hidden layer)
MLP_HIDDEN = (256,)
MLP_ALPHA = 1e-3
MLP_LR_INIT = 5e-4
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)  # step ~0.005

# Ensemble (after selecting best per model)
W_GRID = np.linspace(0.0, 1.0, 41)       # step 0.025

# -------------------------
# Helpers: grouped folds
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

# -------------------------
# Metrics
# -------------------------
def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,   # (a.k.a. BF1)
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba — LR / MLP1
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="lr"):
    """
    Return OOF proba (p(oui)) for each row of df_cv.
    - model_kind="lr": LogisticRegression
    - model_kind="mlp1": MLPClassifier (1 hidden layer) + early stopping
    """
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]

        if model_kind == "lr":
            clf = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                C=1.0,
                random_state=seed
            )
        elif model_kind == "mlp1":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP_HIDDEN,
                alpha=MLP_ALPHA,
                learning_rate_init=MLP_LR_INIT,
                max_iter=MLP_MAX_ITER,
                early_stopping=MLP_EARLY_STOPPING,
                n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                random_state=seed
            )
        else:
            raise ValueError("model_kind must be 'lr' or 'mlp1'")

        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings encoder (pooling/layers) + cache
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    # "stable-ish" hash without blowing up runtime: take n examples + total len
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        if strategy.startswith("concat_last_k:"):
            k = int(strategy.split(":")[1])
            return torch.cat(hidden_states[-k:], dim=-1)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:
                mask = attn.unsqueeze(-1).to(x.dtype)
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)

    del model, tok
    _clear_cuda()
    return X

# -------------------------
# 3 text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD + PREP DATA (labels)
# ============================================================
print("="*80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("="*80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

# Keep only rows with a valid label
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# STEP A — separate evaluation per model
# ============================================================
def run_model_search(short_name, hf_model, batch_size, classifier_kind):
    """
    classifier_kind:
      - "lr"  for SAUL
      - "mlp1" for LLaMA
    """
    rows = []
    best = None

    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{short_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print("\n" + "="*90)
            print(f"[{short_name}] {run_id}  (clf={classifier_kind}) | GOLD={GOLD_ANNOTATOR}")
            print("="*90)

            X = encode_transformer_pool(
                texts=texts,
                model_name=hf_model,
                batch_size=batch_size,
                max_len=LLM_MAX_LEN,
                hf_token=HF_TOKEN,
                use_bf16=USE_BF16,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )
            print(f"✓ Embeddings: {X.shape}")

            proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=classifier_kind)

            # threshold 0.5
            pred_05 = (proba_oof >= 0.5).astype(int)
            met_05 = compute_metrics_from_pred(y_true, pred_05)

            # best MCC threshold
            best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
            pred_best = (proba_oof >= best_thr).astype(int)
            met_best = compute_metrics_from_pred(y_true, pred_best)

            row = {
                "Model": short_name,
                "HF_model": hf_model,
                "Classifier": classifier_kind,
                "Gold_Annotator": GOLD_ANNOTATOR,
                "TextCfg": cfg_id,
                "pooling": pooling,
                "layer_strategy": layer_strategy,

                "Thr(best)": best_thr,
                "MCC(best)": met_best["MCC"],
                "Acc(best)": met_best["Accuracy"],
                "BAcc(best)": met_best["Balanced Acc"],
                "BF1(best)": met_best["Balanced F1"],
                "F1-oui(best)": met_best["F1-oui"],
                "F1-non(best)": met_best["F1-non"],

                "MCC@0.5": met_05["MCC"],
                "Acc@0.5": met_05["Accuracy"],
                "BAcc@0.5": met_05["Balanced Acc"],
                "BF1@0.5": met_05["Balanced F1"],
                "F1-oui@0.5": met_05["F1-oui"],
                "F1-non@0.5": met_05["F1-non"],

                "ΔMCC": met_best["MCC"] - met_05["MCC"],
                "proba_oof_path": None,  # filled in after saving
            }

            # save proba_oof
            proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id.replace('|','__')}.npy")
            np.save(proba_path, proba_oof)
            row["proba_oof_path"] = proba_path

            rows.append(row)

            if (best is None) or (row["MCC(best)"] > best["MCC(best)"]):
                best = row

            print(f"@0.5  MCC={row['MCC@0.5']:.4f} | best thr={row['Thr(best)']:.3f} MCC={row['MCC(best)']:.4f} | Δ={row['ΔMCC']:.4f}")

    df_rows = pd.DataFrame(rows).sort_values(["MCC(best)", "BAcc(best)", "BF1(best)"], ascending=False).reset_index(drop=True)
    return df_rows, best

# --- SAUL (LR)
df_saul, best_saul = run_model_search(
    short_name="SAUL-7B",
    hf_model=SAUL_MODEL,
    batch_size=SAUL_BATCH_SIZE,
    classifier_kind="lr"
)

# --- LLaMA (MLP1)
df_llama, best_llama = run_model_search(
    short_name="LLaMA-3.1-8B",
    hf_model=LLAMA_MODEL,
    batch_size=LLAMA_BATCH_SIZE,
    classifier_kind="mlp1"
)

# save recap
out_a = os.path.join(OUTPUT_PATH, f"STEP_A__saul_lr__llama_mlp1__all_variants__gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_a) as w:
    df_saul.to_excel(w, index=False, sheet_name="SAUL_all")
    df_llama.to_excel(w, index=False, sheet_name="LLaMA_all")
print(f"\n✅ STEP A saved: {out_a}")

best_df = pd.DataFrame([best_saul, best_llama])
out_best = os.path.join(OUTPUT_PATH, f"STEP_A__BEST_PER_MODEL__gold_{GOLD_ANNOTATOR}.xlsx")
best_df.to_excel(out_best, index=False)
print(f"✅ BEST per model: {out_best}")

print("\n" + "="*100)
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("="*100)
print("BEST SAUL:")
print(pd.Series(best_saul)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("\nBEST LLaMA:")
print(pd.Series(best_llama)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("="*100)

# Other A1

In [ ]:
# ============================================================
# GRID SEARCH — TRANSFORMERS & SENTENCE TRANSFORMERS
# PARALLELIZED VERSION + SAVE BEST EMBEDDINGS/OOF
#
# Parallelization:
#   - Embeddings: sequential (GPU-bound, 1 GPU)
#   - Classifiers: parallel with joblib (CPU-bound)
#   - Threshold search: vectorized with numpy
#
# Models tested (ALL with pooling/layer configs):
#   - CamemBERT, CamemBERTav2, JuriBERT-base
#   - ST-MiniLM, ST-MPNet (treated as transformers for pooling/layer)
#
# Grid: 3 text configs × 4 pooling/layer × 3 classifiers = 36 per model
# Total: 5 models × 36 = 180 combinations
#
# NEW: save embeddings + OOF for the best config per model
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc, hashlib, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

from joblib import Parallel, delayed
import torch

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A1"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"oof_proba_final_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Folder for the best results
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Parallelization settings
N_JOBS = -1  # -1 = all CPU cores

# Transformer settings
MAX_LEN = 512
BATCH_SIZE = 16

# MLP configs
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300

MLP2_HIDDEN = (256, 64)
MLP2_ALPHA = 2e-3
MLP2_LR_INIT = 3e-4
MLP2_MAX_ITER = 400

MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# Threshold grid (vectorized)
THR_GRID = np.linspace(0.05, 0.95, 181)

# ============================================================
# ALL MODELS — Treated uniformly as transformers
# ============================================================
ALL_MODELS = {
    "CamemBERT": {
        "hf_model": "camembert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "CamemBERTav2": {
        "hf_model": "almanach/camembertav2-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "JuriBERT-base": {
        "hf_model": "dascim/juribert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    # SentenceTransformers — access the underlying model
    "ST-MiniLM": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
    "ST-MPNet": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
}

# Grid search configs
TEXT_CONFIGS = ["cfg1", "cfg2", "cfg3"]
CLASSIFIERS = ["LR", "MLP1", "MLP2"]

POOLING_LAYER_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
    {"pooling": "cls", "layer_strategy": "last"},
]


# ============================================================
# HELPERS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()


# ============================================================
# TEXT CONFIG BUILDERS
# ============================================================
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")


# ============================================================
# CLASSIFIER FACTORIES
# ============================================================
def make_classifier(clf_type, seed):
    if clf_type == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
    elif clf_type == "MLP1":
        return MLPClassifier(
            hidden_layer_sizes=MLP1_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP1_ALPHA,
            learning_rate_init=MLP1_LR_INIT,
            max_iter=MLP1_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    elif clf_type == "MLP2":
        return MLPClassifier(
            hidden_layer_sizes=MLP2_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP2_ALPHA,
            learning_rate_init=MLP2_LR_INIT,
            max_iter=MLP2_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown classifier type: {clf_type}")


# ============================================================
# METRICS — VECTORIZED THRESHOLD SEARCH
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


def best_threshold_vectorized(y_true, proba, thresholds=THR_GRID):
    """Vectorized threshold search — much faster than loop."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    # Broadcast: (n_samples,) vs (n_thresholds,) -> (n_thresholds, n_samples)
    preds = (proba[np.newaxis, :] >= thresholds[:, np.newaxis]).astype(int)

    # Compute MCC for each threshold
    tp = ((preds == 1) & (y_true == 1)).sum(axis=1)
    tn = ((preds == 0) & (y_true == 0)).sum(axis=1)
    fp = ((preds == 1) & (y_true == 0)).sum(axis=1)
    fn = ((preds == 0) & (y_true == 1)).sum(axis=1)

    # MCC formula
    num = (tp * tn - fp * fn).astype(float)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn).astype(float))
    denom = np.where(denom == 0, 1, denom)  # Avoid division by zero
    mcc = num / denom

    best_idx = np.argmax(mcc)
    return float(thresholds[best_idx]), float(mcc[best_idx])


# ============================================================
# GROUPED FOLDS
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# UNIFIED ENCODER — Works for both transformers and SentenceTransformers
# ============================================================
def encode_with_pooling_layer(texts, model_name, model_type, batch_size, max_len, pooling, layer_strategy, desc):
    """
    Unified encoder that handles both regular transformers and SentenceTransformers.
    For SentenceTransformers, we access the underlying transformer model directly.

    Returns: (embeddings, cache_path) - embeddings array and path where they're cached
    """
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    # Check cache
    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        print(f"  [CACHE HIT] {os.path.basename(cache_path)}")
        return np.load(cache_path), cache_path

    print(f"  [CACHE MISS] Computing embeddings...")

    # Load model
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            k = min(k, len(hidden_states))
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
            batch = texts[i:i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            out = model(**enc, output_hidden_states=True)
            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:  # mean pooling
                mask = attn.unsqueeze(-1).float()
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.cpu().numpy())

            del enc, out, hs, attn, x, emb
            if i % (batch_size * 10) == 0:
                _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)
    print(f"  [SAVED] {os.path.basename(cache_path)} — shape: {X.shape}")

    del model, tok
    _clear_cuda()
    return X, cache_path


# ============================================================
# PARALLEL OOF EVALUATION
# ============================================================
def train_fold(fold, X, y, folds, clf_type, seed):
    """Train on one fold and return OOF predictions for test indices."""
    tr = folds != fold
    te = folds == fold

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[tr])
    Xte = scaler.transform(X[te])
    ytr = y[tr]

    clf = make_classifier(clf_type, seed)  # constant seed (as in the reference script)
    clf.fit(Xtr, ytr)
    proba_te = clf.predict_proba(Xte)[:, 1]

    return np.where(te)[0], proba_te


def oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=N_JOBS):
    """Parallel OOF computation across folds."""
    proba_oof = np.full(len(y), np.nan, dtype=float)

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(train_fold)(fold, X, y, folds, clf_type, seed)
        for fold in range(n_splits)
    )

    for te_idx, proba_te in results:
        proba_oof[te_idx] = proba_te

    assert np.isfinite(proba_oof).all(), "NaN in OOF predictions!"
    return proba_oof


def evaluate_single_classifier(X, y, folds, n_splits, clf_type, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate one classifier configuration — called in parallel."""
    proba_oof = oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=1)

    # Metrics at 0.5
    pred_05 = (proba_oof >= 0.5).astype(int)
    met_05 = compute_metrics(y, pred_05)

    # Best threshold (vectorized)
    best_thr, best_mcc = best_threshold_vectorized(y, proba_oof)
    pred_best = (proba_oof >= best_thr).astype(int)
    met_best = compute_metrics(y, pred_best)

    return {
        "clf_type": clf_type,
        "proba_oof": proba_oof,
        "best_thr": best_thr,
        "met_05": met_05,
        "met_best": met_best,
    }


def evaluate_all_classifiers_parallel(X, y, folds, n_splits, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate all classifiers in parallel for a given embedding."""
    results = Parallel(n_jobs=len(CLASSIFIERS), backend="loky")(
        delayed(evaluate_single_classifier)(
            X, y, folds, n_splits, clf_type, seed,
            model_name, cfg_id, pooling, layer_strategy
        )
        for clf_type in CLASSIFIERS
    )
    return {r["clf_type"]: r for r in results}


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

# Keep only rows with a valid label
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} examples | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values
folds = df_cv["fold"].values


# ============================================================
# PRE-COMPUTE ALL TEXT CONFIGS
# ============================================================
print("\n" + "=" * 80)
print("PRE-COMPUTING TEXT CONFIGS")
print("=" * 80)

texts_cache = {}
for cfg_id in TEXT_CONFIGS:
    texts_cache[cfg_id] = make_text_inputs(df_cv, cfg_id)
    print(f"  {cfg_id}: {len(texts_cache[cfg_id])} texts")


# ============================================================
# MAIN GRID SEARCH — PARALLELIZED + SAVE BEST
# ============================================================
print("\n" + "=" * 80)
print(f"GRID SEARCH — {len(ALL_MODELS)} MODELS × {len(TEXT_CONFIGS)} CONFIGS × {len(POOLING_LAYER_CONFIGS)} POOLING × {len(CLASSIFIERS)} CLASSIFIERS")
print(f"Total combinations: {len(ALL_MODELS) * len(TEXT_CONFIGS) * len(POOLING_LAYER_CONFIGS) * len(CLASSIFIERS)}")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

all_results = []

# To track the best per model (with embeddings path)
best_per_model_tracking = {}

for model_name, model_cfg in ALL_MODELS.items():
    hf_model = model_cfg["hf_model"]
    batch_size = model_cfg["batch_size"]
    model_type = model_cfg["type"]

    print(f"\n{'='*80}")
    print(f"MODEL: {model_name} ({hf_model}) | GOLD={GOLD_ANNOTATOR}")
    print(f"{'='*80}")

    best_for_model = {"MCC(best)": -1e9}

    for cfg_id in TEXT_CONFIGS:
        texts = texts_cache[cfg_id]

        for pcfg in POOLING_LAYER_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{model_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print(f"\n--- {run_id} ---")

            # Get embeddings (GPU, sequential) - now also returns the cache_path
            X, emb_cache_path = encode_with_pooling_layer(
                texts=texts,
                model_name=hf_model,
                model_type=model_type,
                batch_size=batch_size,
                max_len=MAX_LEN,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )

            # Evaluate all classifiers in parallel (CPU)
            print(f"  Evaluating {len(CLASSIFIERS)} classifiers in parallel...")
            clf_results = evaluate_all_classifiers_parallel(
                X, y_true, folds, N_SPLITS, SEED,
                model_name, cfg_id, pooling, layer_strategy
            )

            # Collect results
            for clf_type, res in clf_results.items():
                met_05 = res["met_05"]
                met_best = res["met_best"]
                best_thr = res["best_thr"]
                proba_oof = res["proba_oof"]

                row = {
                    "Model": model_name,
                    "HF_model": hf_model,
                    "Gold_Annotator": GOLD_ANNOTATOR,
                    "TextCfg": cfg_id,
                    "pooling": pooling,
                    "layer_strategy": layer_strategy,
                    "Classifier": clf_type,

                    "Thr(best)": best_thr,
                    "MCC(best)": met_best["MCC"],
                    "Acc(best)": met_best["Accuracy"],
                    "BAcc(best)": met_best["Balanced Acc"],
                    "BF1(best)": met_best["Balanced F1"],
                    "F1-oui(best)": met_best["F1-oui"],
                    "F1-non(best)": met_best["F1-non"],

                    "MCC@0.5": met_05["MCC"],
                    "Acc@0.5": met_05["Accuracy"],
                    "BAcc@0.5": met_05["Balanced Acc"],
                    "ΔMCC": met_best["MCC"] - met_05["MCC"],
                }

                # Save ALL proba (comme avant)
                proba_fname = f"proba_oof_{model_name}_{cfg_id}_{pooling}_{layer_strategy.replace(':', '_')}_{clf_type}.npy"
                proba_path = os.path.join(OUTPUT_PATH, proba_fname)
                np.save(proba_path, proba_oof)
                row["proba_oof_path"] = proba_path
                row["emb_cache_path"] = emb_cache_path  # keep track of the embeddings path

                all_results.append(row)

                # Track the best for this model
                if row["MCC(best)"] > best_for_model.get("MCC(best)", -1e9):
                    best_for_model = row.copy()
                    best_for_model["_proba_oof"] = proba_oof  # keep the data in memory
                    best_for_model["_emb_cache_path"] = emb_cache_path

                print(f"    {clf_type}: MCC@0.5={met_05['MCC']:.4f} | thr={best_thr:.3f} MCC={met_best['MCC']:.4f}")

    # Save the best for this model
    best_per_model_tracking[model_name] = best_for_model

    print(f"\n✓ BEST for {model_name}:")
    print(f"  {best_for_model['TextCfg']}, {best_for_model['pooling']}, {best_for_model['layer_strategy']}, {best_for_model['Classifier']}")
    print(f"  MCC(best): {best_for_model['MCC(best)']:.4f} @ thr={best_for_model['Thr(best)']:.3f}")


# ============================================================
# SAVE BEST EMBEDDINGS & OOF PER MODEL
# ============================================================
print("\n" + "=" * 80)
print("SAVING BEST EMBEDDINGS & OOF PER MODEL")
print("=" * 80)

for model_name, best_info in best_per_model_tracking.items():
    print(f"\n{model_name}:")

    # Standardized file name
    safe_layer = best_info['layer_strategy'].replace(':', '_')
    base_name = f"{model_name}_BEST_{best_info['TextCfg']}_{best_info['pooling']}_{safe_layer}_{best_info['Classifier']}"

    # 1. Save OOF proba
    oof_path = os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy")
    np.save(oof_path, best_info["_proba_oof"])
    print(f"  ✓ OOF proba: {oof_path}")

    # 2. Copy/save embeddings
    emb_src = best_info["_emb_cache_path"]
    emb_dst = os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy")

    # Load and save (or copy)
    if os.path.exists(emb_src):
        import shutil
        shutil.copy(emb_src, emb_dst)
        print(f"  ✓ Embeddings: {emb_dst}")
    else:
        print(f"  ⚠ Embeddings source not found: {emb_src}")

    # 3. Save config JSON
    config = {
        "model_name": model_name,
        "hf_model": best_info["HF_model"],
        "gold_annotator": GOLD_ANNOTATOR,
        "text_cfg": best_info["TextCfg"],
        "pooling": best_info["pooling"],
        "layer_strategy": best_info["layer_strategy"],
        "classifier": best_info["Classifier"],
        "threshold": best_info["Thr(best)"],
        "MCC_best": best_info["MCC(best)"],
        "MCC_05": best_info["MCC@0.5"],
        "oof_path": oof_path,
        "emb_path": emb_dst,
    }
    config_path = os.path.join(BEST_OUTPUT_PATH, f"config_{base_name}.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  ✓ Config: {config_path}")


# ============================================================
# SAVE ALL RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING ALL RESULTS")
print("=" * 80)

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("MCC(best)", ascending=False).reset_index(drop=True)

# Excel with sheets per model
out_xlsx = os.path.join(OUTPUT_PATH, f"GRID_SEARCH_ALL_RESULTS_gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_all.to_excel(writer, index=False, sheet_name="All_Results")
    for model_name in df_all["Model"].unique():
        df_model = df_all[df_all["Model"] == model_name].copy()
        df_model.to_excel(writer, index=False, sheet_name=model_name[:31])
print(f"✅ Saved: {out_xlsx}")

# CSV
out_csv = os.path.join(OUTPUT_PATH, f"GRID_SEARCH_ALL_RESULTS_gold_{GOLD_ANNOTATOR}.csv")
df_all.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# Best per model summary
best_per_model = []
for model_name in df_all["Model"].unique():
    df_model = df_all[df_all["Model"] == model_name]
    best_row = df_model.iloc[0].to_dict()
    best_per_model.append(best_row)

df_best = pd.DataFrame(best_per_model)
out_best = os.path.join(OUTPUT_PATH, f"BEST_PER_MODEL_gold_{GOLD_ANNOTATOR}.xlsx")
df_best.to_excel(out_best, index=False)
print(f"✅ Saved: {out_best}")

# JSON config global
best_configs = {}
for row in best_per_model:
    safe_layer = row['layer_strategy'].replace(':', '_')
    base_name = f"{row['Model']}_BEST_{row['TextCfg']}_{row['pooling']}_{safe_layer}_{row['Classifier']}"

    best_configs[row["Model"]] = {
        "cfg": row["TextCfg"],
        "classifier": row["Classifier"],
        "pooling": row["pooling"],
        "layer_strategy": row["layer_strategy"],
        "threshold": row["Thr(best)"],
        "MCC": row["MCC(best)"],
        "hf_model": row["HF_model"],
        "gold_annotator": GOLD_ANNOTATOR,
        "proba_path": os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy"),
        "emb_path": os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy"),
    }

out_json = os.path.join(OUTPUT_PATH, f"BEST_CONFIGS_gold_{GOLD_ANNOTATOR}.json")
with open(out_json, "w") as f:
    json.dump(best_configs, f, indent=2)
print(f"✅ Saved: {out_json}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 80)
print(f"SUMMARY — BEST CONFIG PER MODEL (GOLD={GOLD_ANNOTATOR})")
print("=" * 80)

summary_cols = ["Model", "TextCfg", "pooling", "layer_strategy", "Classifier", "Thr(best)", "MCC(best)", "MCC@0.5"]
print(df_best[summary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("TOP 15 OVERALL")
print("=" * 80)

print(df_all.head(15)[summary_cols].to_string(index=False))

print(f"\n✅ DONE — All outputs in: {OUTPUT_PATH}")
print(f"✅ Best models saved in: {BEST_OUTPUT_PATH}")
print(f"   Total configurations tested: {len(df_all)}")
print(f"   Gold annotator: {GOLD_ANNOTATOR}")

# Other A2

In [ ]:
# ============================================================
# GRID SEARCH — TRANSFORMERS & SENTENCE TRANSFORMERS
# PARALLELIZED VERSION + SAVE BEST EMBEDDINGS/OOF
#
# Parallelization:
#   - Embeddings: sequential (GPU-bound, 1 GPU)
#   - Classifiers: parallel with joblib (CPU-bound)
#   - Threshold search: vectorized with numpy
#
# Models tested (ALL with pooling/layer configs):
#   - CamemBERT, CamemBERTav2, JuriBERT-base
#   - ST-MiniLM, ST-MPNet (treated as transformers for pooling/layer)
#
# Grid: 3 text configs × 4 pooling/layer × 3 classifiers = 36 per model
# Total: 5 models × 36 = 180 combinations
#
# NEW: save embeddings + OOF for the best config per model
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc, hashlib, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

from joblib import Parallel, delayed
import torch

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A2"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"oof_proba_final_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Folder for the best results
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Parallelization settings
N_JOBS = -1  # -1 = all CPU cores

# Transformer settings
MAX_LEN = 512
BATCH_SIZE = 16

# MLP configs
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300

MLP2_HIDDEN = (256, 64)
MLP2_ALPHA = 2e-3
MLP2_LR_INIT = 3e-4
MLP2_MAX_ITER = 400

MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# Threshold grid (vectorized)
THR_GRID = np.linspace(0.05, 0.95, 181)

# ============================================================
# ALL MODELS — Treated uniformly as transformers
# ============================================================
ALL_MODELS = {
    "CamemBERT": {
        "hf_model": "camembert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "CamemBERTav2": {
        "hf_model": "almanach/camembertav2-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "JuriBERT-base": {
        "hf_model": "dascim/juribert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    # SentenceTransformers — access the underlying model
    "ST-MiniLM": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
    "ST-MPNet": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
}

# Grid search configs
TEXT_CONFIGS = ["cfg1", "cfg2", "cfg3"]
CLASSIFIERS = ["LR", "MLP1", "MLP2"]

POOLING_LAYER_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
    {"pooling": "cls", "layer_strategy": "last"},
]


# ============================================================
# HELPERS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()


# ============================================================
# TEXT CONFIG BUILDERS
# ============================================================
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")


# ============================================================
# CLASSIFIER FACTORIES
# ============================================================
def make_classifier(clf_type, seed):
    if clf_type == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
    elif clf_type == "MLP1":
        return MLPClassifier(
            hidden_layer_sizes=MLP1_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP1_ALPHA,
            learning_rate_init=MLP1_LR_INIT,
            max_iter=MLP1_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    elif clf_type == "MLP2":
        return MLPClassifier(
            hidden_layer_sizes=MLP2_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP2_ALPHA,
            learning_rate_init=MLP2_LR_INIT,
            max_iter=MLP2_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown classifier type: {clf_type}")


# ============================================================
# METRICS — VECTORIZED THRESHOLD SEARCH
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


def best_threshold_vectorized(y_true, proba, thresholds=THR_GRID):
    """Vectorized threshold search — much faster than loop."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    # Broadcast: (n_samples,) vs (n_thresholds,) -> (n_thresholds, n_samples)
    preds = (proba[np.newaxis, :] >= thresholds[:, np.newaxis]).astype(int)

    # Compute MCC for each threshold
    tp = ((preds == 1) & (y_true == 1)).sum(axis=1)
    tn = ((preds == 0) & (y_true == 0)).sum(axis=1)
    fp = ((preds == 1) & (y_true == 0)).sum(axis=1)
    fn = ((preds == 0) & (y_true == 1)).sum(axis=1)

    # MCC formula
    num = (tp * tn - fp * fn).astype(float)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn).astype(float))
    denom = np.where(denom == 0, 1, denom)  # Avoid division by zero
    mcc = num / denom

    best_idx = np.argmax(mcc)
    return float(thresholds[best_idx]), float(mcc[best_idx])


# ============================================================
# GROUPED FOLDS
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# UNIFIED ENCODER — Works for both transformers and SentenceTransformers
# ============================================================
def encode_with_pooling_layer(texts, model_name, model_type, batch_size, max_len, pooling, layer_strategy, desc):
    """
    Unified encoder that handles both regular transformers and SentenceTransformers.
    For SentenceTransformers, we access the underlying transformer model directly.

    Returns: (embeddings, cache_path) - embeddings array and path where they're cached
    """
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    # Check cache
    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        print(f"  [CACHE HIT] {os.path.basename(cache_path)}")
        return np.load(cache_path), cache_path

    print(f"  [CACHE MISS] Computing embeddings...")

    # Load model
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            k = min(k, len(hidden_states))
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
            batch = texts[i:i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            out = model(**enc, output_hidden_states=True)
            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:  # mean pooling
                mask = attn.unsqueeze(-1).float()
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.cpu().numpy())

            del enc, out, hs, attn, x, emb
            if i % (batch_size * 10) == 0:
                _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)
    print(f"  [SAVED] {os.path.basename(cache_path)} — shape: {X.shape}")

    del model, tok
    _clear_cuda()
    return X, cache_path


# ============================================================
# PARALLEL OOF EVALUATION
# ============================================================
def train_fold(fold, X, y, folds, clf_type, seed):
    """Train on one fold and return OOF predictions for test indices."""
    tr = folds != fold
    te = folds == fold

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[tr])
    Xte = scaler.transform(X[te])
    ytr = y[tr]

    clf = make_classifier(clf_type, seed)  # constant seed (as in the reference script)
    clf.fit(Xtr, ytr)
    proba_te = clf.predict_proba(Xte)[:, 1]

    return np.where(te)[0], proba_te


def oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=N_JOBS):
    """Parallel OOF computation across folds."""
    proba_oof = np.full(len(y), np.nan, dtype=float)

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(train_fold)(fold, X, y, folds, clf_type, seed)
        for fold in range(n_splits)
    )

    for te_idx, proba_te in results:
        proba_oof[te_idx] = proba_te

    assert np.isfinite(proba_oof).all(), "NaN in OOF predictions!"
    return proba_oof


def evaluate_single_classifier(X, y, folds, n_splits, clf_type, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate one classifier configuration — called in parallel."""
    proba_oof = oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=1)

    # Metrics at 0.5
    pred_05 = (proba_oof >= 0.5).astype(int)
    met_05 = compute_metrics(y, pred_05)

    # Best threshold (vectorized)
    best_thr, best_mcc = best_threshold_vectorized(y, proba_oof)
    pred_best = (proba_oof >= best_thr).astype(int)
    met_best = compute_metrics(y, pred_best)

    return {
        "clf_type": clf_type,
        "proba_oof": proba_oof,
        "best_thr": best_thr,
        "met_05": met_05,
        "met_best": met_best,
    }


def evaluate_all_classifiers_parallel(X, y, folds, n_splits, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate all classifiers in parallel for a given embedding."""
    results = Parallel(n_jobs=len(CLASSIFIERS), backend="loky")(
        delayed(evaluate_single_classifier)(
            X, y, folds, n_splits, clf_type, seed,
            model_name, cfg_id, pooling, layer_strategy
        )
        for clf_type in CLASSIFIERS
    )
    return {r["clf_type"]: r for r in results}


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

# Keep only rows with a valid label
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} examples | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values
folds = df_cv["fold"].values


# ============================================================
# PRE-COMPUTE ALL TEXT CONFIGS
# ============================================================
print("\n" + "=" * 80)
print("PRE-COMPUTING TEXT CONFIGS")
print("=" * 80)

texts_cache = {}
for cfg_id in TEXT_CONFIGS:
    texts_cache[cfg_id] = make_text_inputs(df_cv, cfg_id)
    print(f"  {cfg_id}: {len(texts_cache[cfg_id])} texts")


# ============================================================
# MAIN GRID SEARCH — PARALLELIZED + SAVE BEST
# ============================================================
print("\n" + "=" * 80)
print(f"GRID SEARCH — {len(ALL_MODELS)} MODELS × {len(TEXT_CONFIGS)} CONFIGS × {len(POOLING_LAYER_CONFIGS)} POOLING × {len(CLASSIFIERS)} CLASSIFIERS")
print(f"Total combinations: {len(ALL_MODELS) * len(TEXT_CONFIGS) * len(POOLING_LAYER_CONFIGS) * len(CLASSIFIERS)}")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

all_results = []

# To track the best per model (with embeddings path)
best_per_model_tracking = {}

for model_name, model_cfg in ALL_MODELS.items():
    hf_model = model_cfg["hf_model"]
    batch_size = model_cfg["batch_size"]
    model_type = model_cfg["type"]

    print(f"\n{'='*80}")
    print(f"MODEL: {model_name} ({hf_model}) | GOLD={GOLD_ANNOTATOR}")
    print(f"{'='*80}")

    best_for_model = {"MCC(best)": -1e9}

    for cfg_id in TEXT_CONFIGS:
        texts = texts_cache[cfg_id]

        for pcfg in POOLING_LAYER_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{model_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print(f"\n--- {run_id} ---")

            # Get embeddings (GPU, sequential) - now also returns the cache_path
            X, emb_cache_path = encode_with_pooling_layer(
                texts=texts,
                model_name=hf_model,
                model_type=model_type,
                batch_size=batch_size,
                max_len=MAX_LEN,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )

            # Evaluate all classifiers in parallel (CPU)
            print(f"  Evaluating {len(CLASSIFIERS)} classifiers in parallel...")
            clf_results = evaluate_all_classifiers_parallel(
                X, y_true, folds, N_SPLITS, SEED,
                model_name, cfg_id, pooling, layer_strategy
            )

            # Collect results
            for clf_type, res in clf_results.items():
                met_05 = res["met_05"]
                met_best = res["met_best"]
                best_thr = res["best_thr"]
                proba_oof = res["proba_oof"]

                row = {
                    "Model": model_name,
                    "HF_model": hf_model,
                    "Gold_Annotator": GOLD_ANNOTATOR,
                    "TextCfg": cfg_id,
                    "pooling": pooling,
                    "layer_strategy": layer_strategy,
                    "Classifier": clf_type,

                    "Thr(best)": best_thr,
                    "MCC(best)": met_best["MCC"],
                    "Acc(best)": met_best["Accuracy"],
                    "BAcc(best)": met_best["Balanced Acc"],
                    "BF1(best)": met_best["Balanced F1"],
                    "F1-oui(best)": met_best["F1-oui"],
                    "F1-non(best)": met_best["F1-non"],

                    "MCC@0.5": met_05["MCC"],
                    "Acc@0.5": met_05["Accuracy"],
                    "BAcc@0.5": met_05["Balanced Acc"],
                    "ΔMCC": met_best["MCC"] - met_05["MCC"],
                }

                # Save ALL proba (comme avant)
                proba_fname = f"proba_oof_{model_name}_{cfg_id}_{pooling}_{layer_strategy.replace(':', '_')}_{clf_type}.npy"
                proba_path = os.path.join(OUTPUT_PATH, proba_fname)
                np.save(proba_path, proba_oof)
                row["proba_oof_path"] = proba_path
                row["emb_cache_path"] = emb_cache_path  # keep track of the embeddings path

                all_results.append(row)

                # Track the best for this model
                if row["MCC(best)"] > best_for_model.get("MCC(best)", -1e9):
                    best_for_model = row.copy()
                    best_for_model["_proba_oof"] = proba_oof  # keep the data in memory
                    best_for_model["_emb_cache_path"] = emb_cache_path

                print(f"    {clf_type}: MCC@0.5={met_05['MCC']:.4f} | thr={best_thr:.3f} MCC={met_best['MCC']:.4f}")

    # Save the best for this model
    best_per_model_tracking[model_name] = best_for_model

    print(f"\n✓ BEST for {model_name}:")
    print(f"  {best_for_model['TextCfg']}, {best_for_model['pooling']}, {best_for_model['layer_strategy']}, {best_for_model['Classifier']}")
    print(f"  MCC(best): {best_for_model['MCC(best)']:.4f} @ thr={best_for_model['Thr(best)']:.3f}")


# ============================================================
# SAVE BEST EMBEDDINGS & OOF PER MODEL
# ============================================================
print("\n" + "=" * 80)
print("SAVING BEST EMBEDDINGS & OOF PER MODEL")
print("=" * 80)

for model_name, best_info in best_per_model_tracking.items():
    print(f"\n{model_name}:")

    # Standardized file name
    safe_layer = best_info['layer_strategy'].replace(':', '_')
    base_name = f"{model_name}_BEST_{best_info['TextCfg']}_{best_info['pooling']}_{safe_layer}_{best_info['Classifier']}"

    # 1. Save OOF proba
    oof_path = os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy")
    np.save(oof_path, best_info["_proba_oof"])
    print(f"  ✓ OOF proba: {oof_path}")

    # 2. Copy/save embeddings
    emb_src = best_info["_emb_cache_path"]
    emb_dst = os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy")

    # Load and save (or copy)
    if os.path.exists(emb_src):
        import shutil
        shutil.copy(emb_src, emb_dst)
        print(f"  ✓ Embeddings: {emb_dst}")
    else:
        print(f"  ⚠ Embeddings source not found: {emb_src}")

    # 3. Save config JSON
    config = {
        "model_name": model_name,
        "hf_model": best_info["HF_model"],
        "gold_annotator": GOLD_ANNOTATOR,
        "text_cfg": best_info["TextCfg"],
        "pooling": best_info["pooling"],
        "layer_strategy": best_info["layer_strategy"],
        "classifier": best_info["Classifier"],
        "threshold": best_info["Thr(best)"],
        "MCC_best": best_info["MCC(best)"],
        "MCC_05": best_info["MCC@0.5"],
        "oof_path": oof_path,
        "emb_path": emb_dst,
    }
    config_path = os.path.join(BEST_OUTPUT_PATH, f"config_{base_name}.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  ✓ Config: {config_path}")


# ============================================================
# SAVE ALL RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING ALL RESULTS")
print("=" * 80)

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("MCC(best)", ascending=False).reset_index(drop=True)

# Excel with sheets per model
out_xlsx = os.path.join(OUTPUT_PATH, f"GRID_SEARCH_ALL_RESULTS_gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_all.to_excel(writer, index=False, sheet_name="All_Results")
    for model_name in df_all["Model"].unique():
        df_model = df_all[df_all["Model"] == model_name].copy()
        df_model.to_excel(writer, index=False, sheet_name=model_name[:31])
print(f"✅ Saved: {out_xlsx}")

# CSV
out_csv = os.path.join(OUTPUT_PATH, f"GRID_SEARCH_ALL_RESULTS_gold_{GOLD_ANNOTATOR}.csv")
df_all.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# Best per model summary
best_per_model = []
for model_name in df_all["Model"].unique():
    df_model = df_all[df_all["Model"] == model_name]
    best_row = df_model.iloc[0].to_dict()
    best_per_model.append(best_row)

df_best = pd.DataFrame(best_per_model)
out_best = os.path.join(OUTPUT_PATH, f"BEST_PER_MODEL_gold_{GOLD_ANNOTATOR}.xlsx")
df_best.to_excel(out_best, index=False)
print(f"✅ Saved: {out_best}")

# JSON config global
best_configs = {}
for row in best_per_model:
    safe_layer = row['layer_strategy'].replace(':', '_')
    base_name = f"{row['Model']}_BEST_{row['TextCfg']}_{row['pooling']}_{safe_layer}_{row['Classifier']}"

    best_configs[row["Model"]] = {
        "cfg": row["TextCfg"],
        "classifier": row["Classifier"],
        "pooling": row["pooling"],
        "layer_strategy": row["layer_strategy"],
        "threshold": row["Thr(best)"],
        "MCC": row["MCC(best)"],
        "hf_model": row["HF_model"],
        "gold_annotator": GOLD_ANNOTATOR,
        "proba_path": os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy"),
        "emb_path": os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy"),
    }

out_json = os.path.join(OUTPUT_PATH, f"BEST_CONFIGS_gold_{GOLD_ANNOTATOR}.json")
with open(out_json, "w") as f:
    json.dump(best_configs, f, indent=2)
print(f"✅ Saved: {out_json}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 80)
print(f"SUMMARY — BEST CONFIG PER MODEL (GOLD={GOLD_ANNOTATOR})")
print("=" * 80)

summary_cols = ["Model", "TextCfg", "pooling", "layer_strategy", "Classifier", "Thr(best)", "MCC(best)", "MCC@0.5"]
print(df_best[summary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("TOP 15 OVERALL")
print("=" * 80)

print(df_all.head(15)[summary_cols].to_string(index=False))

print(f"\n✅ DONE — All outputs in: {OUTPUT_PATH}")
print(f"✅ Best models saved in: {BEST_OUTPUT_PATH}")
print(f"   Total configurations tested: {len(df_all)}")
print(f"   Gold annotator: {GOLD_ANNOTATOR}")

# TF-IDF A1

In [ ]:
# ============================================================
# GRID SEARCH — TF-IDF + CLASSIFIERS (PARALLELIZED)
#
# Grid search complet:
#   - 3 text configs (cfg1, cfg2, cfg3)
#   - Multiple TF-IDF configs (ngrams, max_features, sublinear, etc.)
#   - 3 classifiers (LR, MLP1, MLP2)
#   - With and without StandardScaler
#
# PARALLELIZATION:
#   - Level 1: TF-IDF configs in parallel (joblib)
#   - Level 2: classifiers in parallel for each config
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

from joblib import Parallel, delayed
import multiprocessing

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A1"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"tfidf_results_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Parallelization settings
N_JOBS_OUTER = -1  # Pour la boucle principale (configs)
N_JOBS_INNER = 1   # For the folds (avoids nested parallelism issues)

# MLP configs
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300

MLP2_HIDDEN = (256, 64)
MLP2_ALPHA = 2e-3
MLP2_LR_INIT = 3e-4
MLP2_MAX_ITER = 400

MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# Threshold grid (vectorized)
THR_GRID = np.linspace(0.05, 0.95, 181)

# ============================================================
# TF-IDF CONFIGURATIONS (8 configs)
# ============================================================
TFIDF_CONFIGS = [
    # Unigrams basique
    {
        "name": "uni_5k",
        "ngram_range": (1, 1),
        "max_features": 5000,
        "sublinear_tf": False,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Unigrams + sublinear (often better)
    {
        "name": "uni_10k_sublin",
        "ngram_range": (1, 1),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams basique
    {
        "name": "bi_5k",
        "ngram_range": (1, 2),
        "max_features": 5000,
        "sublinear_tf": False,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams + sublinear
    {
        "name": "bi_10k_sublin",
        "ngram_range": (1, 2),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Trigrams + sublinear
    {
        "name": "tri_5k_sublin",
        "ngram_range": (1, 3),
        "max_features": 5000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams with doc-frequency filtering
    {
        "name": "bi_10k_mindf2",
        "ngram_range": (1, 2),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 2,
        "max_df": 0.95,
    },
    # Character n-grams (morphologie)
    {
        "name": "char_3_5_10k",
        "ngram_range": (3, 5),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
        "analyzer": "char_wb",
    },
    # Bigrams gros vocabulaire
    {
        "name": "bi_20k_sublin",
        "ngram_range": (1, 2),
        "max_features": 20000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
]

# Grid search configs
TEXT_CONFIGS = ["cfg1", "cfg2", "cfg3"]
CLASSIFIERS = ["LR", "MLP1", "MLP2"]
USE_SCALER_OPTIONS = [False, True]


# ============================================================
# HELPERS
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# TEXT CONFIG BUILDERS
# ============================================================
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")


# ============================================================
# CLASSIFIER FACTORIES
# ============================================================
def make_classifier(clf_type, seed):
    if clf_type == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
    elif clf_type == "MLP1":
        return MLPClassifier(
            hidden_layer_sizes=MLP1_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP1_ALPHA,
            learning_rate_init=MLP1_LR_INIT,
            max_iter=MLP1_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    elif clf_type == "MLP2":
        return MLPClassifier(
            hidden_layer_sizes=MLP2_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP2_ALPHA,
            learning_rate_init=MLP2_LR_INIT,
            max_iter=MLP2_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown classifier type: {clf_type}")


# ============================================================
# METRICS — VECTORIZED THRESHOLD SEARCH
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


def best_threshold_vectorized(y_true, proba, thresholds=THR_GRID):
    """Vectorized threshold search — much faster than loop."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    preds = (proba[np.newaxis, :] >= thresholds[:, np.newaxis]).astype(int)

    tp = ((preds == 1) & (y_true == 1)).sum(axis=1)
    tn = ((preds == 0) & (y_true == 0)).sum(axis=1)
    fp = ((preds == 1) & (y_true == 0)).sum(axis=1)
    fn = ((preds == 0) & (y_true == 1)).sum(axis=1)

    num = (tp * tn - fp * fn).astype(float)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn).astype(float))
    denom = np.where(denom == 0, 1, denom)
    mcc = num / denom

    best_idx = np.argmax(mcc)
    return float(thresholds[best_idx]), float(mcc[best_idx])


# ============================================================
# TF-IDF VECTORIZER FACTORY
# ============================================================
def make_tfidf_vectorizer(tfidf_cfg):
    """Create TfidfVectorizer from config dict."""
    return TfidfVectorizer(
        ngram_range=tfidf_cfg["ngram_range"],
        max_features=tfidf_cfg["max_features"],
        sublinear_tf=tfidf_cfg["sublinear_tf"],
        min_df=tfidf_cfg["min_df"],
        max_df=tfidf_cfg["max_df"],
        analyzer=tfidf_cfg.get("analyzer", "word"),
        lowercase=True,
        strip_accents="unicode",
    )


# ============================================================
# OOF EVALUATION WITH TF-IDF (single config)
# ============================================================
def oof_proba_tfidf(texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed):
    """
    Compute OOF probabilities with TF-IDF + classifier.
    TF-IDF is fit on train fold only to avoid leakage.
    """
    proba_oof = np.full(len(y), np.nan, dtype=float)

    for fold in range(n_splits):
        tr_mask = folds != fold
        te_mask = folds == fold

        texts_tr = [texts[i] for i in range(len(texts)) if tr_mask[i]]
        texts_te = [texts[i] for i in range(len(texts)) if te_mask[i]]
        y_tr = y[tr_mask]

        # Fit TF-IDF on train only
        vectorizer = make_tfidf_vectorizer(tfidf_cfg)
        X_tr = vectorizer.fit_transform(texts_tr)
        X_te = vectorizer.transform(texts_te)

        # Convert to dense for MLP (sparse OK for LR)
        if clf_type in ["MLP1", "MLP2"]:
            X_tr = X_tr.toarray()
            X_te = X_te.toarray()

        # Optional scaling
        if use_scaler:
            if clf_type in ["MLP1", "MLP2"]:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_te = scaler.transform(X_te)
            else:
                scaler = StandardScaler(with_mean=False)
                X_tr = scaler.fit_transform(X_tr)
                X_te = scaler.transform(X_te)

        clf = make_classifier(clf_type, seed)
        clf.fit(X_tr, y_tr)
        proba_te = clf.predict_proba(X_te)[:, 1]
        proba_oof[te_mask] = proba_te

    assert np.isfinite(proba_oof).all(), "NaN in OOF predictions!"
    return proba_oof


# ============================================================
# PARALLEL WORKER — evaluate a full combination
# ============================================================
def evaluate_single_config(cfg_id, texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed, output_path, gold_annotator):
    """
    Worker function to evaluate a single configuration.
    Returns a dict with all results.
    """
    tfidf_name = tfidf_cfg["name"]
    scaler_str = "scaled" if use_scaler else "raw"
    run_id = f"{cfg_id}|{tfidf_name}|{clf_type}|{scaler_str}"

    try:
        proba_oof = oof_proba_tfidf(texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed)

        # Metrics at 0.5
        pred_05 = (proba_oof >= 0.5).astype(int)
        met_05 = compute_metrics(y, pred_05)

        # Best threshold
        best_thr, _ = best_threshold_vectorized(y, proba_oof)
        pred_best = (proba_oof >= best_thr).astype(int)
        met_best = compute_metrics(y, pred_best)

        # Save proba_oof
        proba_fname = f"proba_oof_{cfg_id}_{tfidf_name}_{clf_type}_{scaler_str}.npy"
        proba_path = os.path.join(output_path, proba_fname)
        np.save(proba_path, proba_oof)

        row = {
            "Gold_Annotator": gold_annotator,
            "TextCfg": cfg_id,
            "TFIDF_Config": tfidf_name,
            "ngram_range": str(tfidf_cfg["ngram_range"]),
            "max_features": tfidf_cfg["max_features"],
            "sublinear_tf": tfidf_cfg["sublinear_tf"],
            "min_df": tfidf_cfg["min_df"],
            "max_df": tfidf_cfg["max_df"],
            "analyzer": tfidf_cfg.get("analyzer", "word"),
            "Classifier": clf_type,
            "UseScaler": use_scaler,

            "Thr(best)": best_thr,
            "MCC(best)": met_best["MCC"],
            "Acc(best)": met_best["Accuracy"],
            "BAcc(best)": met_best["Balanced Acc"],
            "BF1(best)": met_best["Balanced F1"],
            "F1-oui(best)": met_best["F1-oui"],
            "F1-non(best)": met_best["F1-non"],

            "MCC@0.5": met_05["MCC"],
            "Acc@0.5": met_05["Accuracy"],
            "BAcc@0.5": met_05["Balanced Acc"],
            "BF1@0.5": met_05["Balanced F1"],
            "ΔMCC": met_best["MCC"] - met_05["MCC"],

            "proba_oof_path": proba_path,
            "_proba_oof": proba_oof,  # Pour tracker le best
            "_success": True,
            "_run_id": run_id,
        }

        return row

    except Exception as e:
        return {
            "_success": False,
            "_run_id": run_id,
            "_error": str(e),
        }


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} examples | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values
folds = df_cv["fold"].values


# ============================================================
# PRE-COMPUTE ALL TEXT CONFIGS
# ============================================================
print("\n" + "=" * 80)
print("PRE-COMPUTING TEXT CONFIGS")
print("=" * 80)

texts_cache = {}
for cfg_id in TEXT_CONFIGS:
    texts_cache[cfg_id] = make_text_inputs(df_cv, cfg_id)
    print(f"  {cfg_id}: {len(texts_cache[cfg_id])} texts")


# ============================================================
# BUILD LIST OF ALL JOBS
# ============================================================
all_jobs = []
for cfg_id in TEXT_CONFIGS:
    for tfidf_cfg in TFIDF_CONFIGS:
        for clf_type in CLASSIFIERS:
            for use_scaler in USE_SCALER_OPTIONS:
                all_jobs.append({
                    "cfg_id": cfg_id,
                    "tfidf_cfg": tfidf_cfg,
                    "clf_type": clf_type,
                    "use_scaler": use_scaler,
                })

n_total = len(all_jobs)
n_cores = multiprocessing.cpu_count()

print("\n" + "=" * 80)
print(f"GRID SEARCH — TF-IDF (PARALLELIZED)")
print(f"  {len(TEXT_CONFIGS)} text configs × {len(TFIDF_CONFIGS)} TF-IDF configs × {len(CLASSIFIERS)} classifiers × {len(USE_SCALER_OPTIONS)} scaler options")
print(f"  Total combinations: {n_total}")
print(f"  CPU cores available: {n_cores}")
print(f"  Using {N_JOBS_OUTER} workers (outer loop)")
print(f"  GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)


# ============================================================
# PARALLEL EXECUTION
# ============================================================
print("\n🚀 Starting parallel execution...")

results = Parallel(n_jobs=N_JOBS_OUTER, backend="loky", verbose=10)(
    delayed(evaluate_single_config)(
        job["cfg_id"],
        texts_cache[job["cfg_id"]],
        y_true,
        folds,
        N_SPLITS,
        job["tfidf_cfg"],
        job["clf_type"],
        job["use_scaler"],
        SEED,
        OUTPUT_PATH,
        GOLD_ANNOTATOR
    )
    for job in all_jobs
)


# ============================================================
# COLLECT RESULTS
# ============================================================
print("\n" + "=" * 80)
print("COLLECTING RESULTS")
print("=" * 80)

all_results = []
errors = []
best_overall = {"MCC(best)": -1e9}

for res in results:
    if res.get("_success", False):
        # Track best
        if res["MCC(best)"] > best_overall.get("MCC(best)", -1e9):
            best_overall = res.copy()

        # Remove internal keys before saving
        row = {k: v for k, v in res.items() if not k.startswith("_")}
        all_results.append(row)
    else:
        errors.append(res)

print(f"✓ Successful: {len(all_results)}")
print(f"✗ Errors: {len(errors)}")

if errors:
    print("\nErrors:")
    for err in errors[:10]:
        print(f"  - {err.get('_run_id', '?')}: {err.get('_error', '?')}")


# ============================================================
# SAVE RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("MCC(best)", ascending=False).reset_index(drop=True)

# Excel
out_xlsx = os.path.join(OUTPUT_PATH, f"TFIDF_GRID_SEARCH_gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_all.to_excel(writer, index=False, sheet_name="All_Results")

    for clf in CLASSIFIERS:
        df_clf = df_all[df_all["Classifier"] == clf].copy()
        df_clf.to_excel(writer, index=False, sheet_name=f"{clf}_results")

    for cfg_id in TEXT_CONFIGS:
        df_cfg = df_all[df_all["TextCfg"] == cfg_id].copy()
        df_cfg.to_excel(writer, index=False, sheet_name=f"{cfg_id}_results")

print(f"✅ Saved: {out_xlsx}")

# CSV
out_csv = os.path.join(OUTPUT_PATH, f"TFIDF_GRID_SEARCH_gold_{GOLD_ANNOTATOR}.csv")
df_all.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# Best config JSON
best_config = {
    "gold_annotator": GOLD_ANNOTATOR,
    "text_cfg": best_overall.get("TextCfg", ""),
    "tfidf_config": best_overall.get("TFIDF_Config", ""),
    "ngram_range": best_overall.get("ngram_range", ""),
    "max_features": best_overall.get("max_features", ""),
    "sublinear_tf": best_overall.get("sublinear_tf", ""),
    "classifier": best_overall.get("Classifier", ""),
    "use_scaler": best_overall.get("UseScaler", ""),
    "threshold": best_overall.get("Thr(best)", ""),
    "MCC_best": best_overall.get("MCC(best)", ""),
    "MCC_05": best_overall.get("MCC@0.5", ""),
    "proba_path": best_overall.get("proba_oof_path", ""),
}

out_json = os.path.join(OUTPUT_PATH, f"BEST_CONFIG_gold_{GOLD_ANNOTATOR}.json")
with open(out_json, "w") as f:
    json.dump(best_config, f, indent=2)
print(f"✅ Saved: {out_json}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 80)
print(f"SUMMARY — TOP 20 CONFIGS (GOLD={GOLD_ANNOTATOR})")
print("=" * 80)

summary_cols = ["TextCfg", "TFIDF_Config", "Classifier", "UseScaler", "Thr(best)", "MCC(best)", "MCC@0.5", "BAcc(best)"]
print(df_all.head(20)[summary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("BEST BY CLASSIFIER")
print("=" * 80)

for clf in CLASSIFIERS:
    df_clf = df_all[df_all["Classifier"] == clf]
    if len(df_clf) > 0:
        best_row = df_clf.iloc[0]
        print(f"\n{clf}:")
        print(f"  Config: {best_row['TextCfg']} | {best_row['TFIDF_Config']} | Scaler={best_row['UseScaler']}")
        print(f"  MCC(best): {best_row['MCC(best)']:.4f} @ thr={best_row['Thr(best)']:.3f}")
        print(f"  MCC@0.5:   {best_row['MCC@0.5']:.4f}")

print("\n" + "=" * 80)
print("BEST BY TEXT CONFIG")
print("=" * 80)

for cfg_id in TEXT_CONFIGS:
    df_cfg = df_all[df_all["TextCfg"] == cfg_id]
    if len(df_cfg) > 0:
        best_row = df_cfg.iloc[0]
        print(f"\n{cfg_id}:")
        print(f"  TFIDF: {best_row['TFIDF_Config']} | {best_row['Classifier']} | Scaler={best_row['UseScaler']}")
        print(f"  MCC(best): {best_row['MCC(best)']:.4f} @ thr={best_row['Thr(best)']:.3f}")

print("\n" + "=" * 80)
print("SCALER COMPARISON")
print("=" * 80)

for use_scaler in [False, True]:
    df_scaler = df_all[df_all["UseScaler"] == use_scaler]
    if len(df_scaler) > 0:
        avg_mcc = df_scaler["MCC(best)"].mean()
        max_mcc = df_scaler["MCC(best)"].max()
        print(f"\nScaler={use_scaler}:")
        print(f"  Avg MCC(best): {avg_mcc:.4f}")
        print(f"  Max MCC(best): {max_mcc:.4f}")

print("\n" + "=" * 80)
print("BEST BY TFIDF CONFIG")
print("=" * 80)

for tfidf_cfg in TFIDF_CONFIGS:
    tfidf_name = tfidf_cfg["name"]
    df_tfidf = df_all[df_all["TFIDF_Config"] == tfidf_name]
    if len(df_tfidf) > 0:
        best_row = df_tfidf.iloc[0]
        print(f"{tfidf_name:20s} | {best_row['Classifier']:4s} | Scaler={str(best_row['UseScaler']):5s} | MCC={best_row['MCC(best)']:.4f}")

print(f"\n✅ DONE — All outputs in: {OUTPUT_PATH}")
print(f"   Total configurations tested: {len(df_all)}")
print(f"   Gold annotator: {GOLD_ANNOTATOR}")
print(f"\n🏆 BEST OVERALL:")
print(f"   {best_overall.get('TextCfg', '?')} | {best_overall.get('TFIDF_Config', '?')} | {best_overall.get('Classifier', '?')} | Scaler={best_overall.get('UseScaler', '?')}")
print(f"   MCC(best): {best_overall.get('MCC(best)', 0):.4f} @ thr={best_overall.get('Thr(best)', 0):.3f}")

# TF IDF A2

In [ ]:
# ============================================================
# GRID SEARCH — TF-IDF + CLASSIFIERS (PARALLELIZED)
#
# Grid search complet:
#   - 3 text configs (cfg1, cfg2, cfg3)
#   - Multiple TF-IDF configs (ngrams, max_features, sublinear, etc.)
#   - 3 classifiers (LR, MLP1, MLP2)
#   - With and without StandardScaler
#
# PARALLELIZATION:
#   - Level 1: TF-IDF configs in parallel (joblib)
#   - Level 2: classifiers in parallel for each config
#
# GOLD LABEL = annotation from a single annotator (configurable)
# ============================================================

import os, re, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

from joblib import Parallel, delayed
import multiprocessing

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A2"  # <-- CHANGE HERE to switch the annotator

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"tfidf_results_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Parallelization settings
N_JOBS_OUTER = -1  # Pour la boucle principale (configs)
N_JOBS_INNER = 1   # For the folds (avoids nested parallelism issues)

# MLP configs
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300

MLP2_HIDDEN = (256, 64)
MLP2_ALPHA = 2e-3
MLP2_LR_INIT = 3e-4
MLP2_MAX_ITER = 400

MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# Threshold grid (vectorized)
THR_GRID = np.linspace(0.05, 0.95, 181)

# ============================================================
# TF-IDF CONFIGURATIONS (8 configs)
# ============================================================
TFIDF_CONFIGS = [
    # Unigrams basique
    {
        "name": "uni_5k",
        "ngram_range": (1, 1),
        "max_features": 5000,
        "sublinear_tf": False,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Unigrams + sublinear (often better)
    {
        "name": "uni_10k_sublin",
        "ngram_range": (1, 1),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams basique
    {
        "name": "bi_5k",
        "ngram_range": (1, 2),
        "max_features": 5000,
        "sublinear_tf": False,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams + sublinear
    {
        "name": "bi_10k_sublin",
        "ngram_range": (1, 2),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Trigrams + sublinear
    {
        "name": "tri_5k_sublin",
        "ngram_range": (1, 3),
        "max_features": 5000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
    # Bigrams with doc-frequency filtering
    {
        "name": "bi_10k_mindf2",
        "ngram_range": (1, 2),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 2,
        "max_df": 0.95,
    },
    # Character n-grams (morphologie)
    {
        "name": "char_3_5_10k",
        "ngram_range": (3, 5),
        "max_features": 10000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
        "analyzer": "char_wb",
    },
    # Bigrams gros vocabulaire
    {
        "name": "bi_20k_sublin",
        "ngram_range": (1, 2),
        "max_features": 20000,
        "sublinear_tf": True,
        "min_df": 1,
        "max_df": 1.0,
    },
]

# Grid search configs
TEXT_CONFIGS = ["cfg1", "cfg2", "cfg3"]
CLASSIFIERS = ["LR", "MLP1", "MLP2"]
USE_SCALER_OPTIONS = [False, True]


# ============================================================
# HELPERS
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# TEXT CONFIG BUILDERS
# ============================================================
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")


# ============================================================
# CLASSIFIER FACTORIES
# ============================================================
def make_classifier(clf_type, seed):
    if clf_type == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
    elif clf_type == "MLP1":
        return MLPClassifier(
            hidden_layer_sizes=MLP1_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP1_ALPHA,
            learning_rate_init=MLP1_LR_INIT,
            max_iter=MLP1_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    elif clf_type == "MLP2":
        return MLPClassifier(
            hidden_layer_sizes=MLP2_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP2_ALPHA,
            learning_rate_init=MLP2_LR_INIT,
            max_iter=MLP2_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown classifier type: {clf_type}")


# ============================================================
# METRICS — VECTORIZED THRESHOLD SEARCH
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


def best_threshold_vectorized(y_true, proba, thresholds=THR_GRID):
    """Vectorized threshold search — much faster than loop."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    preds = (proba[np.newaxis, :] >= thresholds[:, np.newaxis]).astype(int)

    tp = ((preds == 1) & (y_true == 1)).sum(axis=1)
    tn = ((preds == 0) & (y_true == 0)).sum(axis=1)
    fp = ((preds == 1) & (y_true == 0)).sum(axis=1)
    fn = ((preds == 0) & (y_true == 1)).sum(axis=1)

    num = (tp * tn - fp * fn).astype(float)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn).astype(float))
    denom = np.where(denom == 0, 1, denom)
    mcc = num / denom

    best_idx = np.argmax(mcc)
    return float(thresholds[best_idx]), float(mcc[best_idx])


# ============================================================
# TF-IDF VECTORIZER FACTORY
# ============================================================
def make_tfidf_vectorizer(tfidf_cfg):
    """Create TfidfVectorizer from config dict."""
    return TfidfVectorizer(
        ngram_range=tfidf_cfg["ngram_range"],
        max_features=tfidf_cfg["max_features"],
        sublinear_tf=tfidf_cfg["sublinear_tf"],
        min_df=tfidf_cfg["min_df"],
        max_df=tfidf_cfg["max_df"],
        analyzer=tfidf_cfg.get("analyzer", "word"),
        lowercase=True,
        strip_accents="unicode",
    )


# ============================================================
# OOF EVALUATION WITH TF-IDF (single config)
# ============================================================
def oof_proba_tfidf(texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed):
    """
    Compute OOF probabilities with TF-IDF + classifier.
    TF-IDF is fit on train fold only to avoid leakage.
    """
    proba_oof = np.full(len(y), np.nan, dtype=float)

    for fold in range(n_splits):
        tr_mask = folds != fold
        te_mask = folds == fold

        texts_tr = [texts[i] for i in range(len(texts)) if tr_mask[i]]
        texts_te = [texts[i] for i in range(len(texts)) if te_mask[i]]
        y_tr = y[tr_mask]

        # Fit TF-IDF on train only
        vectorizer = make_tfidf_vectorizer(tfidf_cfg)
        X_tr = vectorizer.fit_transform(texts_tr)
        X_te = vectorizer.transform(texts_te)

        # Convert to dense for MLP (sparse OK for LR)
        if clf_type in ["MLP1", "MLP2"]:
            X_tr = X_tr.toarray()
            X_te = X_te.toarray()

        # Optional scaling
        if use_scaler:
            if clf_type in ["MLP1", "MLP2"]:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(X_tr)
                X_te = scaler.transform(X_te)
            else:
                scaler = StandardScaler(with_mean=False)
                X_tr = scaler.fit_transform(X_tr)
                X_te = scaler.transform(X_te)

        clf = make_classifier(clf_type, seed)
        clf.fit(X_tr, y_tr)
        proba_te = clf.predict_proba(X_te)[:, 1]
        proba_oof[te_mask] = proba_te

    assert np.isfinite(proba_oof).all(), "NaN in OOF predictions!"
    return proba_oof


# ============================================================
# PARALLEL WORKER — evaluate a full combination
# ============================================================
def evaluate_single_config(cfg_id, texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed, output_path, gold_annotator):
    """
    Worker function to evaluate a single configuration.
    Returns a dict with all results.
    """
    tfidf_name = tfidf_cfg["name"]
    scaler_str = "scaled" if use_scaler else "raw"
    run_id = f"{cfg_id}|{tfidf_name}|{clf_type}|{scaler_str}"

    try:
        proba_oof = oof_proba_tfidf(texts, y, folds, n_splits, tfidf_cfg, clf_type, use_scaler, seed)

        # Metrics at 0.5
        pred_05 = (proba_oof >= 0.5).astype(int)
        met_05 = compute_metrics(y, pred_05)

        # Best threshold
        best_thr, _ = best_threshold_vectorized(y, proba_oof)
        pred_best = (proba_oof >= best_thr).astype(int)
        met_best = compute_metrics(y, pred_best)

        # Save proba_oof
        proba_fname = f"proba_oof_{cfg_id}_{tfidf_name}_{clf_type}_{scaler_str}.npy"
        proba_path = os.path.join(output_path, proba_fname)
        np.save(proba_path, proba_oof)

        row = {
            "Gold_Annotator": gold_annotator,
            "TextCfg": cfg_id,
            "TFIDF_Config": tfidf_name,
            "ngram_range": str(tfidf_cfg["ngram_range"]),
            "max_features": tfidf_cfg["max_features"],
            "sublinear_tf": tfidf_cfg["sublinear_tf"],
            "min_df": tfidf_cfg["min_df"],
            "max_df": tfidf_cfg["max_df"],
            "analyzer": tfidf_cfg.get("analyzer", "word"),
            "Classifier": clf_type,
            "UseScaler": use_scaler,

            "Thr(best)": best_thr,
            "MCC(best)": met_best["MCC"],
            "Acc(best)": met_best["Accuracy"],
            "BAcc(best)": met_best["Balanced Acc"],
            "BF1(best)": met_best["Balanced F1"],
            "F1-oui(best)": met_best["F1-oui"],
            "F1-non(best)": met_best["F1-non"],

            "MCC@0.5": met_05["MCC"],
            "Acc@0.5": met_05["Accuracy"],
            "BAcc@0.5": met_05["Balanced Acc"],
            "BF1@0.5": met_05["Balanced F1"],
            "ΔMCC": met_best["MCC"] - met_05["MCC"],

            "proba_oof_path": proba_path,
            "_proba_oof": proba_oof,  # Pour tracker le best
            "_success": True,
            "_run_id": run_id,
        }

        return row

    except Exception as e:
        return {
            "_success": False,
            "_run_id": run_id,
            "_error": str(e),
        }


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


# ============================================================
# GOLD LABEL = annotation from the chosen annotator
# ============================================================
if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError(f"GOLD_ANNOTATOR must be 'A1' or 'A2', not '{GOLD_ANNOTATOR}'")

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} examples | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values
folds = df_cv["fold"].values


# ============================================================
# PRE-COMPUTE ALL TEXT CONFIGS
# ============================================================
print("\n" + "=" * 80)
print("PRE-COMPUTING TEXT CONFIGS")
print("=" * 80)

texts_cache = {}
for cfg_id in TEXT_CONFIGS:
    texts_cache[cfg_id] = make_text_inputs(df_cv, cfg_id)
    print(f"  {cfg_id}: {len(texts_cache[cfg_id])} texts")


# ============================================================
# BUILD LIST OF ALL JOBS
# ============================================================
all_jobs = []
for cfg_id in TEXT_CONFIGS:
    for tfidf_cfg in TFIDF_CONFIGS:
        for clf_type in CLASSIFIERS:
            for use_scaler in USE_SCALER_OPTIONS:
                all_jobs.append({
                    "cfg_id": cfg_id,
                    "tfidf_cfg": tfidf_cfg,
                    "clf_type": clf_type,
                    "use_scaler": use_scaler,
                })

n_total = len(all_jobs)
n_cores = multiprocessing.cpu_count()

print("\n" + "=" * 80)
print(f"GRID SEARCH — TF-IDF (PARALLELIZED)")
print(f"  {len(TEXT_CONFIGS)} text configs × {len(TFIDF_CONFIGS)} TF-IDF configs × {len(CLASSIFIERS)} classifiers × {len(USE_SCALER_OPTIONS)} scaler options")
print(f"  Total combinations: {n_total}")
print(f"  CPU cores available: {n_cores}")
print(f"  Using {N_JOBS_OUTER} workers (outer loop)")
print(f"  GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("=" * 80)


# ============================================================
# PARALLEL EXECUTION
# ============================================================
print("\n🚀 Starting parallel execution...")

results = Parallel(n_jobs=N_JOBS_OUTER, backend="loky", verbose=10)(
    delayed(evaluate_single_config)(
        job["cfg_id"],
        texts_cache[job["cfg_id"]],
        y_true,
        folds,
        N_SPLITS,
        job["tfidf_cfg"],
        job["clf_type"],
        job["use_scaler"],
        SEED,
        OUTPUT_PATH,
        GOLD_ANNOTATOR
    )
    for job in all_jobs
)


# ============================================================
# COLLECT RESULTS
# ============================================================
print("\n" + "=" * 80)
print("COLLECTING RESULTS")
print("=" * 80)

all_results = []
errors = []
best_overall = {"MCC(best)": -1e9}

for res in results:
    if res.get("_success", False):
        # Track best
        if res["MCC(best)"] > best_overall.get("MCC(best)", -1e9):
            best_overall = res.copy()

        # Remove internal keys before saving
        row = {k: v for k, v in res.items() if not k.startswith("_")}
        all_results.append(row)
    else:
        errors.append(res)

print(f"✓ Successful: {len(all_results)}")
print(f"✗ Errors: {len(errors)}")

if errors:
    print("\nErrors:")
    for err in errors[:10]:
        print(f"  - {err.get('_run_id', '?')}: {err.get('_error', '?')}")


# ============================================================
# SAVE RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("MCC(best)", ascending=False).reset_index(drop=True)

# Excel
out_xlsx = os.path.join(OUTPUT_PATH, f"TFIDF_GRID_SEARCH_gold_{GOLD_ANNOTATOR}.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_all.to_excel(writer, index=False, sheet_name="All_Results")

    for clf in CLASSIFIERS:
        df_clf = df_all[df_all["Classifier"] == clf].copy()
        df_clf.to_excel(writer, index=False, sheet_name=f"{clf}_results")

    for cfg_id in TEXT_CONFIGS:
        df_cfg = df_all[df_all["TextCfg"] == cfg_id].copy()
        df_cfg.to_excel(writer, index=False, sheet_name=f"{cfg_id}_results")

print(f"✅ Saved: {out_xlsx}")

# CSV
out_csv = os.path.join(OUTPUT_PATH, f"TFIDF_GRID_SEARCH_gold_{GOLD_ANNOTATOR}.csv")
df_all.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# Best config JSON
best_config = {
    "gold_annotator": GOLD_ANNOTATOR,
    "text_cfg": best_overall.get("TextCfg", ""),
    "tfidf_config": best_overall.get("TFIDF_Config", ""),
    "ngram_range": best_overall.get("ngram_range", ""),
    "max_features": best_overall.get("max_features", ""),
    "sublinear_tf": best_overall.get("sublinear_tf", ""),
    "classifier": best_overall.get("Classifier", ""),
    "use_scaler": best_overall.get("UseScaler", ""),
    "threshold": best_overall.get("Thr(best)", ""),
    "MCC_best": best_overall.get("MCC(best)", ""),
    "MCC_05": best_overall.get("MCC@0.5", ""),
    "proba_path": best_overall.get("proba_oof_path", ""),
}

out_json = os.path.join(OUTPUT_PATH, f"BEST_CONFIG_gold_{GOLD_ANNOTATOR}.json")
with open(out_json, "w") as f:
    json.dump(best_config, f, indent=2)
print(f"✅ Saved: {out_json}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 80)
print(f"SUMMARY — TOP 20 CONFIGS (GOLD={GOLD_ANNOTATOR})")
print("=" * 80)

summary_cols = ["TextCfg", "TFIDF_Config", "Classifier", "UseScaler", "Thr(best)", "MCC(best)", "MCC@0.5", "BAcc(best)"]
print(df_all.head(20)[summary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("BEST BY CLASSIFIER")
print("=" * 80)

for clf in CLASSIFIERS:
    df_clf = df_all[df_all["Classifier"] == clf]
    if len(df_clf) > 0:
        best_row = df_clf.iloc[0]
        print(f"\n{clf}:")
        print(f"  Config: {best_row['TextCfg']} | {best_row['TFIDF_Config']} | Scaler={best_row['UseScaler']}")
        print(f"  MCC(best): {best_row['MCC(best)']:.4f} @ thr={best_row['Thr(best)']:.3f}")
        print(f"  MCC@0.5:   {best_row['MCC@0.5']:.4f}")

print("\n" + "=" * 80)
print("BEST BY TEXT CONFIG")
print("=" * 80)

for cfg_id in TEXT_CONFIGS:
    df_cfg = df_all[df_all["TextCfg"] == cfg_id]
    if len(df_cfg) > 0:
        best_row = df_cfg.iloc[0]
        print(f"\n{cfg_id}:")
        print(f"  TFIDF: {best_row['TFIDF_Config']} | {best_row['Classifier']} | Scaler={best_row['UseScaler']}")
        print(f"  MCC(best): {best_row['MCC(best)']:.4f} @ thr={best_row['Thr(best)']:.3f}")

print("\n" + "=" * 80)
print("SCALER COMPARISON")
print("=" * 80)

for use_scaler in [False, True]:
    df_scaler = df_all[df_all["UseScaler"] == use_scaler]
    if len(df_scaler) > 0:
        avg_mcc = df_scaler["MCC(best)"].mean()
        max_mcc = df_scaler["MCC(best)"].max()
        print(f"\nScaler={use_scaler}:")
        print(f"  Avg MCC(best): {avg_mcc:.4f}")
        print(f"  Max MCC(best): {max_mcc:.4f}")

print("\n" + "=" * 80)
print("BEST BY TFIDF CONFIG")
print("=" * 80)

for tfidf_cfg in TFIDF_CONFIGS:
    tfidf_name = tfidf_cfg["name"]
    df_tfidf = df_all[df_all["TFIDF_Config"] == tfidf_name]
    if len(df_tfidf) > 0:
        best_row = df_tfidf.iloc[0]
        print(f"{tfidf_name:20s} | {best_row['Classifier']:4s} | Scaler={str(best_row['UseScaler']):5s} | MCC={best_row['MCC(best)']:.4f}")

print(f"\n✅ DONE — All outputs in: {OUTPUT_PATH}")
print(f"   Total configurations tested: {len(df_all)}")
print(f"   Gold annotator: {GOLD_ANNOTATOR}")
print(f"\n🏆 BEST OVERALL:")
print(f"   {best_overall.get('TextCfg', '?')} | {best_overall.get('TFIDF_Config', '?')} | {best_overall.get('Classifier', '?')} | Scaler={best_overall.get('UseScaler', '?')}")
print(f"   MCC(best): {best_overall.get('MCC(best)', 0):.4f} @ thr={best_overall.get('Thr(best)', 0):.3f}")

# Best model A1

In [ ]:
# ============================================================
# SCRIPT — PARALLEL ENSEMBLE SEARCH (GOLD=A1)
#
# Parallelized with joblib for fast execution
# KEEPS fine threshold grid (2000 points) with vectorized search
# SAME structure as original consensus script
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from joblib import Parallel, delayed
import multiprocessing

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
# ============================================================
# GOLD ANNOTATOR
# ============================================================
GOLD_ANNOTATOR = "A1"

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)

# Paths for the gold_A1 results
OUTPUT_PATH = os.path.join(OUTPUT, f"oof_proba_final_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

# Ensemble-search results
ENSEMBLE_OUTPUT = os.path.join(OUTPUT_PATH, "ensemble_search_results")
os.makedirs(ENSEMBLE_OUTPUT, exist_ok=True)

# Path for SAUL/LLaMA (separate script)
SAUL_LLAMA_PATH = os.path.join(BASE_PATH, "outputs", f"outputs_saul_lr_llama_mlp1_3cfg_gold_{GOLD_ANNOTATOR}")

# Path for TF-IDF
TFIDF_PATH = os.path.join(OUTPUT, f"tfidf_results_gold_{GOLD_ANNOTATOR}")

SEED = 42
N_SPLITS = 5
N_JOBS = -1

np.random.seed(SEED)

# ============================================================
# MODEL DEFINITIONS — Based on the gold_A1 results
# ============================================================
# Simplified format like the original
# Simplified format like the original
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_layer_-2_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg3_mean_avg_last_k_4_MLP1.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg1_cls_last_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg3_cls_last_MLP1.npy",

    # SAUL / LLaMA from a separate folder
    "SAUL":  "SAUL_LLAMA_PATH/proba_oof_SAUL-7B__cfg2__mean__avg_last_k:4.npy",
    "LLaMA": "SAUL_LLAMA_PATH/proba_oof_LLaMA-3.1-8B__cfg3__mean__last.npy",
}


# TF-IDF — best config: cfg1 | bi_10k_sublin | LR | Scaler=False
TFIDF_FILE = "proba_oof_cfg1_bi_10k_sublin_LR_raw.npy"


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print(f"LOADING DATA — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

import re

def load_labels():
    EXCEL_PATH = "DATA/outputs/benchmark.csv"
    df0 = pd.read_csv(EXCEL_PATH)
    cols = ["decision_id", "eval_A1", "eval_A2", "eval_A3"]
    df0 = df0[cols].copy()

    def extract(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
        return np.nan

    df0["a"] = df0["eval_A1"].apply(extract)
    df0["t"] = df0["eval_A2"].apply(extract)
    df0["s"] = df0["eval_A3"].apply(extract)

    # GOLD = A1 directement
    df0["label_str"] = df0["a"]
    df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
    df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

    # Folds
    rng = np.random.default_rng(SEED)
    groups = df0.groupby("decision_id").size().to_dict()
    uniq = list(groups.keys())
    rng.shuffle(uniq)
    uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
    loads = np.zeros(N_SPLITS, dtype=int)
    g2f = {}
    for g in uniq:
        f = int(loads.argmin())
        g2f[g] = f
        loads[f] += groups[g]
    df0["fold"] = df0["decision_id"].map(g2f).astype(int)

    return df0["label"].values.astype(int), df0["fold"].values

y_true, folds = load_labels()
print(f"  {len(y_true)} samples | oui={y_true.sum()} | non={len(y_true)-y_true.sum()}")

# Load probas
probas = {}

# Models .npy
for name, fname in MODELS.items():
    if fname.startswith("SAUL_LLAMA_PATH/"):
        path = os.path.join(SAUL_LLAMA_PATH, fname.replace("SAUL_LLAMA_PATH/", ""))
    else:
        path = os.path.join(BEST_OUTPUT_PATH, fname)

    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name}: {os.path.basename(path)}")
    else:
        print(f"  ✗ NOT FOUND: {name} at {path}")

# TF-IDF
tfidf_path = os.path.join(TFIDF_PATH, TFIDF_FILE)
if os.path.exists(tfidf_path):
    probas["TF-IDF"] = np.load(tfidf_path)
    print(f"  ✓ TF-IDF: {TFIDF_FILE}")
else:
    print(f"  ✗ NOT FOUND: TF-IDF at {tfidf_path}")

model_names = list(probas.keys())
n_models = len(model_names)
print(f"\n{n_models} models loaded")
print(f"CPU cores: {multiprocessing.cpu_count()}")

if n_models == 0:
    raise ValueError("No models loaded! Check paths.")

# Pre-stack all probas for fast access
PROBA_MATRIX = np.column_stack([probas[m] for m in model_names])
MODEL_IDX = {m: i for i, m in enumerate(model_names)}

# Precompute y_true as int32 for speed
Y_TRUE_INT = y_true.astype(np.int32)
P_TOTAL = int(Y_TRUE_INT.sum())
N_TOTAL = len(Y_TRUE_INT) - P_TOTAL

# ============================================================
# FAST METRICS - VECTORIZED (2000 POINTS)
# ============================================================

# Fine grid: 2000 points from 0.15 to 0.85
THR_GRID = np.linspace(0.15, 0.85, 2000).astype(np.float64)

def fast_mcc(y_true, y_pred):
    """Fast MCC calculation."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))

    if den == 0:
        return 0.0
    return num / den

def best_threshold_fast(y_true, p_mix):
    """
    Vectorized threshold search over fine grid (2000 points).
    Uses sorting-based approach for O(n log n) instead of O(n * k).
    """
    n = len(p_mix)

    # Sort by probability descending
    order = np.argsort(p_mix)[::-1]
    y_sorted = y_true[order]
    p_sorted = p_mix[order]

    # Cumulative sums for TP and FP as we lower threshold
    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)

    # For each threshold, find how many samples are >= threshold
    # Using searchsorted on reversed array
    k = np.searchsorted(-p_sorted, -THR_GRID, side='right')

    # Clip to valid indices
    k_clipped = np.clip(k - 1, 0, n - 1)

    # Get TP and FP at each threshold
    tp = np.where(k == 0, 0, tp_cum[k_clipped])
    fp = np.where(k == 0, 0, fp_cum[k_clipped])

    fn = P_TOTAL - tp
    tn = N_TOTAL - fp

    # MCC calculation (vectorized)
    num = tp * tn - fp * fn
    den_sq = (tp + fp).astype(np.float64) * (tp + fn) * (tn + fp) * (tn + fn)
    den = np.sqrt(den_sq)

    # Avoid division by zero
    mcc = np.where(den > 0, num / den, 0.0)

    best_idx = np.argmax(mcc)
    return float(THR_GRID[best_idx]), float(mcc[best_idx])

def compute_metrics_fast(y_true, y_pred):
    return {
        "acc": float(np.mean(y_true == y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(fast_mcc(y_true, y_pred)),
    }

# ============================================================
# PARALLEL WEIGHT OPTIMIZATION
# ============================================================

def optimize_2models(combo, y_true):
    idx1, idx2 = MODEL_IDX[combo[0]], MODEL_IDX[combo[1]]

    best = {"mcc": -1e9}
    for w1 in np.linspace(0, 1, 21):  # 21 points
        p_mix = w1 * PROBA_MATRIX[:, idx1] + (1 - w1) * PROBA_MATRIX[:, idx2]
        thr, mcc = best_threshold_fast(y_true, p_mix)
        if mcc > best["mcc"]:
            best = {"weights": (w1, 1 - w1), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def optimize_3models(combo, y_true):
    idxs = [MODEL_IDX[m] for m in combo]

    best = {"mcc": -1e9}
    for w1 in np.linspace(0, 1, 11):  # 11 points
        for w2 in np.linspace(0, 1 - w1, 11):
            w3 = 1 - w1 - w2
            if w3 < -1e-6:
                continue
            p_mix = (w1 * PROBA_MATRIX[:, idxs[0]] +
                     w2 * PROBA_MATRIX[:, idxs[1]] +
                     w3 * PROBA_MATRIX[:, idxs[2]])
            thr, mcc = best_threshold_fast(y_true, p_mix)
            if mcc > best["mcc"]:
                best = {"weights": (w1, w2, w3), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def optimize_nmodels(combo, y_true, n_samples=2500):
    """Random search for n>3 models with 2500 samples."""
    idxs = [MODEL_IDX[m] for m in combo]
    n = len(combo)

    best = {"mcc": -1e9}
    rng = np.random.default_rng(SEED + hash(combo) % 10000)

    for _ in range(n_samples):
        weights = rng.dirichlet(np.ones(n))
        p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(weights, idxs))
        thr, mcc = best_threshold_fast(y_true, p_mix)
        if mcc > best["mcc"]:
            best = {"weights": tuple(weights), "thr": thr, "mcc": mcc}

    # Also try uniform weights
    weights = np.ones(n) / n
    p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(weights, idxs))
    thr, mcc = best_threshold_fast(y_true, p_mix)
    if mcc > best["mcc"]:
        best = {"weights": tuple(weights), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def process_combo(combo, y_true):
    n = len(combo)
    if n == 2:
        return optimize_2models(combo, y_true)
    elif n == 3:
        return optimize_3models(combo, y_true)
    else:
        return optimize_nmodels(combo, y_true)

# ============================================================
# STACKING
# ============================================================

def stacking_combo(combo, y_true, folds, meta_type="LR"):
    idxs = [MODEL_IDX[m] for m in combo]
    X_meta = PROBA_MATRIX[:, idxs]
    n = len(y_true)

    p_stack = np.zeros(n)

    for fold_id in range(N_SPLITS):
        tr = folds != fold_id
        te = folds == fold_id

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_meta[tr])
        X_te = scaler.transform(X_meta[te])

        if meta_type == "LR":
            clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300,
                              random_state=SEED, early_stopping=True)

        clf.fit(X_tr, y_true[tr])
        p_stack[te] = clf.predict_proba(X_te)[:, 1]

    thr, mcc = best_threshold_fast(y_true, p_stack)

    return {
        "method": f"Stacking_{meta_type}",
        "models": combo,
        "weights": None,
        "thr": thr,
        "mcc": mcc,
    }

# ============================================================
# RANK FUSION
# ============================================================

def rank_fusion_combo(combo, y_true):
    idxs = [MODEL_IDX[m] for m in combo]
    n_samples = len(y_true)

    ranks = []
    for idx in idxs:
        r = np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1)
        ranks.append(r)

    p_rank = np.mean(ranks, axis=0)
    thr, mcc = best_threshold_fast(y_true, p_rank)

    return {
        "method": "RankFusion",
        "models": combo,
        "weights": None,
        "thr": thr,
        "mcc": mcc,
    }

# ============================================================
# MAIN PARALLEL EXECUTION
# ============================================================
print("\n" + "=" * 80)
print(f"PARALLEL ENSEMBLE SEARCH — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

all_results = []

# --- 1. Weighted Average ---
print("\n[1/3] Weighted Average Ensembles...")

all_combos = []
for n in range(2, n_models + 1):
    all_combos.extend(list(combinations(model_names, n)))

print(f"  Total combinations: {len(all_combos)}")

results_wa = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(process_combo)(combo, y_true) for combo in all_combos
)
all_results.extend(results_wa)
print(f"  {len(results_wa)} weighted avg ensembles done")

# --- 2. Stacking ---
print("\n[2/3] Stacking Ensembles...")

wa_df = pd.DataFrame(results_wa).sort_values("mcc", ascending=False)
top_combos_for_stacking = [tuple(row["models"]) for _, row in wa_df.head(50).iterrows()]

specific_combos = [
    ("SAUL", "LLaMA"),
    ("SAUL", "TF-IDF"),
    ("SAUL", "LLaMA", "TF-IDF"),
    ("SAUL", "LLaMA", "TF-IDF", "JuriBERT"),
    ("SAUL", "LLaMA", "TF-IDF", "ST-MPNet"),
    ("LLaMA", "ST-MPNet"),
    ("LLaMA", "ST-MPNet", "SAUL"),
    tuple(model_names),
]
for c in specific_combos:
    if all(m in model_names for m in c) and c not in top_combos_for_stacking:
        top_combos_for_stacking.append(c)

stacking_tasks = []
for combo in top_combos_for_stacking:
    stacking_tasks.append((combo, "LR"))
    stacking_tasks.append((combo, "MLP"))

results_stack = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(stacking_combo)(combo, y_true, folds, meta) for combo, meta in stacking_tasks
)
all_results.extend(results_stack)
print(f"  {len(results_stack)} stacking ensembles done")

# --- 3. Rank Fusion ---
print("\n[3/3] Rank Fusion...")

rank_combos = []
for n in range(2, n_models + 1):
    rank_combos.extend(list(combinations(model_names, n)))

results_rank = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(rank_fusion_combo)(combo, y_true) for combo in rank_combos
)
all_results.extend(results_rank)
print(f"  {len(results_rank)} rank fusion ensembles done")

# ============================================================
# FULL METRICS
# ============================================================
print("\n[4/4] Computing full metrics...")

def add_full_metrics(result):
    combo = result["models"]
    idxs = [MODEL_IDX[m] for m in combo]

    if result["method"] == "RankFusion":
        n_samples = len(y_true)
        ranks = [np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1) for idx in idxs]
        p_mix = np.mean(ranks, axis=0)
    elif result["weights"] is not None:
        p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(result["weights"], idxs))
    else:
        return result

    y_pred = (p_mix >= result["thr"]).astype(int)
    metrics = compute_metrics_fast(y_true, y_pred)
    result.update(metrics)
    return result

all_results = [add_full_metrics(r) for r in tqdm(all_results, desc="Full metrics")]

# ============================================================
# RESULTS
# ============================================================
print("\n" + "=" * 80)
print(f"RESULTS — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

results_df = pd.DataFrame(all_results)

for col in ["acc", "bacc", "f1"]:
    if col not in results_df.columns:
        results_df[col] = np.nan

results_df = results_df.sort_values("mcc", ascending=False).reset_index(drop=True)

print("\n=== TOP 25 OVERALL ===")
for i, row in results_df.head(25).iterrows():
    models_str = "+".join(row["models"])
    w_str = ""
    if row["weights"] is not None:
        w_str = f" [{'/'.join([f'{w:.2f}' for w in row['weights']])}]"
    # Show threshold with full precision
    print(f"{i+1:2d}. {row['method']:15s} MCC={row['mcc']:.4f} thr={row['thr']:.6f} | {models_str}{w_str}")

print("\n=== BEST BY METHOD ===")
for method in results_df["method"].unique():
    best = results_df[results_df["method"] == method].iloc[0]
    print(f"{method:15s}: MCC={best['mcc']:.4f} thr={best['thr']:.6f} | {'+'.join(best['models'])}")

print("\n=== BEST WEIGHTED AVG BY SIZE ===")
wa_df = results_df[results_df["method"] == "WeightedAvg"]
for n in range(2, n_models + 1):
    subset = wa_df[wa_df["models"].apply(len) == n]
    if len(subset) > 0:
        best = subset.iloc[0]
        w_str = "/".join([f"{w:.2f}" for w in best["weights"]])
        print(f"{n}-model: MCC={best['mcc']:.4f} thr={best['thr']:.6f} | {'+'.join(best['models'])} [{w_str}]")

# Baselines - with exact thresholds
print("\n=== VS BASELINES (exact thresholds) ===")
baseline_results = {}
for name in model_names:
    thr, mcc = best_threshold_fast(y_true, probas[name])
    y_pred_05 = (probas[name] >= 0.5).astype(int)
    mcc_05 = fast_mcc(y_true, y_pred_05)

    baseline_results[name] = {
        "thr_best": thr,
        "mcc_best": mcc,
        "mcc_05": mcc_05,
        "delta": mcc - mcc_05,
    }
    # Full precision for threshold
    print(f"{name:15s}: MCC={mcc:.4f} (thr={thr:.6f}) | MCC@0.5={mcc_05:.4f} | delta={mcc - mcc_05:+.4f}")

# ============================================================
# SAVE
# ============================================================
print("\n" + "=" * 80)
print("SAVING")
print("=" * 80)

def serialize(row):
    return {
        "method": row["method"],
        "models": list(row["models"]),
        "weights": [float(w) for w in row["weights"]] if row["weights"] is not None else None,
        "threshold": float(row["thr"]) if "thr" in row else float(row.get("threshold", 0.5)),
        "mcc": float(row["mcc"]),
        "bacc": float(row["bacc"]) if pd.notna(row.get("bacc")) else None,
        "acc": float(row["acc"]) if pd.notna(row.get("acc")) else None,
        "f1": float(row["f1"]) if pd.notna(row.get("f1")) else None,
    }

with open(os.path.join(ENSEMBLE_OUTPUT, f"all_results_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump([serialize(row) for _, row in results_df.iterrows()], f, indent=2)

with open(os.path.join(ENSEMBLE_OUTPUT, f"top_100_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump([serialize(row) for _, row in results_df.head(100).iterrows()], f, indent=2)

csv_df = results_df.copy()
csv_df["models"] = csv_df["models"].apply(lambda x: "+".join(x))
csv_df["weights"] = csv_df["weights"].apply(
    lambda x: "/".join([f"{w:.3f}" for w in x]) if x is not None else "N/A"
)
# Keep full precision for threshold in CSV
csv_df["thr"] = csv_df["thr"].apply(lambda x: f"{x:.6f}")
csv_df.to_csv(os.path.join(ENSEMBLE_OUTPUT, f"all_results_gold_{GOLD_ANNOTATOR}.csv"), index=False)

# Best summary with exact thresholds
best_summary = {
    "gold_annotator": GOLD_ANNOTATOR,
    "overall_best": serialize(results_df.iloc[0]),
    "baselines": baseline_results,
    "best_by_method": {m: serialize(results_df[results_df["method"] == m].iloc[0])
                       for m in results_df["method"].unique()},
    "threshold_grid": {
        "min": float(THR_GRID[0]),
        "max": float(THR_GRID[-1]),
        "n_points": len(THR_GRID),
        "step": float(THR_GRID[1] - THR_GRID[0]),
    },
}

with open(os.path.join(ENSEMBLE_OUTPUT, f"best_summary_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump(best_summary, f, indent=2)

print(f"Saved to: {ENSEMBLE_OUTPUT}")

# Save threshold grid for reproducibility
np.save(os.path.join(ENSEMBLE_OUTPUT, f"THR_GRID_gold_{GOLD_ANNOTATOR}.npy"), THR_GRID)
print(f"THR_GRID_gold_{GOLD_ANNOTATOR}.npy saved ({len(THR_GRID)} points)")

# Correlation matrix
print("\nGenerating correlation matrix...")
corr = np.corrcoef(PROBA_MATRIX.T)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(corr, index=model_names, columns=model_names),
            annot=True, fmt=".2f", cmap="RdYlBu_r", center=0.5, ax=ax)
ax.set_title(f"Model Probability Correlations (Gold={GOLD_ANNOTATOR})")
plt.tight_layout()
fig.savefig(os.path.join(ENSEMBLE_OUTPUT, f"correlations_gold_{GOLD_ANNOTATOR}.png"), dpi=150)
plt.close()
print(f"correlations_gold_{GOLD_ANNOTATOR}.png saved")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

# ============================================================
# SAVE BEST MODEL OOF PROBAS
# ============================================================
print("\nSaving best model OOF probas...")

best_row = results_df.iloc[0]
best_combo = best_row["models"]
best_idxs = [MODEL_IDX[m] for m in best_combo]

# Recompute the best-ensemble probabilities
if best_row["method"] == "RankFusion":
    n_samples = len(y_true)
    ranks = [np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1) for idx in best_idxs]
    p_best = np.mean(ranks, axis=0)
elif best_row["weights"] is not None:
    p_best = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(best_row["weights"], best_idxs))
else:
    # Stacking - refaire
    X_meta = PROBA_MATRIX[:, best_idxs]
    p_best = np.zeros(len(y_true))
    meta_type = "LR" if "LR" in best_row["method"] else "MLP"

    for fold_id in range(N_SPLITS):
        tr = folds != fold_id
        te = folds == fold_id

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_meta[tr])
        X_te = scaler.transform(X_meta[te])

        if meta_type == "LR":
            clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300,
                              random_state=SEED, early_stopping=True)

        clf.fit(X_tr, y_true[tr])
        p_best[te] = clf.predict_proba(X_te)[:, 1]

# Save
best_thr = best_row["thr"]
y_pred_best = (p_best >= best_thr).astype(int)

# NPY
np.save(os.path.join(BEST_OUTPUT_PATH, f"proba_oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.npy"), p_best)

# CSV with details
df_best_oof = pd.DataFrame({
    "label": y_true,
    "fold": folds,
    "proba_ensemble": p_best,
    "pred_best_thr": y_pred_best,
})
df_best_oof.to_csv(os.path.join(BEST_OUTPUT_PATH, f"oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv"), index=False)

# Best-model config - with exact threshold
best_config = {
    "gold_annotator": GOLD_ANNOTATOR,
    "method": best_row["method"],
    "models": list(best_combo),
    "weights": [float(w) for w in best_row["weights"]] if best_row["weights"] is not None else None,
    "threshold": float(best_thr),
    "threshold_exact": f"{best_thr:.10f}",
    "mcc": float(best_row["mcc"]),
    "acc": float(best_row.get("acc", 0)),
    "bacc": float(best_row.get("bacc", 0)),
}

with open(os.path.join(BEST_OUTPUT_PATH, f"best_ensemble_config_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump(best_config, f, indent=2)

print(f"proba_oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.npy")
print(f"oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv")
print(f"best_ensemble_config_gold_{GOLD_ANNOTATOR}.json (threshold_exact: {best_thr:.10f})")
print(f"  -> {BEST_OUTPUT_PATH}")

# Best A1 analysis

In [ ]:
# =============================================================================
# DETAILED ANALYSIS OF THE BEST ENSEMBLE — GOLD=A1
# Run after the ensemble-search step
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef
)
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# =============================================================================
# CONFIG
# =============================================================================
GOLD_ANNOTATOR = "A1"

BASE_PATH = f"artifacts/oof_proba_final_gold_{GOLD_ANNOTATOR}/best_models"  # not shipped — see DATA.md
ENSEMBLE_PATH = f"artifacts/oof_proba_final_gold_{GOLD_ANNOTATOR}/ensemble_search_results"  # not shipped — see DATA.md
# Load the best-ensemble data
oof_df = pd.read_csv(f"{BASE_PATH}/oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv")

# Load the best-ensemble config
config_path = f"{BASE_PATH}/best_ensemble_config_gold_{GOLD_ANNOTATOR}.json"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        best_config = json.load(f)
else:
    best_config = {}

y_true = oof_df["label"].values
y_pred = oof_df["pred_best_thr"].values
p_best = oof_df["proba_ensemble"].values

# =============================================================================
# INFO ENSEMBLE
# =============================================================================
print("=" * 70)
print(f"BEST ENSEMBLE ANALYSIS — GOLD={GOLD_ANNOTATOR}")
print("=" * 70)

if best_config:
    print(f"\n📋 Configuration:")
    print(f"   Method:     {best_config.get('method', 'N/A')}")
    print(f"   Models:     {' + '.join(best_config.get('models', []))}")
    if best_config.get('weights'):
        weights_str = " / ".join([f"{w:.3f}" for w in best_config['weights']])
        print(f"   Weights:    [{weights_str}]")
    print(f"   Threshold:  {best_config.get('threshold', 'N/A')}")

# =============================================================================
# CONFUSION MATRIX
# =============================================================================
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
print(f"\n{'':15} Pred NON    Pred OUI")
print(f"{'True NON':15} {tn:8}      {fp:8}")
print(f"{'True OUI':15} {fn:8}      {tp:8}")

# =============================================================================
# GLOBAL METRICS
# =============================================================================
print("\n" + "=" * 70)
print("GLOBAL METRICS")
print("=" * 70)

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print(f"\nAccuracy:              {acc:.4f}  ({acc*100:.2f}%)")
print(f"Balanced Accuracy:     {bacc:.4f}  ({bacc*100:.2f}%)")
print(f"MCC (Matthews):        {mcc:.4f}")
print(f"F1 Macro:              {f1_macro:.4f}")
print(f"F1 Weighted:           {f1_weighted:.4f}")

# =============================================================================
# PER-CLASS METRICS
# =============================================================================
print("\n" + "=" * 70)
print("PER-CLASS METRICS")
print("=" * 70)

print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 54)

# Classe NON (0)
prec_non = precision_score(y_true, y_pred, pos_label=0)
rec_non = recall_score(y_true, y_pred, pos_label=0)
f1_non = f1_score(y_true, y_pred, pos_label=0)
support_non = np.sum(y_true == 0)
print(f"{'NON (0)':<12} {prec_non:>10.4f} {rec_non:>10.4f} {f1_non:>10.4f} {support_non:>10}")

# Classe OUI (1)
prec_oui = precision_score(y_true, y_pred, pos_label=1)
rec_oui = recall_score(y_true, y_pred, pos_label=1)
f1_oui = f1_score(y_true, y_pred, pos_label=1)
support_oui = np.sum(y_true == 1)
print(f"{'OUI (1)':<12} {prec_oui:>10.4f} {rec_oui:>10.4f} {f1_oui:>10.4f} {support_oui:>10}")

print("-" * 54)
print(f"{'Macro avg':<12} {(prec_non+prec_oui)/2:>10.4f} {(rec_non+rec_oui)/2:>10.4f} {f1_macro:>10.4f} {len(y_true):>10}")

# =============================================================================
# DETAILED STATISTICS
# =============================================================================
print("\n" + "=" * 70)
print("DETAILED STATISTICS")
print("=" * 70)

print(f"\n📊 Confusion Matrix Values:")
print(f"   True Positives (TP):   {tp:5d}  (true OUI)")
print(f"   True Negatives (TN):   {tn:5d}  (true NON)")
print(f"   False Positives (FP):  {fp:5d}  (false OUI)")
print(f"   False Negatives (FN):  {fn:5d}  (false NON)")

print(f"\n📈 Rates:")
print(f"   Sensitivity (TPR/Recall OUI): {tp/(tp+fn):.4f}  ({tp/(tp+fn)*100:.1f}%)")
print(f"   Specificity (TNR/Recall NON): {tn/(tn+fp):.4f}  ({tn/(tn+fp)*100:.1f}%)")
print(f"   Precision OUI (PPV):          {tp/(tp+fp):.4f}  ({tp/(tp+fp)*100:.1f}%)")
print(f"   Precision NON (NPV):          {tn/(tn+fn):.4f}  ({tn/(tn+fn)*100:.1f}%)")

print(f"\n📋 Distribution:")
print(f"   Total samples:     {len(y_true)}")
print(f"   Class NON (0):     {support_non} ({support_non/len(y_true)*100:.1f}%)")
print(f"   Class OUI (1):     {support_oui} ({support_oui/len(y_true)*100:.1f}%)")
if support_oui < support_non:
    print(f"   Imbalance ratio:   1:{support_non/support_oui:.2f}")
else:
    print(f"   Imbalance ratio:   {support_oui/support_non:.2f}:1")

# =============================================================================
# VISUALISATION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix absolue
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 18, 'weight': 'bold'})
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('True', fontsize=12)
ax1.set_title(f'Confusion Matrix\nBest Ensemble (MCC={mcc:.4f}) — Gold={GOLD_ANNOTATOR}', fontsize=14)

# Normalized confusion matrix
ax2 = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax2,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 16})
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('True', fontsize=12)
ax2.set_title('Normalized Matrix (by row)', fontsize=14)

plt.tight_layout()
fig_path = f'{BASE_PATH}/confusion_matrix_best_ensemble_gold_{GOLD_ANNOTATOR}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print(f"✓ Figure saved: {fig_path}")
print("=" * 70)

# =============================================================================
# PROBABILITY DISTRIBUTION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability histogram per class
ax1 = axes[0]
ax1.hist(p_best[y_true == 0], bins=50, alpha=0.7, label='NON (true)', color='blue')
ax1.hist(p_best[y_true == 1], bins=50, alpha=0.7, label='OUI (true)', color='red')
if best_config.get('threshold'):
    ax1.axvline(x=best_config['threshold'], color='green', linestyle='--', linewidth=2, label=f"Seuil={best_config['threshold']:.3f}")
ax1.set_xlabel('Predicted probability', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title(f'Probability distribution — Gold={GOLD_ANNOTATOR}', fontsize=14)
ax1.legend()

# Box plot
ax2 = axes[1]
data_box = [p_best[y_true == 0], p_best[y_true == 1]]
bp = ax2.boxplot(data_box, labels=['NON', 'OUI'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
if best_config.get('threshold'):
    ax2.axhline(y=best_config['threshold'], color='green', linestyle='--', linewidth=2, label=f"Seuil={best_config['threshold']:.3f}")
ax2.set_ylabel('Predicted probability', fontsize=12)
ax2.set_title('Distribution by class', fontsize=14)
ax2.legend()

plt.tight_layout()
fig_path2 = f'{BASE_PATH}/proba_distribution_best_ensemble_gold_{GOLD_ANNOTATOR}.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved: {fig_path2}")

# =============================================================================
# LaTeX summary (for the paper)
# =============================================================================
print("\n" + "=" * 70)
print("LATEX TABLE")
print("=" * 70)

models_str = " + ".join(best_config.get('models', ['?'])) if best_config else "?"

print(f"""
\\begin{{table}}[h]
\\centering
\\begin{{tabular}}{{lc}}
\\toprule
\\textbf{{Metric}} & \\textbf{{Value}} \\\\
\\midrule
Accuracy & {acc:.4f} \\\\
Balanced Accuracy & {bacc:.4f} \\\\
MCC & {mcc:.4f} \\\\
F1 Macro & {f1_macro:.4f} \\\\
F1 (NON) & {f1_non:.4f} \\\\
F1 (OUI) & {f1_oui:.4f} \\\\
\\bottomrule
\\end{{tabular}}
\\caption{{Performance du meilleur ensemble (Gold={GOLD_ANNOTATOR}): {best_config.get('method', '?')}}}
\\label{{tab:best_ensemble_metrics_{GOLD_ANNOTATOR.lower()}}}
\\end{{table}}
""")

# =============================================================================
# SAVE METRICS TO JSON
# =============================================================================
metrics_summary = {
    "gold_annotator": GOLD_ANNOTATOR,
    "ensemble_config": best_config,
    "metrics": {
        "accuracy": float(acc),
        "balanced_accuracy": float(bacc),
        "mcc": float(mcc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "f1_non": float(f1_non),
        "f1_oui": float(f1_oui),
        "precision_non": float(prec_non),
        "precision_oui": float(prec_oui),
        "recall_non": float(rec_non),
        "recall_oui": float(rec_oui),
    },
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    },
    "distribution": {
        "total": int(len(y_true)),
        "support_non": int(support_non),
        "support_oui": int(support_oui),
    }
}

metrics_path = f'{BASE_PATH}/metrics_summary_best_ensemble_gold_{GOLD_ANNOTATOR}.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\n✓ Metrics saved: {metrics_path}")
print("=" * 70)

# Best A2

In [ ]:
# ============================================================
# SCRIPT — PARALLEL ENSEMBLE SEARCH (GOLD=A2)
#
# Parallelized with joblib for fast execution
# KEEPS fine threshold grid (2000 points) with vectorized search
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from joblib import Parallel, delayed
import multiprocessing

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
# ============================================================
# GOLD ANNOTATOR
# ============================================================
GOLD_ANNOTATOR = "A2"

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)

# Paths for the gold_A2 results
OUTPUT_PATH = os.path.join(OUTPUT, f"oof_proba_final_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)

# Ensemble-search results
ENSEMBLE_OUTPUT = os.path.join(OUTPUT_PATH, "ensemble_search_results")
os.makedirs(ENSEMBLE_OUTPUT, exist_ok=True)

# Path for SAUL/LLaMA (separate script)
SAUL_LLAMA_PATH = os.path.join(BASE_PATH, "outputs", f"outputs_saul_lr_llama_mlp1_3cfg_gold_{GOLD_ANNOTATOR}")

# Path for TF-IDF
TFIDF_PATH = os.path.join(OUTPUT, f"tfidf_results_gold_{GOLD_ANNOTATOR}")

SEED = 42
N_SPLITS = 5
N_JOBS = -1

np.random.seed(SEED)

# ============================================================
# MODEL DEFINITIONS — Based on the gold_A2 results
# ============================================================
MODELS = {
    # From grid_search (5 models) - BEST for A2
    "CamemBERT": {
        "path": os.path.join(BEST_OUTPUT_PATH, "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy"),
        "source": "grid_search"
    },
    "CamemBERTav2": {
        "path": os.path.join(BEST_OUTPUT_PATH, "proba_oof_CamemBERTav2_BEST_cfg1_mean_last_MLP1.npy"),
        "source": "grid_search"
    },
    "JuriBERT": {
        "path": os.path.join(BEST_OUTPUT_PATH, "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_MLP1.npy"),
        "source": "grid_search"
    },
    "ST-MiniLM": {
        "path": os.path.join(BEST_OUTPUT_PATH, "proba_oof_ST-MiniLM_BEST_cfg3_cls_last_MLP1.npy"),
        "source": "grid_search"
    },
    "ST-MPNet": {
        "path": os.path.join(BEST_OUTPUT_PATH, "proba_oof_ST-MPNet_BEST_cfg3_cls_last_MLP2.npy"),
        "source": "grid_search"
    },
    # From SAUL/LLaMA script - BEST for A2
    "SAUL": {
        "path": os.path.join(SAUL_LLAMA_PATH, "proba_oof_SAUL-7B__cfg1__mean__last.npy"),
        "source": "saul_llama"
    },
    "LLaMA": {
        "path": os.path.join(SAUL_LLAMA_PATH, "proba_oof_LLaMA-3.1-8B__cfg3__mean__avg_last_k:4.npy"),
        "source": "saul_llama"
    },
}

# TF-IDF — best config for A2: cfg1 | uni_5k | LR | Scaler=False
TFIDF_BEST = {
    "path": os.path.join(TFIDF_PATH, "proba_oof_cfg1_uni_5k_LR_raw.npy"),
    "source": "tfidf"
}

# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print(f"LOADING DATA — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

import re

def load_labels():
    EXCEL_PATH = "DATA/outputs/benchmark.csv"
    df0 = pd.read_csv(EXCEL_PATH)
    cols = ["decision_id", "eval_A1", "eval_A2", "eval_A3"]
    df0 = df0[cols].copy()

    def extract(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
        return np.nan

    df0["a"] = df0["eval_A1"].apply(extract)
    df0["t"] = df0["eval_A2"].apply(extract)
    df0["s"] = df0["eval_A3"].apply(extract)

    # GOLD = A2 directement
    df0["label_str"] = df0["t"]
    df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
    df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

    # Folds
    rng = np.random.default_rng(SEED)
    groups = df0.groupby("decision_id").size().to_dict()
    uniq = list(groups.keys())
    rng.shuffle(uniq)
    uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
    loads = np.zeros(N_SPLITS, dtype=int)
    g2f = {}
    for g in uniq:
        f = int(loads.argmin())
        g2f[g] = f
        loads[f] += groups[g]
    df0["fold"] = df0["decision_id"].map(g2f).astype(int)

    return df0["label"].values.astype(int), df0["fold"].values

y_true, folds = load_labels()
print(f"  {len(y_true)} samples | oui={y_true.sum()} | non={len(y_true)-y_true.sum()}")

# Load probas
probas = {}

# Models .npy
for name, cfg in MODELS.items():
    path = cfg["path"]
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name}: {os.path.basename(path)}")
    else:
        print(f"  ✗ NOT FOUND: {name} at {path}")

# TF-IDF
if os.path.exists(TFIDF_BEST["path"]):
    probas["TF-IDF"] = np.load(TFIDF_BEST["path"])
    print(f"  ✓ TF-IDF: {os.path.basename(TFIDF_BEST['path'])}")
else:
    print(f"  ✗ NOT FOUND: TF-IDF at {TFIDF_BEST['path']}")

model_names = list(probas.keys())
n_models = len(model_names)
print(f"\n{n_models} models loaded")
print(f"CPU cores: {multiprocessing.cpu_count()}")

if n_models == 0:
    raise ValueError("No models loaded! Check paths.")

# Pre-stack all probas for fast access
PROBA_MATRIX = np.column_stack([probas[m] for m in model_names])
MODEL_IDX = {m: i for i, m in enumerate(model_names)}

# Precompute y_true as int32 for speed
Y_TRUE_INT = y_true.astype(np.int32)
P_TOTAL = int(Y_TRUE_INT.sum())
N_TOTAL = len(Y_TRUE_INT) - P_TOTAL

# ============================================================
# FAST METRICS - VECTORIZED
# ============================================================

# Fine grid: 2000 points from 0.15 to 0.85
THR_GRID = np.linspace(0.15, 0.85, 2000).astype(np.float64)

def fast_mcc(y_true, y_pred):
    """Fast MCC calculation."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))

    if den == 0:
        return 0.0
    return num / den

def best_threshold_fast(y_true, p_mix):
    """
    Vectorized threshold search over fine grid.
    Uses sorting-based approach for O(n log n) instead of O(n * k).
    """
    n = len(p_mix)

    # Sort by probability descending
    order = np.argsort(p_mix)[::-1]
    y_sorted = y_true[order]
    p_sorted = p_mix[order]

    # Cumulative sums for TP and FP as we lower threshold
    tp_cum = np.cumsum(y_sorted)
    fp_cum = np.cumsum(1 - y_sorted)

    # For each threshold, find how many samples are >= threshold
    k = np.searchsorted(-p_sorted, -THR_GRID, side='right')

    # Clip to valid indices
    k_clipped = np.clip(k - 1, 0, n - 1)

    # Get TP and FP at each threshold
    tp = np.where(k == 0, 0, tp_cum[k_clipped])
    fp = np.where(k == 0, 0, fp_cum[k_clipped])

    fn = P_TOTAL - tp
    tn = N_TOTAL - fp

    # MCC calculation (vectorized)
    num = tp * tn - fp * fn
    den_sq = (tp + fp).astype(np.float64) * (tp + fn) * (tn + fp) * (tn + fn)
    den = np.sqrt(den_sq)

    # Avoid division by zero
    mcc = np.where(den > 0, num / den, 0.0)

    best_idx = np.argmax(mcc)
    return float(THR_GRID[best_idx]), float(mcc[best_idx])

def compute_metrics_fast(y_true, y_pred):
    return {
        "acc": float(np.mean(y_true == y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(fast_mcc(y_true, y_pred)),
    }

# ============================================================
# PARALLEL WEIGHT OPTIMIZATION
# ============================================================

def optimize_2models(combo, y_true):
    idx1, idx2 = MODEL_IDX[combo[0]], MODEL_IDX[combo[1]]

    best = {"mcc": -1e9}
    for w1 in np.linspace(0, 1, 21):
        p_mix = w1 * PROBA_MATRIX[:, idx1] + (1 - w1) * PROBA_MATRIX[:, idx2]
        thr, mcc = best_threshold_fast(y_true, p_mix)
        if mcc > best["mcc"]:
            best = {"weights": (w1, 1 - w1), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def optimize_3models(combo, y_true):
    idxs = [MODEL_IDX[m] for m in combo]

    best = {"mcc": -1e9}
    for w1 in np.linspace(0, 1, 11):
        for w2 in np.linspace(0, 1 - w1, 11):
            w3 = 1 - w1 - w2
            if w3 < -1e-6:
                continue
            p_mix = (w1 * PROBA_MATRIX[:, idxs[0]] +
                     w2 * PROBA_MATRIX[:, idxs[1]] +
                     w3 * PROBA_MATRIX[:, idxs[2]])
            thr, mcc = best_threshold_fast(y_true, p_mix)
            if mcc > best["mcc"]:
                best = {"weights": (w1, w2, w3), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def optimize_nmodels(combo, y_true, n_samples=2500):
    """Random search for n>3 models."""
    idxs = [MODEL_IDX[m] for m in combo]
    n = len(combo)

    best = {"mcc": -1e9}
    rng = np.random.default_rng(SEED + hash(combo) % 10000)

    for _ in range(n_samples):
        weights = rng.dirichlet(np.ones(n))
        p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(weights, idxs))
        thr, mcc = best_threshold_fast(y_true, p_mix)
        if mcc > best["mcc"]:
            best = {"weights": tuple(weights), "thr": thr, "mcc": mcc}

    # Also try uniform weights
    weights = np.ones(n) / n
    p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(weights, idxs))
    thr, mcc = best_threshold_fast(y_true, p_mix)
    if mcc > best["mcc"]:
        best = {"weights": tuple(weights), "thr": thr, "mcc": mcc}

    return {"method": "WeightedAvg", "models": combo, **best}

def process_combo(combo, y_true):
    n = len(combo)
    if n == 2:
        return optimize_2models(combo, y_true)
    elif n == 3:
        return optimize_3models(combo, y_true)
    else:
        return optimize_nmodels(combo, y_true)

# ============================================================
# STACKING
# ============================================================

def stacking_combo(combo, y_true, folds, meta_type="LR"):
    idxs = [MODEL_IDX[m] for m in combo]
    X_meta = PROBA_MATRIX[:, idxs]
    n = len(y_true)

    p_stack = np.zeros(n)

    for fold_id in range(N_SPLITS):
        tr = folds != fold_id
        te = folds == fold_id

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_meta[tr])
        X_te = scaler.transform(X_meta[te])

        if meta_type == "LR":
            clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300,
                              random_state=SEED, early_stopping=True)

        clf.fit(X_tr, y_true[tr])
        p_stack[te] = clf.predict_proba(X_te)[:, 1]

    thr, mcc = best_threshold_fast(y_true, p_stack)

    return {
        "method": f"Stacking_{meta_type}",
        "models": combo,
        "weights": None,
        "thr": thr,
        "mcc": mcc,
    }

# ============================================================
# RANK FUSION
# ============================================================

def rank_fusion_combo(combo, y_true):
    idxs = [MODEL_IDX[m] for m in combo]
    n_samples = len(y_true)

    ranks = []
    for idx in idxs:
        r = np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1)
        ranks.append(r)

    p_rank = np.mean(ranks, axis=0)
    thr, mcc = best_threshold_fast(y_true, p_rank)

    return {
        "method": "RankFusion",
        "models": combo,
        "weights": None,
        "thr": thr,
        "mcc": mcc,
    }

# ============================================================
# MAIN PARALLEL EXECUTION
# ============================================================
print("\n" + "=" * 80)
print(f"PARALLEL ENSEMBLE SEARCH — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

all_results = []

# --- 1. Weighted Average ---
print("\n[1/3] Weighted Average Ensembles...")

all_combos = []
for n in range(2, n_models + 1):
    all_combos.extend(list(combinations(model_names, n)))

print(f"  Total combinations: {len(all_combos)}")

results_wa = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(process_combo)(combo, y_true) for combo in all_combos
)
all_results.extend(results_wa)
print(f"  {len(results_wa)} weighted avg ensembles done")

# --- 2. Stacking ---
print("\n[2/3] Stacking Ensembles...")

wa_df = pd.DataFrame(results_wa).sort_values("mcc", ascending=False)
top_combos_for_stacking = [tuple(row["models"]) for _, row in wa_df.head(50).iterrows()]

specific_combos = [
    ("SAUL", "LLaMA"),
    ("SAUL", "TF-IDF"),
    ("SAUL", "LLaMA", "TF-IDF"),
    ("SAUL", "LLaMA", "TF-IDF", "JuriBERT"),
    ("SAUL", "LLaMA", "TF-IDF", "ST-MPNet"),
    ("LLaMA", "ST-MPNet"),
    ("LLaMA", "ST-MPNet", "SAUL"),
    ("ST-MPNet", "TF-IDF"),
    ("ST-MPNet", "SAUL", "TF-IDF"),
    tuple(model_names),
]
for c in specific_combos:
    # Check that all models exist
    if all(m in model_names for m in c) and c not in top_combos_for_stacking:
        top_combos_for_stacking.append(c)

stacking_tasks = []
for combo in top_combos_for_stacking:
    stacking_tasks.append((combo, "LR"))
    stacking_tasks.append((combo, "MLP"))

results_stack = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(stacking_combo)(combo, y_true, folds, meta) for combo, meta in stacking_tasks
)
all_results.extend(results_stack)
print(f"  {len(results_stack)} stacking ensembles done")

# --- 3. Rank Fusion ---
print("\n[3/3] Rank Fusion...")

rank_combos = []
for n in range(2, n_models + 1):
    rank_combos.extend(list(combinations(model_names, n)))

results_rank = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(rank_fusion_combo)(combo, y_true) for combo in rank_combos
)
all_results.extend(results_rank)
print(f"  {len(results_rank)} rank fusion ensembles done")

# ============================================================
# FULL METRICS
# ============================================================
print("\n[4/4] Computing full metrics...")

def add_full_metrics(result):
    combo = result["models"]
    idxs = [MODEL_IDX[m] for m in combo]

    if result["method"] == "RankFusion":
        n_samples = len(y_true)
        ranks = [np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1) for idx in idxs]
        p_mix = np.mean(ranks, axis=0)
    elif result["weights"] is not None:
        p_mix = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(result["weights"], idxs))
    else:
        return result

    y_pred = (p_mix >= result["thr"]).astype(int)
    metrics = compute_metrics_fast(y_true, y_pred)
    result.update(metrics)
    return result

all_results = [add_full_metrics(r) for r in tqdm(all_results, desc="Full metrics")]

# ============================================================
# RESULTS
# ============================================================
print("\n" + "=" * 80)
print(f"RESULTS — GOLD={GOLD_ANNOTATOR}")
print("=" * 80)

results_df = pd.DataFrame(all_results)

for col in ["acc", "bacc", "f1"]:
    if col not in results_df.columns:
        results_df[col] = np.nan

results_df = results_df.sort_values("mcc", ascending=False).reset_index(drop=True)

print("\n=== TOP 25 OVERALL ===")
for i, row in results_df.head(25).iterrows():
    models_str = "+".join(row["models"])
    w_str = ""
    if row["weights"] is not None:
        w_str = f" [{'/'.join([f'{w:.2f}' for w in row['weights']])}]"
    print(f"{i+1:2d}. {row['method']:15s} MCC={row['mcc']:.4f} thr={row['thr']:.4f} | {models_str}{w_str}")

print("\n=== BEST BY METHOD ===")
for method in results_df["method"].unique():
    best = results_df[results_df["method"] == method].iloc[0]
    print(f"{method:15s}: MCC={best['mcc']:.4f} thr={best['thr']:.4f} | {'+'.join(best['models'])}")

print("\n=== BEST WEIGHTED AVG BY SIZE ===")
wa_df = results_df[results_df["method"] == "WeightedAvg"]
for n in range(2, n_models + 1):
    subset = wa_df[wa_df["models"].apply(len) == n]
    if len(subset) > 0:
        best = subset.iloc[0]
        w_str = "/".join([f"{w:.2f}" for w in best["weights"]])
        print(f"{n}-model: MCC={best['mcc']:.4f} thr={best['thr']:.4f} | {'+'.join(best['models'])} [{w_str}]")

# Baselines
print("\n=== VS BASELINES ===")
baseline_results = {}
for name in model_names:
    thr, mcc = best_threshold_fast(y_true, probas[name])
    y_pred_05 = (probas[name] >= 0.5).astype(int)
    mcc_05 = fast_mcc(y_true, y_pred_05)

    baseline_results[name] = {
        "thr_best": thr,
        "mcc_best": mcc,
        "mcc_05": mcc_05,
        "delta": mcc - mcc_05,
    }
    print(f"{name:15s}: MCC={mcc:.4f} (thr={thr:.4f}) | MCC@0.5={mcc_05:.4f} | delta={mcc - mcc_05:+.4f}")

# ============================================================
# SAVE
# ============================================================
print("\n" + "=" * 80)
print("SAVING")
print("=" * 80)

def serialize(row):
    return {
        "method": row["method"],
        "models": list(row["models"]),
        "weights": [float(w) for w in row["weights"]] if row["weights"] is not None else None,
        "threshold": float(row["thr"]) if "thr" in row else float(row.get("threshold", 0.5)),
        "mcc": float(row["mcc"]),
        "bacc": float(row["bacc"]) if pd.notna(row.get("bacc")) else None,
        "acc": float(row["acc"]) if pd.notna(row.get("acc")) else None,
        "f1": float(row["f1"]) if pd.notna(row.get("f1")) else None,
    }

with open(os.path.join(ENSEMBLE_OUTPUT, f"all_results_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump([serialize(row) for _, row in results_df.iterrows()], f, indent=2)

with open(os.path.join(ENSEMBLE_OUTPUT, f"top_100_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump([serialize(row) for _, row in results_df.head(100).iterrows()], f, indent=2)

csv_df = results_df.copy()
csv_df["models"] = csv_df["models"].apply(lambda x: "+".join(x))
csv_df["weights"] = csv_df["weights"].apply(
    lambda x: "/".join([f"{w:.3f}" for w in x]) if x is not None else "N/A"
)
csv_df["thr"] = csv_df["thr"].apply(lambda x: f"{x:.6f}")
csv_df.to_csv(os.path.join(ENSEMBLE_OUTPUT, f"all_results_gold_{GOLD_ANNOTATOR}.csv"), index=False)

# Best summary
best_summary = {
    "gold_annotator": GOLD_ANNOTATOR,
    "overall_best": serialize(results_df.iloc[0]),
    "baselines": baseline_results,
    "best_by_method": {m: serialize(results_df[results_df["method"] == m].iloc[0])
                       for m in results_df["method"].unique()},
    "threshold_grid": {
        "min": float(THR_GRID[0]),
        "max": float(THR_GRID[-1]),
        "n_points": len(THR_GRID),
    },
}

with open(os.path.join(ENSEMBLE_OUTPUT, f"best_summary_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump(best_summary, f, indent=2)

print(f"Saved to: {ENSEMBLE_OUTPUT}")

# Correlation matrix
print("\nGenerating correlation matrix...")
corr = np.corrcoef(PROBA_MATRIX.T)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(corr, index=model_names, columns=model_names),
            annot=True, fmt=".2f", cmap="RdYlBu_r", center=0.5, ax=ax)
ax.set_title(f"Model Probability Correlations (Gold={GOLD_ANNOTATOR})")
plt.tight_layout()
fig.savefig(os.path.join(ENSEMBLE_OUTPUT, f"correlations_gold_{GOLD_ANNOTATOR}.png"), dpi=150)
plt.close()
print(f"correlations_gold_{GOLD_ANNOTATOR}.png saved")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

# ============================================================
# SAVE BEST MODEL OOF PROBAS
# ============================================================
print("\nSaving best model OOF probas...")

best_row = results_df.iloc[0]
best_combo = best_row["models"]
best_idxs = [MODEL_IDX[m] for m in best_combo]

# Recompute the best-ensemble probabilities
if best_row["method"] == "RankFusion":
    n_samples = len(y_true)
    ranks = [np.argsort(np.argsort(PROBA_MATRIX[:, idx])) / (n_samples - 1) for idx in best_idxs]
    p_best = np.mean(ranks, axis=0)
elif best_row["weights"] is not None:
    p_best = sum(w * PROBA_MATRIX[:, idx] for w, idx in zip(best_row["weights"], best_idxs))
else:
    # Stacking - refaire
    X_meta = PROBA_MATRIX[:, best_idxs]
    p_best = np.zeros(len(y_true))
    meta_type = "LR" if "LR" in best_row["method"] else "MLP"

    for fold_id in range(N_SPLITS):
        tr = folds != fold_id
        te = folds == fold_id

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_meta[tr])
        X_te = scaler.transform(X_meta[te])

        if meta_type == "LR":
            clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300,
                              random_state=SEED, early_stopping=True)

        clf.fit(X_tr, y_true[tr])
        p_best[te] = clf.predict_proba(X_te)[:, 1]

# Save
best_thr = best_row["thr"]
y_pred_best = (p_best >= best_thr).astype(int)

# NPY
np.save(os.path.join(BEST_OUTPUT_PATH, f"proba_oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.npy"), p_best)

# CSV with details
df_best_oof = pd.DataFrame({
    "label": y_true,
    "fold": folds,
    "proba_ensemble": p_best,
    "pred_best_thr": y_pred_best,
})
df_best_oof.to_csv(os.path.join(BEST_OUTPUT_PATH, f"oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv"), index=False)

# Best-model config
best_config = {
    "gold_annotator": GOLD_ANNOTATOR,
    "method": best_row["method"],
    "models": list(best_combo),
    "weights": [float(w) for w in best_row["weights"]] if best_row["weights"] is not None else None,
    "threshold": float(best_thr),
    "mcc": float(best_row["mcc"]),
    "acc": float(best_row.get("acc", 0)),
    "bacc": float(best_row.get("bacc", 0)),
}

with open(os.path.join(BEST_OUTPUT_PATH, f"best_ensemble_config_gold_{GOLD_ANNOTATOR}.json"), "w") as f:
    json.dump(best_config, f, indent=2)

print(f"proba_oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.npy")
print(f"oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv")
print(f"best_ensemble_config_gold_{GOLD_ANNOTATOR}.json")
print(f"  -> {BEST_OUTPUT_PATH}")

# Best A2 analysis

In [ ]:
# =============================================================================
# DETAILED ANALYSIS OF THE BEST ENSEMBLE — GOLD=A1
# Run after the ensemble-search step
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef
)
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# =============================================================================
# CONFIG
# =============================================================================
GOLD_ANNOTATOR = "A2"

BASE_PATH = f"artifacts/oof_proba_final_gold_{GOLD_ANNOTATOR}/best_models"  # not shipped — see DATA.md
ENSEMBLE_PATH = f"artifacts/oof_proba_final_gold_{GOLD_ANNOTATOR}/ensemble_search_results"  # not shipped — see DATA.md
# Load the best-ensemble data
oof_df = pd.read_csv(f"{BASE_PATH}/oof_BEST_ENSEMBLE_gold_{GOLD_ANNOTATOR}.csv")

# Load the best-ensemble config
config_path = f"{BASE_PATH}/best_ensemble_config_gold_{GOLD_ANNOTATOR}.json"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        best_config = json.load(f)
else:
    best_config = {}

y_true = oof_df["label"].values
y_pred = oof_df["pred_best_thr"].values
p_best = oof_df["proba_ensemble"].values

# =============================================================================
# INFO ENSEMBLE
# =============================================================================
print("=" * 70)
print(f"BEST ENSEMBLE ANALYSIS — GOLD={GOLD_ANNOTATOR}")
print("=" * 70)

if best_config:
    print(f"\n📋 Configuration:")
    print(f"   Method:     {best_config.get('method', 'N/A')}")
    print(f"   Models:     {' + '.join(best_config.get('models', []))}")
    if best_config.get('weights'):
        weights_str = " / ".join([f"{w:.3f}" for w in best_config['weights']])
        print(f"   Weights:    [{weights_str}]")
    print(f"   Threshold:  {best_config.get('threshold', 'N/A')}")

# =============================================================================
# CONFUSION MATRIX
# =============================================================================
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
print(f"\n{'':15} Pred NON    Pred OUI")
print(f"{'True NON':15} {tn:8}      {fp:8}")
print(f"{'True OUI':15} {fn:8}      {tp:8}")

# =============================================================================
# GLOBAL METRICS
# =============================================================================
print("\n" + "=" * 70)
print("GLOBAL METRICS")
print("=" * 70)

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print(f"\nAccuracy:              {acc:.4f}  ({acc*100:.2f}%)")
print(f"Balanced Accuracy:     {bacc:.4f}  ({bacc*100:.2f}%)")
print(f"MCC (Matthews):        {mcc:.4f}")
print(f"F1 Macro:              {f1_macro:.4f}")
print(f"F1 Weighted:           {f1_weighted:.4f}")

# =============================================================================
# PER-CLASS METRICS
# =============================================================================
print("\n" + "=" * 70)
print("PER-CLASS METRICS")
print("=" * 70)

print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 54)

# Classe NON (0)
prec_non = precision_score(y_true, y_pred, pos_label=0)
rec_non = recall_score(y_true, y_pred, pos_label=0)
f1_non = f1_score(y_true, y_pred, pos_label=0)
support_non = np.sum(y_true == 0)
print(f"{'NON (0)':<12} {prec_non:>10.4f} {rec_non:>10.4f} {f1_non:>10.4f} {support_non:>10}")

# Classe OUI (1)
prec_oui = precision_score(y_true, y_pred, pos_label=1)
rec_oui = recall_score(y_true, y_pred, pos_label=1)
f1_oui = f1_score(y_true, y_pred, pos_label=1)
support_oui = np.sum(y_true == 1)
print(f"{'OUI (1)':<12} {prec_oui:>10.4f} {rec_oui:>10.4f} {f1_oui:>10.4f} {support_oui:>10}")

print("-" * 54)
print(f"{'Macro avg':<12} {(prec_non+prec_oui)/2:>10.4f} {(rec_non+rec_oui)/2:>10.4f} {f1_macro:>10.4f} {len(y_true):>10}")

# =============================================================================
# DETAILED STATISTICS
# =============================================================================
print("\n" + "=" * 70)
print("DETAILED STATISTICS")
print("=" * 70)

print(f"\n📊 Confusion Matrix Values:")
print(f"   True Positives (TP):   {tp:5d}  (true OUI)")
print(f"   True Negatives (TN):   {tn:5d}  (true NON)")
print(f"   False Positives (FP):  {fp:5d}  (false OUI)")
print(f"   False Negatives (FN):  {fn:5d}  (false NON)")

print(f"\n📈 Rates:")
print(f"   Sensitivity (TPR/Recall OUI): {tp/(tp+fn):.4f}  ({tp/(tp+fn)*100:.1f}%)")
print(f"   Specificity (TNR/Recall NON): {tn/(tn+fp):.4f}  ({tn/(tn+fp)*100:.1f}%)")
print(f"   Precision OUI (PPV):          {tp/(tp+fp):.4f}  ({tp/(tp+fp)*100:.1f}%)")
print(f"   Precision NON (NPV):          {tn/(tn+fn):.4f}  ({tn/(tn+fn)*100:.1f}%)")

print(f"\n📋 Distribution:")
print(f"   Total samples:     {len(y_true)}")
print(f"   Class NON (0):     {support_non} ({support_non/len(y_true)*100:.1f}%)")
print(f"   Class OUI (1):     {support_oui} ({support_oui/len(y_true)*100:.1f}%)")
if support_oui < support_non:
    print(f"   Imbalance ratio:   1:{support_non/support_oui:.2f}")
else:
    print(f"   Imbalance ratio:   {support_oui/support_non:.2f}:1")

# =============================================================================
# VISUALISATION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix absolue
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 18, 'weight': 'bold'})
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('True', fontsize=12)
ax1.set_title(f'Confusion Matrix\nBest Ensemble (MCC={mcc:.4f}) — Gold={GOLD_ANNOTATOR}', fontsize=14)

# Normalized confusion matrix
ax2 = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax2,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 16})
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('True', fontsize=12)
ax2.set_title('Normalized Matrix (by row)', fontsize=14)

plt.tight_layout()
fig_path = f'{BASE_PATH}/confusion_matrix_best_ensemble_gold_{GOLD_ANNOTATOR}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print(f"✓ Figure saved: {fig_path}")
print("=" * 70)

# =============================================================================
# PROBABILITY DISTRIBUTION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability histogram per class
ax1 = axes[0]
ax1.hist(p_best[y_true == 0], bins=50, alpha=0.7, label='NON (true)', color='blue')
ax1.hist(p_best[y_true == 1], bins=50, alpha=0.7, label='OUI (true)', color='red')
if best_config.get('threshold'):
    ax1.axvline(x=best_config['threshold'], color='green', linestyle='--', linewidth=2, label=f"Seuil={best_config['threshold']:.3f}")
ax1.set_xlabel('Predicted probability', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title(f'Probability distribution — Gold={GOLD_ANNOTATOR}', fontsize=14)
ax1.legend()

# Box plot
ax2 = axes[1]
data_box = [p_best[y_true == 0], p_best[y_true == 1]]
bp = ax2.boxplot(data_box, labels=['NON', 'OUI'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
if best_config.get('threshold'):
    ax2.axhline(y=best_config['threshold'], color='green', linestyle='--', linewidth=2, label=f"Seuil={best_config['threshold']:.3f}")
ax2.set_ylabel('Predicted probability', fontsize=12)
ax2.set_title('Distribution by class', fontsize=14)
ax2.legend()

plt.tight_layout()
fig_path2 = f'{BASE_PATH}/proba_distribution_best_ensemble_gold_{GOLD_ANNOTATOR}.png'
plt.savefig(fig_path2, dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved: {fig_path2}")

# =============================================================================
# LaTeX summary (for the paper)
# =============================================================================
print("\n" + "=" * 70)
print("LATEX TABLE")
print("=" * 70)

models_str = " + ".join(best_config.get('models', ['?'])) if best_config else "?"

print(f"""
\\begin{{table}}[h]
\\centering
\\begin{{tabular}}{{lc}}
\\toprule
\\textbf{{Metric}} & \\textbf{{Value}} \\\\
\\midrule
Accuracy & {acc:.4f} \\\\
Balanced Accuracy & {bacc:.4f} \\\\
MCC & {mcc:.4f} \\\\
F1 Macro & {f1_macro:.4f} \\\\
F1 (NON) & {f1_non:.4f} \\\\
F1 (OUI) & {f1_oui:.4f} \\\\
\\bottomrule
\\end{{tabular}}
\\caption{{Performance du meilleur ensemble (Gold={GOLD_ANNOTATOR}): {best_config.get('method', '?')}}}
\\label{{tab:best_ensemble_metrics_{GOLD_ANNOTATOR.lower()}}}
\\end{{table}}
""")

# =============================================================================
# SAVE METRICS TO JSON
# =============================================================================
metrics_summary = {
    "gold_annotator": GOLD_ANNOTATOR,
    "ensemble_config": best_config,
    "metrics": {
        "accuracy": float(acc),
        "balanced_accuracy": float(bacc),
        "mcc": float(mcc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "f1_non": float(f1_non),
        "f1_oui": float(f1_oui),
        "precision_non": float(prec_non),
        "precision_oui": float(prec_oui),
        "recall_non": float(rec_non),
        "recall_oui": float(rec_oui),
    },
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    },
    "distribution": {
        "total": int(len(y_true)),
        "support_non": int(support_non),
        "support_oui": int(support_oui),
    }
}

metrics_path = f'{BASE_PATH}/metrics_summary_best_ensemble_gold_{GOLD_ANNOTATOR}.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\n✓ Metrics saved: {metrics_path}")
print("=" * 70)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


# Qwen 2.5-7B - A2

In [ ]:
# ============================================================
# SCRIPT: Active Learning with Qwen 2.5 7B (4-bit quantized)
# VERSION: 5-Fold Cross-Validation + (0-shot + few-shot NN)
#
# Model: Qwen/Qwen2.5-7B-Instruct
# Optimization: BitsAndBytes 4-bit (requires GPU)
#
# Strategies:
#   - 0-shot: k = 0
#   - few-shot: k in {2,4,8} via nearest neighbors
# ============================================================

import os
import re
import warnings
import gc
import hashlib
import json
from datetime import datetime
from collections import Counter

# Data handling
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_7b_4bit")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- MAJOR CHANGE HERE: Qwen 2.5 7B model ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Set your HF token here if the model is gated (Qwen is open, but keep it just in case)
HF_TOKEN = os.environ["HF_TOKEN"]

# CV / experiments
N_FOLDS = 5
K_VALUES = [0, 2, 4, 8]
MAX_NEW_TOKENS = 128  # Qwen is a bit more verbose, leave some margin
MAX_PROMPT_LEN = 4096 # Qwen handles up to 32k/128k, plenty of room

# Perf
EMBEDDING_BATCH_SIZE = 8 # Increased since 4-bit frees up VRAM
GENERATION_BATCH_SIZE = 4 # Qwen 7B est plus gros, attention au batch size

# ============================================================
# PROMPT (optimized for Qwen Instruct)
# ============================================================
# Qwen follows system instructions very well.
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication mais raisonne avant de répondre juste "oui" ou "non".
""".strip()



# ############################################################
# GPU helpers
# ############################################################
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ############################################################
# PARTIE 2 : DATA
# ############################################################
def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text", "eval_A2"]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["label_str"] = df["eval_A2"].apply(extract_oui_non)
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)
    df = df.reset_index(drop=True)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} | Non: {(df['label']==0).sum()}")
    return df


def create_text_for_embedding(row):
    # Simple format to compute similarity
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"


def create_decision_folds(df, n_folds=N_FOLDS, seed=SEED):
    decision_ids = df["decision_id"].dropna().unique()
    np.random.seed(seed)
    np.random.shuffle(decision_ids)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for train_idx, test_idx in kf.split(decision_ids):
        folds.append((set(decision_ids[train_idx]), set(decision_ids[test_idx])))
    return folds


# ############################################################
# PART 3: EMBEDDINGS (with 4-bit handling)
# ############################################################
def get_cache_key(texts, model_name):
    if not texts: return "empty"
    sample = texts[0][:50] + texts[-1][:50]
    h = hashlib.md5(f"{model_name}_{len(texts)}_{sample}".encode()).hexdigest()
    return os.path.join(CACHE_PATH, f"emb_{h}.npy")

def compute_embeddings_cached(texts, model, tokenizer, batch_size=EMBEDDING_BATCH_SIZE):
    cache_path = get_cache_key(texts, MODEL_NAME)
    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing {len(texts)} embeddings...")
    all_embeddings = []

    # Needed to ensure the pad token is set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)

            # With 4-bit, output_hidden_states usually works fine
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)

            # Take the last hidden layer
            hidden = outputs.hidden_states[-1]

            # Mean pooling with attention mask (standard)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            # Explicit float32 conversion to avoid 4-bit precision issues during the sum
            hidden = hidden.float()
            embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

            all_embeddings.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embeddings)
    np.save(cache_path, result)
    return result


# ############################################################
# PART 4: KNN SELECTION
# ############################################################
def select_similar_examples(query_emb, pool_embs, pool_df, k):
    if k == 0: return None
    sims = cosine_similarity(query_emb.reshape(1, -1), pool_embs)[0]

    # Balanced strategy
    k_half = k // 2
    idx_oui = np.where(pool_df["label"].values == 1)[0]
    idx_non = np.where(pool_df["label"].values == 0)[0]

    top_oui = sorted([(i, sims[i]) for i in idx_oui], key=lambda x: -x[1])[:k_half + (k%2)]
    top_non = sorted([(i, sims[i]) for i in idx_non], key=lambda x: -x[1])[:k_half]

    selected_indices = [x[0] for x in top_oui] + [x[0] for x in top_non]

    # Fallback if not enough examples in a class
    if len(selected_indices) < k:
        all_sorted = np.argsort(sims)[::-1]
        for idx in all_sorted:
            if idx not in selected_indices:
                selected_indices.append(idx)
            if len(selected_indices) >= k: break

    return pool_df.iloc[selected_indices].copy()


# ############################################################
# PARTIE 5 : PROMPT BUILDER
# ############################################################
def build_prompt(row, examples_df, grille=GRILLE_ANNOTATION):
    # ChatML format for Qwen
    messages = [{"role": "system", "content": grille}]

    user_content = ""

    if examples_df is not None and not examples_df.empty:
        user_content += "Voici quelques exemples correctement analysés :\n\n"
        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            user_content += f"--- Exemple {i} ---\n"
            user_content += f"Article: {str(ex.get('article_text', ''))[:300]}...\n"
            user_content += f"Extrait: {str(ex.get('text', ''))[:400]}...\n"
            user_content += f"Réponse: {label}\n\n"

    user_content += "--- CAS À TRAITER ---\n"
    user_content += f"Article: {str(row.get('article_text','')).strip()}\n"
    user_content += f"Extrait: {str(row.get('text','')).strip()}\n\n"
    user_content += "Réponse (oui/non) :"

    messages.append({"role": "user", "content": user_content})
    return messages


# ############################################################
# PARTIE 6 : GENERATION
# ############################################################
def generate_batch(model, tokenizer, prompt_messages_list, max_new_tokens=MAX_NEW_TOKENS):
    # Conversion en template chat
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in prompt_messages_list
    ]

    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # Deterministic (greedy)
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )

    results = []
    # Decode only the newly generated part
    for i in range(len(texts)):
        input_len = inputs["input_ids"][i].shape[0]
        # Attention au padding: il faut slicer correctement
        # Simple trick: decode everything and split on the generation header if needed,
        # but here transformers handles return_full_text=False well in pipeline,
        # en manuel on slice :
        generated_ids = outputs[i][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        results.append(decoded.strip())

    return results

def parse_response(response_text):
    text = str(response_text).lower().strip()
    # Basic punctuation cleanup
    text_clean = re.sub(r"[^\w\s]", "", text)

    if text_clean.startswith("oui") or text_clean == "oui": return (1, "oui_start")
    if text_clean.startswith("non") or text_clean == "non": return (0, "non_start")

    # Search the text in case the model rambled
    words = text_clean.split()
    if "oui" in words and "non" not in words: return (1, "oui_unique")
    if "non" in words and "oui" not in words: return (0, "non_unique")

    return (-1, "invalide")


# ############################################################
# PARTIE 7 : MAIN EXECUTION
# ############################################################
def main():
    print("=" * 60)
    print(f"ACTIVE LEARNING - {MODEL_NAME}")
    print("Configuration: BitsAndBytes 4-bit Quantization")
    print("=" * 60)

    # 1. Load Data
    df = load_and_prepare_data(EXCEL_PATH)

    # 2. Load Model & Tokenizer (4-BIT CONFIG)
    print(f"\n[2] Loading model {MODEL_NAME} in 4-bit...")

    # BITSANDBYTES CONFIGURATION (the crucial part for the A10)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left" # Important for batch generation

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,  # <-- Application de la quantification
        device_map="auto",
        trust_remote_code=True
    )

    print("    ✓ Model loaded.")

    # 3. Embeddings (needed if k > 0)
    embeddings_all = None
    if any(k > 0 for k in K_VALUES):
        print("\n[3] Computing embeddings...")
        texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
        embeddings_all = compute_embeddings_cached(texts_all, model, tokenizer)

    # 4. Folds & Loop
    folds = create_decision_folds(df)
    all_results = []

    for fold_idx, (train_ids, test_ids) in enumerate(folds):
        print(f"\n--- FOLD {fold_idx+1}/{N_FOLDS} ---")

        train_mask = df["decision_id"].isin(train_ids)
        test_mask = df["decision_id"].isin(test_ids)
        df_train, df_test = df[train_mask], df[test_mask]

        emb_train = embeddings_all[train_mask.values] if embeddings_all is not None else None
        emb_test = embeddings_all[test_mask.values] if embeddings_all is not None else None

        for k in K_VALUES:
            config_name = f"fold{fold_idx+1}_k{k}"
            print(f"    Config: {config_name}")

            prompts_list = []
            for i in range(len(df_test)):
                row = df_test.iloc[i]
                ex_df = None
                if k > 0:
                    ex_df = select_similar_examples(emb_test[i], emb_train, df_train, k)
                prompts_list.append(build_prompt(row, ex_df))

            # Batch Generation
            all_preds = []
            # Batch by GENERATION_BATCH_SIZE
            for i in tqdm(range(0, len(prompts_list), GENERATION_BATCH_SIZE), desc="Gen"):
                batch_prompts = prompts_list[i:i+GENERATION_BATCH_SIZE]
                batch_responses = generate_batch(model, tokenizer, batch_prompts)

                for resp in batch_responses:
                    val, cat = parse_response(resp)
                    all_preds.append(val)

            # Eval
            y_true = df_test["label"].values
            res = evaluate_predictions(y_true, all_preds) # uses the earlier function
            res.update({"fold": fold_idx+1, "k": k, "config": config_name})
            all_results.append(res)
            print(f"        -> Acc: {res.get('acc', 0):.3f} | MCC: {res.get('mcc', 0):.3f}")

    # 5. Save Summary
    final_df = pd.DataFrame(all_results)
    summary = final_df.groupby("k")[["acc", "mcc", "f1_macro"]].mean()
    print("\n=== FINAL RESULTS (Mean) ===")
    print(summary)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    final_df.to_excel(os.path.join(OUTPUT_PATH, f"results_qwen7b_{timestamp}.xlsx"))

def evaluate_predictions(y_true, y_pred):
    # Small helper to avoid an error if the import section is missing
    y_pred_clean = [y if y >= 0 else 0 for y in y_pred] # treat invalid as 0, or filter
    valid_mask = [y >= 0 for y in y_pred]

    if sum(valid_mask) < 5: return {"acc": 0, "mcc": 0}

    # Compute on valid ones only for the metrics
    yt = np.array(y_true)[valid_mask]
    yp = np.array(y_pred)[valid_mask]

    return {
        "acc": accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "n_valid": sum(valid_mask)
    }

if __name__ == "__main__":
    main()

# Qwen 2.5-7B GOLD

In [ ]:
# ============================================================
# SCRIPT: Active Learning with Qwen 2.5 7B (4-bit quantized)
# VERSION: 5-Fold Cross-Validation + (0-shot + few-shot NN)
#
# Model: Qwen/Qwen2.5-7B-Instruct
# Optimization: BitsAndBytes 4-bit (requires GPU)
#
# Strategies:
#   - 0-shot: k = 0
#   - few-shot: k in {2,4,8} via nearest neighbors
# ============================================================

import os
import re
import warnings
import gc
import hashlib
import json
from datetime import datetime
from collections import Counter

# Data handling
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_7b_4bit")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- MAJOR CHANGE HERE: Qwen 2.5 7B model ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Set your HF token here if the model is gated (Qwen is open, but keep it just in case)
HF_TOKEN = os.environ["HF_TOKEN"]

# CV / experiments
N_FOLDS = 5
K_VALUES = [0, 2, 4, 8]
MAX_NEW_TOKENS = 128  # Qwen is a bit more verbose, leave some margin
MAX_PROMPT_LEN = 4096 # Qwen handles up to 32k/128k, plenty of room

# Perf
EMBEDDING_BATCH_SIZE = 8 # Increased since 4-bit frees up VRAM
GENERATION_BATCH_SIZE = 4 # Qwen 7B est plus gros, attention au batch size

# ============================================================
# PROMPT (optimized for Qwen Instruct)
# ============================================================
# Qwen follows system instructions very well.
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication mais raisonne avant de répondre juste "oui" ou "non".
""".strip()



# ############################################################
# GPU helpers
# ############################################################
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ############################################################
# PARTIE 2 : DATA
# ############################################################
def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    def extract_oui_non(x):
        if pd.isna(x):
            return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
            return "oui"
        if re.search(r"\bnon\b", s):
            return "non"
        return np.nan

    # normalize the 3 annotators
    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)

    # resolved gold: agreement -> (A1/A2), disagreement -> A3
    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])

    # keep only rows where the resolved gold is defined
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)
    df = df.reset_index(drop=True)

    # (useful for debugging)
    n_agree = int(same_mask.loc[df.index].sum()) if len(df) else 0  # safe
    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} | Non: {(df['label']==0).sum()}")
    return df



def create_text_for_embedding(row):
    # Simple format to compute similarity
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"


def create_decision_folds(df, n_folds=N_FOLDS, seed=SEED):
    decision_ids = df["decision_id"].dropna().unique()
    np.random.seed(seed)
    np.random.shuffle(decision_ids)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for train_idx, test_idx in kf.split(decision_ids):
        folds.append((set(decision_ids[train_idx]), set(decision_ids[test_idx])))
    return folds


# ############################################################
# PART 3: EMBEDDINGS (with 4-bit handling)
# ############################################################
def get_cache_key(texts, model_name):
    if not texts: return "empty"
    sample = texts[0][:50] + texts[-1][:50]
    h = hashlib.md5(f"{model_name}_{len(texts)}_{sample}".encode()).hexdigest()
    return os.path.join(CACHE_PATH, f"emb_{h}.npy")

def compute_embeddings_cached(texts, model, tokenizer, batch_size=EMBEDDING_BATCH_SIZE):
    cache_path = get_cache_key(texts, MODEL_NAME)
    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing {len(texts)} embeddings...")
    all_embeddings = []

    # Needed to ensure the pad token is set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)

            # With 4-bit, output_hidden_states usually works fine
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)

            # Take the last hidden layer
            hidden = outputs.hidden_states[-1]

            # Mean pooling with attention mask (standard)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            # Explicit float32 conversion to avoid 4-bit precision issues during the sum
            hidden = hidden.float()
            embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

            all_embeddings.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embeddings)
    np.save(cache_path, result)
    return result


# ############################################################
# PART 4: KNN SELECTION
# ############################################################
def select_similar_examples(query_emb, pool_embs, pool_df, k):
    if k == 0: return None
    sims = cosine_similarity(query_emb.reshape(1, -1), pool_embs)[0]

    # Balanced strategy
    k_half = k // 2
    idx_oui = np.where(pool_df["label"].values == 1)[0]
    idx_non = np.where(pool_df["label"].values == 0)[0]

    top_oui = sorted([(i, sims[i]) for i in idx_oui], key=lambda x: -x[1])[:k_half + (k%2)]
    top_non = sorted([(i, sims[i]) for i in idx_non], key=lambda x: -x[1])[:k_half]

    selected_indices = [x[0] for x in top_oui] + [x[0] for x in top_non]

    # Fallback if not enough examples in a class
    if len(selected_indices) < k:
        all_sorted = np.argsort(sims)[::-1]
        for idx in all_sorted:
            if idx not in selected_indices:
                selected_indices.append(idx)
            if len(selected_indices) >= k: break

    return pool_df.iloc[selected_indices].copy()


# ############################################################
# PARTIE 5 : PROMPT BUILDER
# ############################################################
def build_prompt(row, examples_df, grille=GRILLE_ANNOTATION):
    # ChatML format for Qwen
    messages = [{"role": "system", "content": grille}]

    user_content = ""

    if examples_df is not None and not examples_df.empty:
        user_content += "Voici quelques exemples correctement analysés :\n\n"
        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            user_content += f"--- Exemple {i} ---\n"
            user_content += f"Article: {str(ex.get('article_text', ''))[:300]}...\n"
            user_content += f"Extrait: {str(ex.get('text', ''))[:400]}...\n"
            user_content += f"Réponse: {label}\n\n"

    user_content += "--- CAS À TRAITER ---\n"
    user_content += f"Article: {str(row.get('article_text','')).strip()}\n"
    user_content += f"Extrait: {str(row.get('text','')).strip()}\n\n"
    user_content += "Réponse (oui/non) :"

    messages.append({"role": "user", "content": user_content})
    return messages


# ############################################################
# PARTIE 6 : GENERATION
# ############################################################
def generate_batch(model, tokenizer, prompt_messages_list, max_new_tokens=MAX_NEW_TOKENS):
    # Conversion en template chat
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in prompt_messages_list
    ]

    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # Deterministic (greedy)
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )

    results = []
    # Decode only the newly generated part
    for i in range(len(texts)):
        input_len = inputs["input_ids"][i].shape[0]
        # Attention au padding: il faut slicer correctement
        # Simple trick: decode everything and split on the generation header if needed,
        # but here transformers handles return_full_text=False well in pipeline,
        # en manuel on slice :
        generated_ids = outputs[i][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        results.append(decoded.strip())

    return results

def parse_response(response_text):
    text = str(response_text).lower().strip()
    # Basic punctuation cleanup
    text_clean = re.sub(r"[^\w\s]", "", text)

    if text_clean.startswith("oui") or text_clean == "oui": return (1, "oui_start")
    if text_clean.startswith("non") or text_clean == "non": return (0, "non_start")

    # Search the text in case the model rambled
    words = text_clean.split()
    if "oui" in words and "non" not in words: return (1, "oui_unique")
    if "non" in words and "oui" not in words: return (0, "non_unique")

    return (-1, "invalide")


# ############################################################
# PARTIE 7 : MAIN EXECUTION
# ############################################################
def main():
    print("=" * 60)
    print(f"ACTIVE LEARNING - {MODEL_NAME}")
    print("Configuration: BitsAndBytes 4-bit Quantization")
    print("=" * 60)

    # 1. Load Data
    df = load_and_prepare_data(EXCEL_PATH)

    # 2. Load Model & Tokenizer (4-BIT CONFIG)
    print(f"\n[2] Loading model {MODEL_NAME} in 4-bit...")

    # BITSANDBYTES CONFIGURATION (the crucial part for the A10)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left" # Important for batch generation

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,  # <-- Application de la quantification
        device_map="auto",
        trust_remote_code=True
    )

    print("    ✓ Model loaded.")

    # 3. Embeddings (needed if k > 0)
    embeddings_all = None
    if any(k > 0 for k in K_VALUES):
        print("\n[3] Computing embeddings...")
        texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
        embeddings_all = compute_embeddings_cached(texts_all, model, tokenizer)

    # 4. Folds & Loop
    folds = create_decision_folds(df)
    all_results = []

    for fold_idx, (train_ids, test_ids) in enumerate(folds):
        print(f"\n--- FOLD {fold_idx+1}/{N_FOLDS} ---")

        train_mask = df["decision_id"].isin(train_ids)
        test_mask = df["decision_id"].isin(test_ids)
        df_train, df_test = df[train_mask], df[test_mask]

        emb_train = embeddings_all[train_mask.values] if embeddings_all is not None else None
        emb_test = embeddings_all[test_mask.values] if embeddings_all is not None else None

        for k in K_VALUES:
            config_name = f"fold{fold_idx+1}_k{k}"
            print(f"    Config: {config_name}")

            prompts_list = []
            for i in range(len(df_test)):
                row = df_test.iloc[i]
                ex_df = None
                if k > 0:
                    ex_df = select_similar_examples(emb_test[i], emb_train, df_train, k)
                prompts_list.append(build_prompt(row, ex_df))

            # Batch Generation
            all_preds = []
            # Batch by GENERATION_BATCH_SIZE
            for i in tqdm(range(0, len(prompts_list), GENERATION_BATCH_SIZE), desc="Gen"):
                batch_prompts = prompts_list[i:i+GENERATION_BATCH_SIZE]
                batch_responses = generate_batch(model, tokenizer, batch_prompts)

                for resp in batch_responses:
                    val, cat = parse_response(resp)
                    all_preds.append(val)

            # Eval
            y_true = df_test["label"].values
            res = evaluate_predictions(y_true, all_preds) # uses the earlier function
            res.update({"fold": fold_idx+1, "k": k, "config": config_name})
            all_results.append(res)
            print(f"        -> Acc: {res.get('acc', 0):.3f} | MCC: {res.get('mcc', 0):.3f}")

    # 5. Save Summary
    final_df = pd.DataFrame(all_results)
    summary = final_df.groupby("k")[["acc", "mcc", "f1_macro"]].mean()
    print("\n=== FINAL RESULTS (Mean) ===")
    print(summary)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    final_df.to_excel(os.path.join(OUTPUT_PATH, f"results_qwen7b_{timestamp}.xlsx"))

def evaluate_predictions(y_true, y_pred):
    # Small helper to avoid an error if the import section is missing
    y_pred_clean = [y if y >= 0 else 0 for y in y_pred] # treat invalid as 0, or filter
    valid_mask = [y >= 0 for y in y_pred]

    if sum(valid_mask) < 5: return {"acc": 0, "mcc": 0}

    # Compute on valid ones only for the metrics
    yt = np.array(y_true)[valid_mask]
    yp = np.array(y_pred)[valid_mask]

    return {
        "acc": accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "n_valid": sum(valid_mask)
    }

if __name__ == "__main__":
    main()

# Qwen 2.5-7B A1

In [ ]:
# ============================================================
# SCRIPT: Active Learning with Qwen 2.5 7B (4-bit quantized)
# VERSION: 5-Fold Cross-Validation + (0-shot + few-shot NN)
#
# Model: Qwen/Qwen2.5-7B-Instruct
# Optimization: BitsAndBytes 4-bit (requires GPU)
#
# Strategies:
#   - 0-shot: k = 0
#   - few-shot: k in {2,4,8} via nearest neighbors
# ============================================================

import os
import re
import warnings
import gc
import hashlib
import json
from datetime import datetime
from collections import Counter

# Data handling
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_7b_4bit")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- MAJOR CHANGE HERE: Qwen 2.5 7B model ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Set your HF token here if the model is gated (Qwen is open, but keep it just in case)
HF_TOKEN = os.environ["HF_TOKEN"]

# CV / experiments
N_FOLDS = 5
K_VALUES = [0, 2, 4, 8]
MAX_NEW_TOKENS = 128  # Qwen is a bit more verbose, leave some margin
MAX_PROMPT_LEN = 4096 # Qwen handles up to 32k/128k, plenty of room

# Perf
EMBEDDING_BATCH_SIZE = 8 # Increased since 4-bit frees up VRAM
GENERATION_BATCH_SIZE = 4 # Qwen 7B est plus gros, attention au batch size

# ============================================================
# PROMPT (optimized for Qwen Instruct)
# ============================================================
# Qwen follows system instructions very well.
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication mais raisonne avant de répondre juste "oui" ou "non".
""".strip()



# ############################################################
# GPU helpers
# ############################################################
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ############################################################
# PARTIE 2 : DATA
# ############################################################
def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text", "eval_A1"]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["label_str"] = df["eval_A1"].apply(extract_oui_non)
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)
    df = df.reset_index(drop=True)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} | Non: {(df['label']==0).sum()}")
    return df


def create_text_for_embedding(row):
    # Simple format to compute similarity
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"


def create_decision_folds(df, n_folds=N_FOLDS, seed=SEED):
    decision_ids = df["decision_id"].dropna().unique()
    np.random.seed(seed)
    np.random.shuffle(decision_ids)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for train_idx, test_idx in kf.split(decision_ids):
        folds.append((set(decision_ids[train_idx]), set(decision_ids[test_idx])))
    return folds


# ############################################################
# PART 3: EMBEDDINGS (with 4-bit handling)
# ############################################################
def get_cache_key(texts, model_name):
    if not texts: return "empty"
    sample = texts[0][:50] + texts[-1][:50]
    h = hashlib.md5(f"{model_name}_{len(texts)}_{sample}".encode()).hexdigest()
    return os.path.join(CACHE_PATH, f"emb_{h}.npy")

def compute_embeddings_cached(texts, model, tokenizer, batch_size=EMBEDDING_BATCH_SIZE):
    cache_path = get_cache_key(texts, MODEL_NAME)
    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing {len(texts)} embeddings...")
    all_embeddings = []

    # Needed to ensure the pad token is set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)

            # With 4-bit, output_hidden_states usually works fine
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)

            # Take the last hidden layer
            hidden = outputs.hidden_states[-1]

            # Mean pooling with attention mask (standard)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            # Explicit float32 conversion to avoid 4-bit precision issues during the sum
            hidden = hidden.float()
            embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

            all_embeddings.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embeddings)
    np.save(cache_path, result)
    return result


# ############################################################
# PART 4: KNN SELECTION
# ############################################################
def select_similar_examples(query_emb, pool_embs, pool_df, k):
    if k == 0: return None
    sims = cosine_similarity(query_emb.reshape(1, -1), pool_embs)[0]

    # Balanced strategy
    k_half = k // 2
    idx_oui = np.where(pool_df["label"].values == 1)[0]
    idx_non = np.where(pool_df["label"].values == 0)[0]

    top_oui = sorted([(i, sims[i]) for i in idx_oui], key=lambda x: -x[1])[:k_half + (k%2)]
    top_non = sorted([(i, sims[i]) for i in idx_non], key=lambda x: -x[1])[:k_half]

    selected_indices = [x[0] for x in top_oui] + [x[0] for x in top_non]

    # Fallback if not enough examples in a class
    if len(selected_indices) < k:
        all_sorted = np.argsort(sims)[::-1]
        for idx in all_sorted:
            if idx not in selected_indices:
                selected_indices.append(idx)
            if len(selected_indices) >= k: break

    return pool_df.iloc[selected_indices].copy()


# ############################################################
# PARTIE 5 : PROMPT BUILDER
# ############################################################
def build_prompt(row, examples_df, grille=GRILLE_ANNOTATION):
    # ChatML format for Qwen
    messages = [{"role": "system", "content": grille}]

    user_content = ""

    if examples_df is not None and not examples_df.empty:
        user_content += "Voici quelques exemples correctement analysés :\n\n"
        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            user_content += f"--- Exemple {i} ---\n"
            user_content += f"Article: {str(ex.get('article_text', ''))[:300]}...\n"
            user_content += f"Extrait: {str(ex.get('text', ''))[:400]}...\n"
            user_content += f"Réponse: {label}\n\n"

    user_content += "--- CAS À TRAITER ---\n"
    user_content += f"Article: {str(row.get('article_text','')).strip()}\n"
    user_content += f"Extrait: {str(row.get('text','')).strip()}\n\n"
    user_content += "Réponse (oui/non) :"

    messages.append({"role": "user", "content": user_content})
    return messages


# ############################################################
# PARTIE 6 : GENERATION
# ############################################################
def generate_batch(model, tokenizer, prompt_messages_list, max_new_tokens=MAX_NEW_TOKENS):
    # Conversion en template chat
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in prompt_messages_list
    ]

    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # Deterministic (greedy)
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )

    results = []
    # Decode only the newly generated part
    for i in range(len(texts)):
        input_len = inputs["input_ids"][i].shape[0]
        # Attention au padding: il faut slicer correctement
        # Simple trick: decode everything and split on the generation header if needed,
        # but here transformers handles return_full_text=False well in pipeline,
        # en manuel on slice :
        generated_ids = outputs[i][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        results.append(decoded.strip())

    return results

def parse_response(response_text):
    text = str(response_text).lower().strip()
    # Basic punctuation cleanup
    text_clean = re.sub(r"[^\w\s]", "", text)

    if text_clean.startswith("oui") or text_clean == "oui": return (1, "oui_start")
    if text_clean.startswith("non") or text_clean == "non": return (0, "non_start")

    # Search the text in case the model rambled
    words = text_clean.split()
    if "oui" in words and "non" not in words: return (1, "oui_unique")
    if "non" in words and "oui" not in words: return (0, "non_unique")

    return (-1, "invalide")


# ############################################################
# PARTIE 7 : MAIN EXECUTION
# ############################################################
def main():
    print("=" * 60)
    print(f"ACTIVE LEARNING - {MODEL_NAME}")
    print("Configuration: BitsAndBytes 4-bit Quantization")
    print("=" * 60)

    # 1. Load Data
    df = load_and_prepare_data(EXCEL_PATH)

    # 2. Load Model & Tokenizer (4-BIT CONFIG)
    print(f"\n[2] Loading model {MODEL_NAME} in 4-bit...")

    # BITSANDBYTES CONFIGURATION (the crucial part for the A10)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left" # Important for batch generation

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,  # <-- Application de la quantification
        device_map="auto",
        trust_remote_code=True
    )

    print("    ✓ Model loaded.")

    # 3. Embeddings (needed if k > 0)
    embeddings_all = None
    if any(k > 0 for k in K_VALUES):
        print("\n[3] Computing embeddings...")
        texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
        embeddings_all = compute_embeddings_cached(texts_all, model, tokenizer)

    # 4. Folds & Loop
    folds = create_decision_folds(df)
    all_results = []

    for fold_idx, (train_ids, test_ids) in enumerate(folds):
        print(f"\n--- FOLD {fold_idx+1}/{N_FOLDS} ---")

        train_mask = df["decision_id"].isin(train_ids)
        test_mask = df["decision_id"].isin(test_ids)
        df_train, df_test = df[train_mask], df[test_mask]

        emb_train = embeddings_all[train_mask.values] if embeddings_all is not None else None
        emb_test = embeddings_all[test_mask.values] if embeddings_all is not None else None

        for k in K_VALUES:
            config_name = f"fold{fold_idx+1}_k{k}"
            print(f"    Config: {config_name}")

            prompts_list = []
            for i in range(len(df_test)):
                row = df_test.iloc[i]
                ex_df = None
                if k > 0:
                    ex_df = select_similar_examples(emb_test[i], emb_train, df_train, k)
                prompts_list.append(build_prompt(row, ex_df))

            # Batch Generation
            all_preds = []
            # Batch by GENERATION_BATCH_SIZE
            for i in tqdm(range(0, len(prompts_list), GENERATION_BATCH_SIZE), desc="Gen"):
                batch_prompts = prompts_list[i:i+GENERATION_BATCH_SIZE]
                batch_responses = generate_batch(model, tokenizer, batch_prompts)

                for resp in batch_responses:
                    val, cat = parse_response(resp)
                    all_preds.append(val)

            # Eval
            y_true = df_test["label"].values
            res = evaluate_predictions(y_true, all_preds) # uses the earlier function
            res.update({"fold": fold_idx+1, "k": k, "config": config_name})
            all_results.append(res)
            print(f"        -> Acc: {res.get('acc', 0):.3f} | MCC: {res.get('mcc', 0):.3f}")

    # 5. Save Summary
    final_df = pd.DataFrame(all_results)
    summary = final_df.groupby("k")[["acc", "mcc", "f1_macro"]].mean()
    print("\n=== FINAL RESULTS (Mean) ===")
    print(summary)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    final_df.to_excel(os.path.join(OUTPUT_PATH, f"results_qwen7b_{timestamp}.xlsx"))

def evaluate_predictions(y_true, y_pred):
    # Small helper to avoid an error if the import section is missing
    y_pred_clean = [y if y >= 0 else 0 for y in y_pred] # treat invalid as 0, or filter
    valid_mask = [y >= 0 for y in y_pred]

    if sum(valid_mask) < 5: return {"acc": 0, "mcc": 0}

    # Compute on valid ones only for the metrics
    yt = np.array(y_true)[valid_mask]
    yp = np.array(y_pred)[valid_mask]

    return {
        "acc": accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "n_valid": sum(valid_mask)
    }

if __name__ == "__main__":
    main()

# Qwen 0 shot (A1, A2, Gold) 4 bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen 2.5 7B (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen_0shot_analysis")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 8
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Qwen_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME}...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Qwen_Pred"] = preds
    df["Qwen_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()


# Qwen 0 ATS float16

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen 2.5 7B (0-shot) — Metrics + "OOF" saves
# bf16 VERSION — aligned with the few-shot CV script
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen_0shot_analysis_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# Aligned with the CV script
BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Qwen bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Qwen_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Loading in bf16 (like the CV script)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Qwen_Pred"] = preds
    df["Qwen_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Qwen bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_qwen_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_qwen_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Qwen Gold Centroid

In [ ]:
# ============================================================
# SCRIPT: Active Learning with Qwen 2.5 7B (4-bit quantized)
# VERSION: "CENTRAL" STRATEGY (Prototypical Examples)
#
# Model: Qwen/Qwen2.5-7B-Instruct
# Optimisation: BitsAndBytes 4-bit
#
# Few-shot strategy:
#   - Compute the centroid (mean) of the OUI and NON classes in Train.
#   - Take the k examples closest to these centers.
#   - These examples are fixed for the whole fold (independent of the query).
# ============================================================

import os
import re
import warnings
import gc
import hashlib
import json
from datetime import datetime
from collections import Counter

# Data handling
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

# --- CHANGEMENT NOM DOSSIER SORTIE ---
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_central")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# CV / experiments
N_FOLDS = 5
# --- CHANGEMENT K VALUES ---
K_VALUES = [2, 4, 8] # dropped 0

MAX_NEW_TOKENS = 128
MAX_PROMPT_LEN = 4096

# Perf
EMBEDDING_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4

# ============================================================
# PROMPT
# ============================================================
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication mais raisonne avant de répondre juste "oui" ou "non".
""".strip()

# ############################################################
# GPU helpers
# ############################################################
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ############################################################
# PARTIE 2 : DATA
# ############################################################
def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)

    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])

    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)
    df = df.reset_index(drop=True)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} | Non: {(df['label']==0).sum()}")
    return df

def create_text_for_embedding(row):
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"

def create_decision_folds(df, n_folds=N_FOLDS, seed=SEED):
    decision_ids = df["decision_id"].dropna().unique()
    np.random.seed(seed)
    np.random.shuffle(decision_ids)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for train_idx, test_idx in kf.split(decision_ids):
        folds.append((set(decision_ids[train_idx]), set(decision_ids[test_idx])))
    return folds

# ############################################################
# PARTIE 3 : EMBEDDINGS (Cached)
# ############################################################
def get_cache_key(texts, model_name):
    if not texts: return "empty"
    sample = texts[0][:50] + texts[-1][:50]
    h = hashlib.md5(f"{model_name}_{len(texts)}_{sample}".encode()).hexdigest()
    return os.path.join(CACHE_PATH, f"emb_{h}.npy")

def compute_embeddings_cached(texts, model, tokenizer, batch_size=EMBEDDING_BATCH_SIZE):
    cache_path = get_cache_key(texts, MODEL_NAME)
    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing {len(texts)} embeddings...")
    all_embeddings = []
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
            hidden = outputs.hidden_states[-1].float() # float32 pour precision
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_embeddings.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embeddings)
    np.save(cache_path, result)
    return result

# ############################################################
# PART 4: CENTRAL SELECTION (CHANGE HERE)
# ############################################################
def get_central_examples(pool_df, pool_embs, k):
    """
    Select the k most representative examples (closest to their class centroid).
    """
    if k == 0: return None

    # Split indices by class
    idx_oui = np.where(pool_df["label"].values == 1)[0]
    idx_non = np.where(pool_df["label"].values == 0)[0]

    # Compute the centroids (mean of the vectors)
    centroid_oui = np.mean(pool_embs[idx_oui], axis=0).reshape(1, -1)
    centroid_non = np.mean(pool_embs[idx_non], axis=0).reshape(1, -1)

    # Compute similarity to the centroid
    # Flatten to get a 1D array of scores
    sims_oui = cosine_similarity(pool_embs[idx_oui], centroid_oui).flatten()
    sims_non = cosine_similarity(pool_embs[idx_non], centroid_non).flatten()

    # Select the tops
    k_half = k // 2
    # k%2 to handle odd numbers (adds 1 to Oui by default)
    k_oui = k_half + (k % 2)
    k_non = k_half

    # argsort returns ascending indices, take the tail ([::-1])
    top_local_idx_oui = np.argsort(sims_oui)[::-1][:k_oui]
    top_local_idx_non = np.argsort(sims_non)[::-1][:k_non]

    # Recover the global indices in pool_df
    final_indices_oui = idx_oui[top_local_idx_oui]
    final_indices_non = idx_non[top_local_idx_non]

    # Alternate the examples in the prompt (Oui, Non, Oui, Non...) to avoid a position bias
    # (Astuce Python : zip_longest ou simple liste)
    combined_indices = []
    max_len = max(len(final_indices_oui), len(final_indices_non))
    for i in range(max_len):
        if i < len(final_indices_oui): combined_indices.append(final_indices_oui[i])
        if i < len(final_indices_non): combined_indices.append(final_indices_non[i])

    return pool_df.iloc[combined_indices].copy()

# ############################################################
# PARTIE 5 : PROMPT BUILDER
# ############################################################
def build_prompt(row, examples_df, grille=GRILLE_ANNOTATION):
    messages = [{"role": "system", "content": grille}]
    user_content = ""

    if examples_df is not None and not examples_df.empty:
        user_content += "Voici quelques exemples de référence correctement analysés :\n\n"
        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            user_content += f"--- Exemple {i} ---\n"
            user_content += f"Article: {str(ex.get('article_text', ''))[:300]}...\n"
            user_content += f"Extrait: {str(ex.get('text', ''))[:400]}...\n"
            user_content += f"Réponse: {label}\n\n"

    user_content += "--- CAS À TRAITER ---\n"
    user_content += f"Article: {str(row.get('article_text','')).strip()}\n"
    user_content += f"Extrait: {str(row.get('text','')).strip()}\n\n"
    user_content += "Réponse (oui/non) :"

    messages.append({"role": "user", "content": user_content})
    return messages

# ############################################################
# PARTIE 6 : GENERATION
# ############################################################
def generate_batch(model, tokenizer, prompt_messages_list, max_new_tokens=MAX_NEW_TOKENS):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in prompt_messages_list
    ]
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    results = []
    for i in range(len(texts)):
        generated_ids = outputs[i][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        results.append(decoded.strip())
    return results

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text_clean = re.sub(r"[^\w\s]", "", text)
    if text_clean.startswith("oui") or text_clean == "oui": return (1, "oui_start")
    if text_clean.startswith("non") or text_clean == "non": return (0, "non_start")
    words = text_clean.split()
    if "oui" in words and "non" not in words: return (1, "oui_unique")
    if "non" in words and "oui" not in words: return (0, "non_unique")
    return (-1, "invalide")

def evaluate_predictions(y_true, y_pred):
    valid_mask = [y >= 0 for y in y_pred]
    if sum(valid_mask) < 5: return {"acc": 0, "mcc": 0}
    yt = np.array(y_true)[valid_mask]
    yp = np.array(y_pred)[valid_mask]
    return {
        "acc": accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "n_valid": sum(valid_mask)
    }

# ############################################################
# PARTIE 7 : MAIN
# ############################################################
def main():
    print("=" * 60)
    print(f"AL - {MODEL_NAME} - CENTRAL STRATEGY")
    print("=" * 60)

    # 1. Load Data
    df = load_and_prepare_data(EXCEL_PATH)

    # 2. Load Model 4-bit
    print(f"\n[2] Loading model (4-bit)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("    ✓ Model loaded.")

    # 3. Embeddings (always needed to compute the centroids)
    print("\n[3] Computing embeddings (needed for the centroids)...")
    texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
    embeddings_all = compute_embeddings_cached(texts_all, model, tokenizer)

    # 4. Folds & Loop
    folds = create_decision_folds(df)
    all_results = []

    for fold_idx, (train_ids, test_ids) in enumerate(folds):
        print(f"\n--- FOLD {fold_idx+1}/{N_FOLDS} ---")

        train_mask = df["decision_id"].isin(train_ids)
        test_mask = df["decision_id"].isin(test_ids)
        df_train, df_test = df[train_mask], df[test_mask]

        # Embeddings for this fold (to compute the train centroids)
        emb_train = embeddings_all[train_mask.values]

        for k in K_VALUES:
            config_name = f"fold{fold_idx+1}_k{k}_central"
            print(f"    Config: {config_name}")

            # --- KEY CHANGE: SINGLE SELECTION ---
            # Instead of selecting per query, select the central TRAIN examples once
            central_examples_df = get_central_examples(df_train, emb_train, k)

            # Build the prompts (they all share the same few-shot examples)
            prompts_list = []
            for i in range(len(df_test)):
                row = df_test.iloc[i]
                prompts_list.append(build_prompt(row, central_examples_df))

            # Batch Generation
            all_preds = []
            for i in tqdm(range(0, len(prompts_list), GENERATION_BATCH_SIZE), desc="Gen"):
                batch_prompts = prompts_list[i:i+GENERATION_BATCH_SIZE]
                batch_responses = generate_batch(model, tokenizer, batch_prompts)
                for resp in batch_responses:
                    val, cat = parse_response(resp)
                    all_preds.append(val)

            # Eval
            y_true = df_test["label"].values
            res = evaluate_predictions(y_true, all_preds)
            res.update({"fold": fold_idx+1, "k": k, "config": config_name})
            all_results.append(res)
            print(f"        -> Acc: {res.get('acc', 0):.3f} | MCC: {res.get('mcc', 0):.3f}")

    # 5. Save
    final_df = pd.DataFrame(all_results)
    summary = final_df.groupby("k")[["acc", "mcc", "f1_macro"]].mean()
    print("\n=== FINAL RESULTS (Central) ===")
    print(summary)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    # Updated file name
    final_df.to_excel(os.path.join(OUTPUT_PATH, f"results_qwen7b_central_{timestamp}.xlsx"))

if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# SCRIPT: Active Learning with Qwen 2.5 7B (4-bit quantized)
# VERSION: "CENTRAL" STRATEGY + FULL OOF SAVE
# ============================================================

import os
import re
import warnings
import gc
import hashlib
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

# Dossier de sortie
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_central_oof")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# Experiment parameters
N_FOLDS = 5
K_VALUES = [2, 4, 8]

MAX_NEW_TOKENS = 128
MAX_PROMPT_LEN = 4096

# Perf (Batch sizes)
EMBEDDING_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4

# ============================================================
# PROMPT
# ============================================================
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication mais raisonne avant de répondre juste "oui" ou "non".
""".strip()

# ############################################################
# GPU helpers
# ############################################################
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ############################################################
# PARTIE 2 : DATA & PREP
# ############################################################
def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    # Useful columns (keep the IDs for the OOF)
    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)

    # Majority vote / simple consensus
    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])

    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)
    # Create a unique ID for OOF tracking
    df = df.reset_index(drop=True)
    df["unique_id"] = df.index

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} | Non: {(df['label']==0).sum()}")
    return df

def create_text_for_embedding(row):
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"

def create_decision_folds(df, n_folds=N_FOLDS, seed=SEED):
    decision_ids = df["decision_id"].dropna().unique()
    np.random.seed(seed)
    np.random.shuffle(decision_ids)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for train_idx, test_idx in kf.split(decision_ids):
        folds.append((set(decision_ids[train_idx]), set(decision_ids[test_idx])))
    return folds

# ############################################################
# PARTIE 3 : EMBEDDINGS (Cached)
# ############################################################
def get_cache_key(texts, model_name):
    if not texts: return "empty"
    sample = texts[0][:50] + texts[-1][:50]
    h = hashlib.md5(f"{model_name}_{len(texts)}_{sample}".encode()).hexdigest()
    return os.path.join(CACHE_PATH, f"emb_{h}.npy")

def compute_embeddings_cached(texts, model, tokenizer, batch_size=EMBEDDING_BATCH_SIZE):
    cache_path = get_cache_key(texts, MODEL_NAME)
    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing {len(texts)} embeddings...")
    all_embeddings = []
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
            hidden = outputs.hidden_states[-1].float()
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            embeddings = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_embeddings.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embeddings)
    np.save(cache_path, result)
    return result

# ############################################################
# PART 4: CENTRAL SELECTION
# ############################################################
def get_central_examples(pool_df, pool_embs, k):
    if k == 0: return None
    idx_oui = np.where(pool_df["label"].values == 1)[0]
    idx_non = np.where(pool_df["label"].values == 0)[0]

    centroid_oui = np.mean(pool_embs[idx_oui], axis=0).reshape(1, -1)
    centroid_non = np.mean(pool_embs[idx_non], axis=0).reshape(1, -1)

    sims_oui = cosine_similarity(pool_embs[idx_oui], centroid_oui).flatten()
    sims_non = cosine_similarity(pool_embs[idx_non], centroid_non).flatten()

    k_half = k // 2
    k_oui = k_half + (k % 2)
    k_non = k_half

    top_local_idx_oui = np.argsort(sims_oui)[::-1][:k_oui]
    top_local_idx_non = np.argsort(sims_non)[::-1][:k_non]

    final_indices_oui = idx_oui[top_local_idx_oui]
    final_indices_non = idx_non[top_local_idx_non]

    combined_indices = []
    max_len = max(len(final_indices_oui), len(final_indices_non))
    for i in range(max_len):
        if i < len(final_indices_oui): combined_indices.append(final_indices_oui[i])
        if i < len(final_indices_non): combined_indices.append(final_indices_non[i])

    return pool_df.iloc[combined_indices].copy()

# ############################################################
# PARTIE 5 : PROMPT & GEN
# ############################################################
def build_prompt(row, examples_df, grille=GRILLE_ANNOTATION):
    messages = [{"role": "system", "content": grille}]
    user_content = ""

    if examples_df is not None and not examples_df.empty:
        user_content += "Voici quelques exemples de référence correctement analysés :\n\n"
        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            user_content += f"--- Exemple {i} ---\n"
            user_content += f"Article: {str(ex.get('article_text', ''))[:300]}...\n"
            user_content += f"Extrait: {str(ex.get('text', ''))[:400]}...\n"
            user_content += f"Réponse: {label}\n\n"

    user_content += "--- CAS À TRAITER ---\n"
    user_content += f"Article: {str(row.get('article_text','')).strip()}\n"
    user_content += f"Extrait: {str(row.get('text','')).strip()}\n\n"
    user_content += "Réponse (oui/non) :"

    messages.append({"role": "user", "content": user_content})
    return messages

def generate_batch(model, tokenizer, prompt_messages_list, max_new_tokens=MAX_NEW_TOKENS):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in prompt_messages_list
    ]
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_PROMPT_LEN, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    results = []
    for i in range(len(texts)):
        generated_ids = outputs[i][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        results.append(decoded.strip())
    return results

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text_clean = re.sub(r"[^\w\s]", "", text)
    if text_clean.startswith("oui") or text_clean == "oui": return (1, "oui_start")
    if text_clean.startswith("non") or text_clean == "non": return (0, "non_start")
    words = text_clean.split()
    if "oui" in words and "non" not in words: return (1, "oui_unique")
    if "non" in words and "oui" not in words: return (0, "non_unique")
    return (-1, "invalide")

def evaluate_predictions(y_true, y_pred):
    valid_mask = [y >= 0 for y in y_pred]
    if sum(valid_mask) < 5:
        return {"acc": 0, "mcc": 0, "precision": 0, "recall": 0, "f1_macro": 0, "n_valid": 0}

    yt = np.array(y_true)[valid_mask]
    yp = np.array(y_pred)[valid_mask]

    return {
        "acc": accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        # Precision/Recall focus on class 1 (OUI)
        "precision": precision_score(yt, yp, pos_label=1, zero_division=0),
        "recall": recall_score(yt, yp, pos_label=1, zero_division=0),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "n_valid": sum(valid_mask)
    }

# ############################################################
# PARTIE 7 : MAIN
# ############################################################
def main():
    print("=" * 60)
    print(f"AL - {MODEL_NAME} - CENTRAUX + SAUVEGARDE OOF")
    print("=" * 60)

    # 1. Load Data
    df = load_and_prepare_data(EXCEL_PATH)

    # 2. Load Model
    print(f"\n[2] Loading model (4-bit)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 3. Embeddings
    print("\n[3] Computing embeddings...")
    texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
    embeddings_all = compute_embeddings_cached(texts_all, model, tokenizer)

    # 4. Loop
    folds = create_decision_folds(df)

    # Lists to store global results and OOF details
    all_metrics = []
    all_oof_predictions = []

    for fold_idx, (train_ids, test_ids) in enumerate(folds):
        print(f"\n--- FOLD {fold_idx+1}/{N_FOLDS} ---")

        train_mask = df["decision_id"].isin(train_ids)
        test_mask = df["decision_id"].isin(test_ids)

        df_train = df[train_mask]
        df_test = df[test_mask].copy() # Copy pour ajout OOF

        emb_train = embeddings_all[train_mask.values]

        for k in K_VALUES:
            config_name = f"fold{fold_idx+1}_k{k}_central"
            print(f"    Config: {config_name}")

            # Few-shot selection
            central_examples_df = get_central_examples(df_train, emb_train, k)

            # Prompts
            prompts_list = []
            for i in range(len(df_test)):
                row = df_test.iloc[i]
                prompts_list.append(build_prompt(row, central_examples_df))

            # Generation
            preds_val = []
            preds_raw = []

            for i in tqdm(range(0, len(prompts_list), GENERATION_BATCH_SIZE), desc="Gen"):
                batch_prompts = prompts_list[i:i+GENERATION_BATCH_SIZE]
                batch_responses = generate_batch(model, tokenizer, batch_prompts)

                for resp in batch_responses:
                    val, cat = parse_response(resp)
                    preds_val.append(val)
                    preds_raw.append(resp)

            # Eval metrics
            y_true = df_test["label"].values
            res = evaluate_predictions(y_true, preds_val)
            res.update({"fold": fold_idx+1, "k": k, "config": config_name})
            all_metrics.append(res)

            print(f"        -> Acc: {res['acc']:.3f} | F1: {res['f1_macro']:.3f} | Prec: {res['precision']:.3f} | Rec: {res['recall']:.3f}")

            # --- SAUVEGARDE OOF ---
            # Add the info to a list for detailed saving
            df_test_oof = df_test.copy()
            df_test_oof["fold"] = fold_idx + 1
            df_test_oof["k"] = k
            df_test_oof["pred_label"] = preds_val
            df_test_oof["pred_raw"] = preds_raw
            df_test_oof["is_correct"] = (df_test_oof["label"] == df_test_oof["pred_label"])

            # Keep just the essentials for the light file
            cols_oof = ["unique_id", "decision_id", "fold", "k", "label", "pred_label", "is_correct", "text", "article_text", "pred_raw"]
            all_oof_predictions.append(df_test_oof[cols_oof])

    # 5. Save Final Results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # a) Metrics summary
    df_metrics = pd.DataFrame(all_metrics)
    summary = df_metrics.groupby("k")[["acc", "mcc", "f1_macro", "precision", "recall"]].mean()

    print("\n=== FINAL RESULTS (MEAN) ===")
    print(summary)

    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_central_{timestamp}.xlsx")
    df_metrics.to_excel(metrics_path, index=False)
    print(f"\n[SAVE] Global metrics saved to: {metrics_path}")

    # b) OOF (Out-Of-Fold) details - THE KEY FILE FOR ERROR ANALYSIS
    df_oof_final = pd.concat(all_oof_predictions, ignore_index=True)
    oof_path = os.path.join(OUTPUT_PATH, f"oof_predictions_central_{timestamp}.xlsx")
    df_oof_final.to_excel(oof_path, index=False)
    print(f"[SAVE] Detailed predictions (OOF) saved to: {oof_path}")

if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# EXTRA: Precision "OUI" (class 1) from the OOF file
# - Global
# - Par k
# - Par fold & k
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import precision_score, confusion_matrix

OOF_PATH = "artifacts/outputs_qwen2.5_central_oof/oof_predictions_central_20260110_1722.xlsx"  # not shipped — see DATA.md
# ^^^ set the right file here (the one just saved)

df = pd.read_excel(OOF_PATH)

# Cleanup: keep only valid predictions (0/1)
df = df[df["pred_label"].isin([0, 1]) & df["label"].isin([0, 1])].copy()

y_true = df["label"].astype(int).values
y_pred = df["pred_label"].astype(int).values

# 1) Global "oui" precision
prec_oui = precision_score(y_true, y_pred, pos_label=1, zero_division=0)

# Confusion matrix to check TP/FP
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

print("=== PRECISION 'OUI' (globale) ===")
print(f"Precision_oui = {prec_oui:.4f}  (TP={tp}, FP={fp}, TP/(TP+FP)={(tp/(tp+fp) if (tp+fp)>0 else 0):.4f})")
print()

# 2) "oui" precision by k
print("=== PRECISION 'OUI' par k ===")
for k, sub in df.groupby("k"):
    p = precision_score(sub["label"], sub["pred_label"], pos_label=1, zero_division=0)
    print(f"k={int(k)} -> Precision_oui = {p:.4f}  (n={len(sub)})")
print()

# 3) "oui" precision by fold & k (useful to see the variance)
print("=== PRECISION 'OUI' par fold & k ===")
for (fold, k), sub in df.groupby(["fold", "k"]):
    p = precision_score(sub["label"], sub["pred_label"], pos_label=1, zero_division=0)
    print(f"fold={int(fold)} | k={int(k)} -> Precision_oui = {p:.4f}  (n={len(sub)})")


# Qwen smart sampling

In [ ]:
# ============================================================
# SCRIPT: Active Learning - Robustness (multiple random splits)
# MODE: N rounds of (150 Train / rest Test)
# STRATEGIES: Centroids vs Similarity
# ============================================================

import os
import re
import warnings
import gc
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    matthews_corrcoef,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen2.5_robustesse")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42 # Seed de base
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# --- NEW PARAMETERS ---
N_ROUNDS = 4               # Nombre de fois qu'on tire 150 exemples au hasard
N_ANNOTATED = 150          # Taille du set d'entrainement fixe
K_VALUES = [0, 2, 4, 8]    # Few-shot
STRATEGIES = ["centroid", "similarity"]

MAX_NEW_TOKENS = 10
MAX_PROMPT_LEN = 4096
EMBEDDING_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4

# ============================================================
# PROMPT & UTILS
# ============================================================
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non".
- "oui" = l'article est implicitement appliqué.
- "non" = l'article n'est pas appliqué.

Ne donne aucune explication.
""".strip()

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)
    # Label extraction (same as previous versions)
    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)
    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    if "unique_id" not in df.columns:
        df["unique_id"] = df.index.astype(str)

    print(f"    Total valid rows: {len(df)}")
    return df.reset_index(drop=True)

def create_text_for_embedding(row):
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"

# ============================================================
# EMBEDDINGS (Cached)
# ============================================================
def get_embeddings(texts, model, tokenizer):
    content_hash = hashlib.md5((str(len(texts)) + texts[0] + texts[-1]).encode()).hexdigest()
    cache_file = os.path.join(CACHE_PATH, f"emb_{content_hash}.npy")

    if os.path.exists(cache_file):
        print("    -> Embeddings loaded from cache.")
        return np.load(cache_file)

    print("    -> Computing embeddings...")
    all_embs = []
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE)):
            batch = texts[i : i + EMBEDDING_BATCH_SIZE]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]
            mask = inputs["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
            embeddings = torch.sum(hidden * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
            all_embs.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embs)
    np.save(cache_file, result)
    return result

# ============================================================
# SELECTION STRATEGIES
# ============================================================
def get_examples_centroid(train_df, train_embs, k):
    if k == 0: return pd.DataFrame()
    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    centroid_oui = np.mean(train_embs[idx_oui], axis=0).reshape(1, -1)
    centroid_non = np.mean(train_embs[idx_non], axis=0).reshape(1, -1)

    sim_oui = cosine_similarity(train_embs[idx_oui], centroid_oui).flatten()
    sim_non = cosine_similarity(train_embs[idx_non], centroid_non).flatten()

    k_half = k // 2
    top_oui = idx_oui[np.argsort(sim_oui)[::-1][:k_half + (k%2)]]
    top_non = idx_non[np.argsort(sim_non)[::-1][:k_half]]

    indices = []
    for i in range(max(len(top_oui), len(top_non))):
        if i < len(top_oui): indices.append(top_oui[i])
        if i < len(top_non): indices.append(top_non[i])

    return train_df.iloc[indices].copy()

def get_examples_similarity(query_emb, train_df, train_embs, k):
    if k == 0: return pd.DataFrame()
    query_emb = query_emb.reshape(1, -1)
    sims = cosine_similarity(train_embs, query_emb).flatten()

    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    scores_oui = sims[idx_oui]
    scores_non = sims[idx_non]

    k_half = k // 2
    best_local_oui = np.argsort(scores_oui)[::-1][:k_half + (k%2)]
    best_local_non = np.argsort(scores_non)[::-1][:k_half]

    indices = np.concatenate([idx_oui[best_local_oui], idx_non[best_local_non]])
    return train_df.iloc[indices].copy()

# ============================================================
# GENERATION
# ============================================================
def build_prompt(row, examples_df):
    messages = [{"role": "system", "content": GRILLE_ANNOTATION}]
    user_txt = ""
    if not examples_df.empty:
        user_txt += "Exemples de référence :\n\n"
        for _, ex in examples_df.iterrows():
            lbl = "oui" if ex["label"] == 1 else "non"
            user_txt += f"Extrait: {str(ex['text'])[:300]}...\nArticle: {str(ex['article_text'])[:200]}...\nRéponse: {lbl}\n\n"
    user_txt += f"--- CAS À TRAITER ---\nExtrait: {str(row['text'])}\nArticle: {str(row['article_text'])}\n\nRéponse :"
    messages.append({"role": "user", "content": user_txt})
    return messages

def parse_output(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    if text.startswith("oui"): return 1
    if text.startswith("non"): return 0
    return -1

# ============================================================
# MAIN
# ============================================================
def main():
    # 1. Load Data & Model (ONCE)
    df = load_and_prepare_data(EXCEL_PATH)

    print("[2] Loading model...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN)

    print("[3] Computing embeddings (Global)...")
    texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
    embs_all = get_embeddings(texts_all, model, tokenizer)

    detailed_results = []

    # 2. BOUCLE PRINCIPALE (ROUNDS)
    print(f"\n[4] Launching {N_ROUNDS} validation rounds...")

    for round_idx in range(N_ROUNDS):
        current_seed = SEED + round_idx
        print(f"\n{'='*40}")
        print(f"ROUND {round_idx + 1}/{N_ROUNDS} (Seed={current_seed})")
        print(f"{'='*40}")

        # Different random split at each round
        train_df = df.sample(n=N_ANNOTATED, random_state=current_seed)
        test_df = df.drop(train_df.index)

        # Fetch the corresponding embeddings
        train_indices = [df.index.get_loc(i) for i in train_df.index]
        test_indices = [df.index.get_loc(i) for i in test_df.index]
        train_embs = embs_all[train_indices]
        test_embs = embs_all[test_indices]

        # Experimental loop (K / Strategy)
        for k in K_VALUES:
            for strategy in STRATEGIES:

                # Zero-shot does not depend on the train set, but redo it each round since the TEST set changes
                if k == 0 and strategy == "similarity": continue # Skip doublon

                run_name = f"Round{round_idx+1}_k{k}_{strategy}"
                if k == 0: run_name = f"Round{round_idx+1}_k0_zeroshot"

                print(f"  -> Config: {run_name}")

                # Precompute centroid (if applicable)
                fixed_examples = None
                if k > 0 and strategy == "centroid":
                    fixed_examples = get_examples_centroid(train_df, train_embs, k)

                # Prompt preparation
                prompts_data = []
                for i in range(len(test_df)):
                    row = test_df.iloc[i]
                    # Selection
                    if k == 0: examples = pd.DataFrame()
                    elif strategy == "centroid": examples = fixed_examples
                    else: examples = get_examples_similarity(test_embs[i], train_df, train_embs, k)

                    # Store
                    ex_ids = str(examples.index.tolist()) if not examples.empty else "[]"
                    prompts_data.append({"row": row, "msgs": build_prompt(row, examples), "ex_ids": ex_ids})

                # Generation
                texts_gen = [tokenizer.apply_chat_template(p["msgs"], tokenize=False, add_generation_prompt=True) for p in prompts_data]

                # Batch Loop
                current_preds = []
                for i in tqdm(range(0, len(texts_gen), GENERATION_BATCH_SIZE), desc="    Gen", leave=False):
                    batch = texts_gen[i:i+GENERATION_BATCH_SIZE]
                    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_PROMPT_LEN).to(model.device)
                    with torch.no_grad():
                        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id, do_sample=False)

                    for idx_b, seq in enumerate(out):
                        gen_txt = tokenizer.decode(seq[inputs["input_ids"][idx_b].shape[0]:], skip_special_tokens=True)
                        val = parse_output(gen_txt)

                        p_info = prompts_data[i + idx_b]
                        detailed_results.append({
                            "round_id": round_idx + 1,
                            "decision_id": p_info["row"].get("decision_id"),
                            "strategy": strategy if k > 0 else "zero_shot",
                            "k": k,
                            "true_label": p_info["row"]["label"],
                            "pred_label": val,
                            "raw_output": gen_txt,
                            "few_shot_ids": p_info["ex_ids"]
                        })

    # 3. REPORTING
    print("\n[5] Analyse et Sauvegarde...")
    res_df = pd.DataFrame(detailed_results)

    # Aggregate by Strategy/K/Round to get per-round scores
    round_stats = []
    for (strat, k_val, r_id), group in res_df.groupby(["strategy", "k", "round_id"]):
        valid = group[group["pred_label"] != -1]
        if len(valid) == 0: continue
        acc = accuracy_score(valid["true_label"], valid["pred_label"])
        mcc = matthews_corrcoef(valid["true_label"], valid["pred_label"])
        round_stats.append({"strategy": strat, "k": k_val, "round": r_id, "acc": acc, "mcc": mcc})

    stats_df = pd.DataFrame(round_stats)

    # Final summary (mean + std over the rounds)
    final_summary = stats_df.groupby(["strategy", "k"]).agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        mcc_mean=("mcc", "mean"),
        mcc_std=("mcc", "std")
    ).reset_index()

    print("\n=== MEAN RESULTS (over {} rounds) ===".format(N_ROUNDS))
    print(final_summary)

    # Save
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    outfile = os.path.join(OUTPUT_PATH, f"results_multisplit_{ts}.xlsx")
    with pd.ExcelWriter(outfile) as writer:
        final_summary.to_excel(writer, sheet_name="Global_Summary")
        stats_df.to_excel(writer, sheet_name="Per_Round_Stats")
        res_df.to_excel(writer, sheet_name="All_Predictions", index=False)

    print(f"\nSaved: {outfile}")

if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# SCRIPT: Post-analysis (focus on class OUI)
# INPUT: existing Excel file
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- CONFIGURATION ---
INPUT_FILE = "artifacts/outputs_qwen2.5_robustesse/results_multisplit_20260110_1934.xlsx"  # not shipped — see DATA.md
OUTPUT_FILE = INPUT_FILE.replace(".xlsx", "_ANALYSIS_OUI.xlsx")

def run_analysis_oui():
    print(f"[1] Reading file: {INPUT_FILE}")
    try:
        df = pd.read_excel(INPUT_FILE, sheet_name="All_Predictions")
    except Exception as e:
        print(f"ERROR: could not read the 'All_Predictions' sheet.\n{e}")
        return

    print(f"    -> {len(df)} rows loaded.")

    # Store the stats per round
    round_stats = []

    # Group by configuration
    # Reminder: label 1 = OUI, label 0 = NON
    groups = df.groupby(["strategy", "k", "round_id"])

    print("[2] Computing OUI-specific metrics (Class 1)...")

    for (strat, k_val, r_id), group in groups:
        # Exclude technical errors (-1)
        valid = group[group["pred_label"] != -1]

        if len(valid) == 0:
            continue

        y_true = valid["true_label"]
        y_pred = valid["pred_label"]

        # --- THIS IS THE KEY PART ---
        # pos_label=1 forces the computation on "OUI" only
        stats = {
            "strategy": strat,
            "k": k_val,
            "round": r_id,
            "nb_samples": len(valid),

            # Global metric
            "Accuracy": accuracy_score(y_true, y_pred),

            # Metrics on OUI (pos_label=1)
            "Precision_OUI": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "Recall_OUI": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            "F1_OUI": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        }
        round_stats.append(stats)

    # Build DataFrame
    df_rounds = pd.DataFrame(round_stats)

    # Compute means and standard deviations over the 4 rounds
    summary = df_rounds.groupby(["strategy", "k"]).agg({
        "Accuracy": ["mean", "std"],
        "Precision_OUI": ["mean", "std"],
        "Recall_OUI": ["mean", "std"],
        "F1_OUI": ["mean", "std"]
    })

    # Format the columns (e.g. Precision_OUI_mean)
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.reset_index()

    # Console output
    print("\n" + "="*90)
    print(" MEAN RESULTS ON THE 'OUI' CLASS")
    print("="*90)
    # Select the key columns for display
    cols_show = ["strategy", "k", "Precision_OUI_mean", "Recall_OUI_mean", "F1_OUI_mean", "Accuracy_mean"]
    # Round for readability
    print(summary[cols_show].round(4).to_string(index=False))

    # Save to Excel
    print(f"\n[3] Saving to: {OUTPUT_FILE}")
    with pd.ExcelWriter(OUTPUT_FILE) as writer:
        summary.to_excel(writer, sheet_name="Moyennes_OUI", index=False)
        df_rounds.to_excel(writer, sheet_name="Details_Par_Round", index=False)

    print("Done.")

if __name__ == "__main__":
    run_analysis_oui()

# LLaMa few close golden short

In [ ]:
# ============================================================
# SCRIPT — 5-FOLD OOF: 0-SHOT vs FEW-SHOT (Nearest Neighbors)
# Implicit application of the Civil Code (oui/non)
#
# - 5-fold CV split by decision_id (no intra-decision leakage)
# - Example retrieval (k-NN) ONLY within the 4 train folds
# - k ∈ {0, 2, 4, 8} (modifiable)
#
# Embeddings: LLaMA-3.1-8B (mean pooling, last layer)
# Generation: LLaMA-3.1-8B-Instruct
#
# Exports:
# - OOF predictions (par fold, par k)
# - Global metrics table (by k + by fold)
# - Figure MCC/BAcc/F1 vs k
# ============================================================

import os, re, gc, hashlib, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from datetime import datetime

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_fewshot_llama_cv_nn")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# CV
N_FOLDS = 5

# Few-shot
K_VALUES = [0, 2, 4, 8]  # tu peux mettre [0,2,4,8] comme tu dis
MAX_NEW_TOKENS = 20
GEN_BATCH_SIZE = 4

# Embeddings
EMB_MAX_LEN = 512
EMB_BATCH_SIZE = 16
USE_BF16 = True

# Models
LLAMA_MODEL_BASE = "meta-llama/Llama-3.1-8B"             # embeddings
LLAMA_MODEL_INSTRUCT = "meta-llama/Llama-3.1-8B-Instruct" # generation
HF_TOKEN = os.environ["HF_TOKEN"]          # <-- mets ton token

# Plot
DO_PLOTS = True


# ============================================================
# GRILLE (prompt court neutre)
# ============================================================

GRILLE_BINAIRE_NEUTRE = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()


# ============================================================
# HELPERS (cuda / parsing / texte embedding)
# ============================================================

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def extract_oui_non(x):
    """Extract oui/non from the human annotations (robust, conservative)."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


def make_text_for_embedding(df):
    """Embedding text = (article + chunk) pair with stable tags."""
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()
    return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()


def _hash_texts(texts, n=200):
    """Stable hash over a sample + total N (for cache)."""
    if len(texts) == 0:
        return "empty"
    sample = texts[:n] + texts[-n:] if len(texts) > 2 * n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


# ============================================================
# EMBEDDINGS LLaMA (mean pooling last layer) + cache
# ============================================================

def encode_with_llama(texts, model, tokenizer, batch_size=EMB_BATCH_SIZE, max_len=EMB_MAX_LEN, desc="Encoding"):
    device = next(model.parameters()).device
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]

            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16 and torch.cuda.is_available()):
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hidden = out.hidden_states[-1]  # (B, T, D)
            attn_mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)

            emb = (hidden * attn_mask).sum(dim=1) / attn_mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hidden, attn_mask, emb
            _clear_cuda()

    return np.vstack(all_embs)


def load_or_compute_embeddings(texts, model, tokenizer, cache_name):
    texts_hash = _hash_texts(texts)
    cache_path = os.path.join(CACHE_DIR, f"emb_{cache_name}_{texts_hash[:12]}.npy")

    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing embeddings ({cache_name})...")
    embeddings = encode_with_llama(texts, model, tokenizer, desc=f"    {cache_name}")
    np.save(cache_path, embeddings)
    print(f"    ✓ Saved: {cache_path}")
    return embeddings


# ============================================================
# NEAREST NEIGHBORS (balanced 50/50 si possible)
# ============================================================

def select_nearest_neighbors(df_pool, embeddings_pool, query_embedding, k, seed=SEED):
    """
    k-NN selection within df_pool (train folds), with class balancing.
    df_pool doit contenir 'label' (0/1).
    """
    if k == 0:
        return None

    rng = np.random.default_rng(seed)

    query_emb = query_embedding.reshape(1, -1)
    similarities = cosine_similarity(embeddings_pool, query_emb).flatten()

    k_per_class = k // 2
    selected_indices = []

    for label in [0, 1]:
        mask = (df_pool["label"].values == label)
        idxs = np.where(mask)[0]
        if len(idxs) == 0:
            continue

        sims_class = similarities[mask]
        top_k = min(k_per_class, len(sims_class))

        # argsort ascending -> take last top_k
        top_local = np.argsort(sims_class)[-top_k:]
        top_global = idxs[top_local]
        selected_indices.extend(top_global.tolist())

    # top up if not enough
    if len(selected_indices) < k:
        remaining = k - len(selected_indices)
        all_indices = np.arange(len(df_pool))
        available = np.setdiff1d(all_indices, np.array(selected_indices, dtype=int), assume_unique=False)

        if len(available) > 0:
            sims_available = similarities[available]
            take = min(remaining, len(available))
            top = np.argsort(sims_available)[-take:]
            selected_indices.extend(available[top].tolist())

    examples = df_pool.iloc[selected_indices].copy()
    # shuffle stable
    examples = examples.sample(frac=1, random_state=seed).reset_index(drop=True)
    return examples


# ============================================================
# PROMPTS
# ============================================================

def build_prompt(row, examples_df=None):
    parts = []
    parts.append(GRILLE_BINAIRE_NEUTRE)
    parts.append("\n" + "═" * 79 + "\n")

    if examples_df is not None and len(examples_df) > 0:
        parts.append("                            EXEMPLES\n")
        parts.append("═" * 79 + "\n")

        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            parts.append(f"--- Exemple {i} ---")
            parts.append(f"Article {ex.get('pred_art', 'N/A')}: {str(ex.get('article_text', ''))[:300]}...")
            parts.append(f"Extrait: {str(ex.get('text', ''))[:400]}...")
            parts.append(f"→ Réponse: {label}\n")

        parts.append("═" * 79 + "\n")

    parts.append("                         CAS À ÉVALUER\n")
    parts.append("═" * 79 + "\n")

    pred_art = row.get("pred_art", "N/A")
    article_text = str(row.get("article_text", "")).strip()
    chunk_text = str(row.get("text", "")).strip()

    parts.append(f"Article {pred_art}: {article_text}\n")
    parts.append(f"Extrait: {chunk_text}\n")

    parts.append("═" * 79)
    parts.append("\nRéponse:")

    return "\n".join(parts)


# ============================================================
# RESPONSE PARSING
# ============================================================

def parse_response(response):
    """Return 1 (oui), 0 (non), -1 (invalid)."""
    if response is None:
        return -1

    s = str(response).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s.startswith("oui"):
        return 1
    if s.startswith("non"):
        return 0

    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None

    if has_oui and not has_non:
        return 1
    if has_non and not has_oui:
        return 0

    return -1


# ============================================================
# METRICS
# ============================================================

def evaluate_predictions(y_true, y_pred):
    valid_mask = np.array(y_pred) >= 0
    y_true_valid = np.array(y_true)[valid_mask]
    y_pred_valid = np.array(y_pred)[valid_mask]

    results = {
        "n_total": int(len(y_true)),
        "n_valid": int(valid_mask.sum()),
        "n_invalid": int((~valid_mask).sum()),
        "pct_valid": float(100 * valid_mask.sum() / max(1, len(y_true))),
    }

    if len(y_true_valid) < 10:
        return results

    results["acc"] = float(accuracy_score(y_true_valid, y_pred_valid))
    results["bacc"] = float(balanced_accuracy_score(y_true_valid, y_pred_valid))
    results["f1_macro"] = float(f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0))
    results["f1_oui"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=1, average="binary", zero_division=0))
    results["f1_non"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=0, average="binary", zero_division=0))
    results["mcc"] = float(matthews_corrcoef(y_true_valid, y_pred_valid))

    cm = confusion_matrix(y_true_valid, y_pred_valid)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        results.update({
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "precision_oui": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
            "recall_oui": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
            "precision_non": float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0,
            "recall_non": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        })

    return results


# ============================================================
# GENERATION (batched, chat_template)
# ============================================================

def generate_responses(prompts, model, tokenizer, batch_size=GEN_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    """
    Generate one response per prompt (batch).
    """
    all_responses = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="    Generation"):
        batch_prompts = prompts[i:i+batch_size]

        formatted_prompts = []
        for prompt in batch_prompts:
            messages = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            formatted_prompts.append(formatted)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        for j in range(outputs.shape[0]):
            input_len = inputs["input_ids"][j].shape[0]
            gen_tokens = outputs[j][input_len:]
            generated = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            all_responses.append(generated.strip())

        del inputs, outputs
        _clear_cuda()

    return all_responses


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 80)
    print("5-FOLD OOF FEW-SHOT (NN) — LLaMA-3.1-8B")
    print(f"Folds: {N_FOLDS} | k={K_VALUES}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1) LOAD DATA
    # --------------------------------------------------------
    print("\n[1] Loading data...")
    df = pd.read_csv(EXCEL_PATH)

    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    # --------------------------------------------------------
    # 2) LABELS (gold: A1/A2 agreement else A3)
    # --------------------------------------------------------
    print("\n[2] Extracting labels...")
    df["a"] = df["eval_A1"].apply(extract_oui_non)
    df["t"] = df["eval_A2"].apply(extract_oui_non)
    df["s"] = df["eval_A3"].apply(extract_oui_non)

    def resolve(r):
        a, t, s = r["a"], r["t"], r["s"]
        if pd.notna(a) and pd.notna(t) and a == t:
            return a
        if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
            return s
        return np.nan

    df["label_str"] = df.apply(resolve, axis=1)
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} ({100*(df['label']==1).mean():.1f}%) | Non: {(df['label']==0).sum()}")

    # index embeddings global
    df = df.reset_index(drop=True)
    df["emb_idx"] = np.arange(len(df))

    # --------------------------------------------------------
    # 3) LOAD EMBEDDING MODEL
    # --------------------------------------------------------
    print("\n[3] Chargement LLaMA base (embeddings)...")
    from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

    tokenizer_base = AutoTokenizer.from_pretrained(LLAMA_MODEL_BASE, token=HF_TOKEN, use_fast=True)
    if tokenizer_base.pad_token is None:
        tokenizer_base.pad_token = tokenizer_base.eos_token
    tokenizer_base.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and USE_BF16) else torch.float32

    model_base = AutoModel.from_pretrained(
        LLAMA_MODEL_BASE,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_base.eval()
    print(f"    ✓ Base model loaded: {LLAMA_MODEL_BASE}")

    # --------------------------------------------------------
    # 4) COMPUTE / LOAD EMBEDDINGS (GLOBAL)
    # --------------------------------------------------------
    print("\n[4] Embeddings globaux (1 seule fois)...")
    texts_all = make_text_for_embedding(df)
    embeddings_all = load_or_compute_embeddings(texts_all, model_base, tokenizer_base, "all")
    print(f"    embeddings_all: {embeddings_all.shape}")

    del model_base, tokenizer_base
    _clear_cuda()

    # --------------------------------------------------------
    # 5) LOAD GENERATION MODEL
    # --------------------------------------------------------
    print("\n[5] Loading LLaMA Instruct (generation)...")

    tokenizer_instruct = AutoTokenizer.from_pretrained(LLAMA_MODEL_INSTRUCT, token=HF_TOKEN)
    if tokenizer_instruct.pad_token is None:
        tokenizer_instruct.pad_token = tokenizer_instruct.eos_token
    tokenizer_instruct.padding_side = "left"

    model_instruct = AutoModelForCausalLM.from_pretrained(
        LLAMA_MODEL_INSTRUCT,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_instruct.eval()
    print(f"    ✓ Instruct model loaded: {LLAMA_MODEL_INSTRUCT}")

    # --------------------------------------------------------
    # 6) PREPARE FOLDS (by decision_id)
    # --------------------------------------------------------
    print("\n[6] Preparing folds (by decision_id)...")

    decision_ids = df["decision_id"].dropna().unique()
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(kf.split(decision_ids)):
        train_decisions = set(decision_ids[train_idx])
        test_decisions = set(decision_ids[test_idx])
        folds.append((fold_id, train_decisions, test_decisions))

    print(f"    ✓ {len(folds)} folds ready.")

    # --------------------------------------------------------
    # 7) OOF LOOP
    # --------------------------------------------------------
    print("\n[7] Boucle OOF (folds × k)...")

    all_oof_rows = []
    all_fold_metrics = []

    for fold_id, train_decisions, test_decisions in folds:
        print("\n" + "=" * 80)
        print(f"FOLD {fold_id}/{N_FOLDS-1}")
        print("=" * 80)

        df_pool = df[df["decision_id"].isin(train_decisions)].reset_index(drop=True)
        df_test = df[df["decision_id"].isin(test_decisions)].reset_index(drop=True)

        emb_pool = embeddings_all[df_pool["emb_idx"].values]
        emb_test = embeddings_all[df_test["emb_idx"].values]

        print(f"    Pool: {len(df_pool)} (Oui={int((df_pool['label']==1).sum())} | Non={int((df_pool['label']==0).sum())})")
        print(f"    Test: {len(df_test)} (Oui={int((df_test['label']==1).sum())} | Non={int((df_test['label']==0).sum())})")

        for k in K_VALUES:
            print("\n" + "-" * 70)
            print(f"Fold {fold_id} | k={k}")
            print("-" * 70)

            # 1) build prompts (and store retrieval ids for debug)
            prompts = []
            meta = []  # store (decision_id, chunk_id, label, pred_art)

            if k == 0:
                for i in tqdm(range(len(df_test)), desc="    Prompts (0-shot)"):
                    row = df_test.iloc[i]
                    prompts.append(build_prompt(row, examples_df=None))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))
            else:
                for i in tqdm(range(len(df_test)), desc=f"    Prompts (kNN k={k})"):
                    row = df_test.iloc[i]
                    query_emb = emb_test[i]

                    examples_df = select_nearest_neighbors(
                        df_pool, emb_pool, query_emb, k=k,
                        seed=SEED + 1000 * fold_id + i
                    )

                    prompts.append(build_prompt(row, examples_df=examples_df))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))

            # 2) generate in batches
            print("    Generating responses...")
            responses = generate_responses(
                prompts,
                model=model_instruct,
                tokenizer=tokenizer_instruct,
                batch_size=GEN_BATCH_SIZE,
                max_new_tokens=MAX_NEW_TOKENS
            )

            # 3) parse
            preds = [parse_response(r) for r in responses]

            # 4) evaluation (fold-level)
            y_true = [m[2] for m in meta]
            fold_res = evaluate_predictions(y_true, preds)
            fold_res.update({"fold": fold_id, "k": k})
            all_fold_metrics.append(fold_res)

            print(f"    📊 Foldւ Fold {fold_id} | k={k}")
            print(f"       MCC={fold_res.get('mcc', np.nan):.4f} | BAcc={fold_res.get('bacc', np.nan):.3f} | Acc={fold_res.get('acc', np.nan):.3f}")
            print(f"       F1_macro={fold_res.get('f1_macro', np.nan):.3f} | F1_oui={fold_res.get('f1_oui', np.nan):.3f} | F1_non={fold_res.get('f1_non', np.nan):.3f}")
            print(f"       Valides={fold_res['n_valid']}/{fold_res['n_total']} ({fold_res['pct_valid']:.1f}%)")

            if "tp" in fold_res:
                print(f"       Confusion: TP={fold_res['tp']} FP={fold_res['fp']} TN={fold_res['tn']} FN={fold_res['fn']}")

            # 5) store OOF rows
            for (dec_id, chunk_id, gold, pred_art), resp, pred in zip(meta, responses, preds):
                all_oof_rows.append({
                    "decision_id": dec_id,
                    "chunk_id": chunk_id,
                    "pred_art": pred_art,
                    "fold": fold_id,
                    "k": k,
                    "label": int(gold),
                    "pred": int(pred),
                    "response_raw": resp
                })

    # --------------------------------------------------------
    # 8) GLOBAL OOF METRICS
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("OOF GLOBAL METRICS (par k)")
    print("=" * 80)

    oof_df = pd.DataFrame(all_oof_rows)
    metrics_rows = []
    for k in K_VALUES:
        sub = oof_df[oof_df["k"] == k].copy()
        res = evaluate_predictions(sub["label"].values, sub["pred"].values)
        res["k"] = k
        res["fold"] = "ALL"
        metrics_rows.append(res)

        print(f"\n    k={k}")
        print(f"      MCC={res.get('mcc', np.nan):.4f} | BAcc={res.get('bacc', np.nan):.3f} | Acc={res.get('acc', np.nan):.3f}")
        print(f"      F1_macro={res.get('f1_macro', np.nan):.3f} | F1_oui={res.get('f1_oui', np.nan):.3f} | F1_non={res.get('f1_non', np.nan):.3f}")
        print(f"      Valides={res['n_valid']}/{res['n_total']} ({res['pct_valid']:.1f}%)")

    fold_metrics_df = pd.DataFrame(all_fold_metrics)
    global_metrics_df = pd.DataFrame(metrics_rows)

    # --------------------------------------------------------
    # 9) EXPORTS
    # --------------------------------------------------------
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_oof = os.path.join(OUTPUT_PATH, f"oof_predictions_{ts}.xlsx")
    out_metrics = os.path.join(OUTPUT_PATH, f"metrics_{ts}.xlsx")

    with pd.ExcelWriter(out_metrics) as writer:
        fold_metrics_df.to_excel(writer, sheet_name="fold_metrics", index=False)
        global_metrics_df.to_excel(writer, sheet_name="global_metrics", index=False)

    oof_df.to_excel(out_oof, index=False)

    print("\n" + "=" * 80)
    print("EXPORTS")
    print("=" * 80)
    print(f"    ✓ OOF:     {out_oof}")
    print(f"    ✓ Metrics: {out_metrics}")

    # --------------------------------------------------------
    # 10) PLOTS
    # --------------------------------------------------------
    if DO_PLOTS:
        try:
            plot_df = global_metrics_df.copy().sort_values("k")
            fig = plt.figure(figsize=(11, 5))

            # MCC
            plt.plot(plot_df["k"], plot_df["mcc"], marker="o", linewidth=2, label="MCC")
            # BAcc
            if "bacc" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["bacc"], marker="s", linewidth=2, label="Balanced Acc")
            # F1 macro
            if "f1_macro" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["f1_macro"], marker="^", linewidth=2, label="F1 macro")

            plt.xlabel("k (nb d'exemples few-shot)")
            plt.ylabel("Score")
            plt.title("OOF — Performances en fonction de k (retrieval NN dans 4 folds)")
            plt.xticks(K_VALUES)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()

            fig_path = os.path.join(OUTPUT_PATH, f"metrics_vs_k_{ts}.png")
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            plt.show()

            print(f"    ✓ Figure: {fig_path}")
        except Exception as e:
            print(f"[WARN] plot failed: {e}")

    # Cleanup
    del model_instruct, tokenizer_instruct
    _clear_cuda()

    print("\n" + "=" * 80)
    print("FIN")
    print("=" * 80)

    return oof_df, fold_metrics_df, global_metrics_df


if __name__ == "__main__":
    oof_df, fold_metrics_df, global_metrics_df = main()


# LLaMa few close - A1

In [ ]:
# ============================================================
# SCRIPT — 5-FOLD OOF: 0-SHOT vs FEW-SHOT (Nearest Neighbors)
# Implicit application of the Civil Code (oui/non)
#
# - 5-fold CV split by decision_id (no intra-decision leakage)
# - Example retrieval (k-NN) ONLY within the 4 train folds
# - k ∈ {0, 2, 4, 8} (modifiable)
#
# Embeddings: LLaMA-3.1-8B (mean pooling, last layer)
# Generation: LLaMA-3.1-8B-Instruct
#
# Exports:
# - OOF predictions (par fold, par k)
# - Global metrics table (by k + by fold)
# - Figure MCC/BAcc/F1 vs k
# ============================================================

import os, re, gc, hashlib, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from datetime import datetime

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_fewshot_llama_cv_nn")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# CV
N_FOLDS = 5

# Few-shot
K_VALUES = [0, 2, 4, 8]  # tu peux mettre [0,2,4,8] comme tu dis
MAX_NEW_TOKENS = 20
GEN_BATCH_SIZE = 4

# Embeddings
EMB_MAX_LEN = 512
EMB_BATCH_SIZE = 16
USE_BF16 = True

# Models
LLAMA_MODEL_BASE = "meta-llama/Llama-3.1-8B"             # embeddings
LLAMA_MODEL_INSTRUCT = "meta-llama/Llama-3.1-8B-Instruct" # generation
HF_TOKEN = os.environ["HF_TOKEN"]          # <-- mets ton token

# Plot
DO_PLOTS = True


# ============================================================
# GRILLE (prompt court neutre)
# ============================================================

GRILLE_BINAIRE_NEUTRE = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()


# ============================================================
# HELPERS (cuda / parsing / texte embedding)
# ============================================================

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def extract_oui_non(x):
    """Extract oui/non from the human annotations (robust, conservative)."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


def make_text_for_embedding(df):
    """Embedding text = (article + chunk) pair with stable tags."""
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()
    return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()


def _hash_texts(texts, n=200):
    """Stable hash over a sample + total N (for cache)."""
    if len(texts) == 0:
        return "empty"
    sample = texts[:n] + texts[-n:] if len(texts) > 2 * n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


# ============================================================
# EMBEDDINGS LLaMA (mean pooling last layer) + cache
# ============================================================

def encode_with_llama(texts, model, tokenizer, batch_size=EMB_BATCH_SIZE, max_len=EMB_MAX_LEN, desc="Encoding"):
    device = next(model.parameters()).device
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]

            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16 and torch.cuda.is_available()):
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hidden = out.hidden_states[-1]  # (B, T, D)
            attn_mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)

            emb = (hidden * attn_mask).sum(dim=1) / attn_mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hidden, attn_mask, emb
            _clear_cuda()

    return np.vstack(all_embs)


def load_or_compute_embeddings(texts, model, tokenizer, cache_name):
    texts_hash = _hash_texts(texts)
    cache_path = os.path.join(CACHE_DIR, f"emb_{cache_name}_{texts_hash[:12]}.npy")

    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing embeddings ({cache_name})...")
    embeddings = encode_with_llama(texts, model, tokenizer, desc=f"    {cache_name}")
    np.save(cache_path, embeddings)
    print(f"    ✓ Saved: {cache_path}")
    return embeddings


# ============================================================
# NEAREST NEIGHBORS (balanced 50/50 si possible)
# ============================================================

def select_nearest_neighbors(df_pool, embeddings_pool, query_embedding, k, seed=SEED):
    """
    k-NN selection within df_pool (train folds), with class balancing.
    df_pool doit contenir 'label' (0/1).
    """
    if k == 0:
        return None

    rng = np.random.default_rng(seed)

    query_emb = query_embedding.reshape(1, -1)
    similarities = cosine_similarity(embeddings_pool, query_emb).flatten()

    k_per_class = k // 2
    selected_indices = []

    for label in [0, 1]:
        mask = (df_pool["label"].values == label)
        idxs = np.where(mask)[0]
        if len(idxs) == 0:
            continue

        sims_class = similarities[mask]
        top_k = min(k_per_class, len(sims_class))

        # argsort ascending -> take last top_k
        top_local = np.argsort(sims_class)[-top_k:]
        top_global = idxs[top_local]
        selected_indices.extend(top_global.tolist())

    # top up if not enough
    if len(selected_indices) < k:
        remaining = k - len(selected_indices)
        all_indices = np.arange(len(df_pool))
        available = np.setdiff1d(all_indices, np.array(selected_indices, dtype=int), assume_unique=False)

        if len(available) > 0:
            sims_available = similarities[available]
            take = min(remaining, len(available))
            top = np.argsort(sims_available)[-take:]
            selected_indices.extend(available[top].tolist())

    examples = df_pool.iloc[selected_indices].copy()
    # shuffle stable
    examples = examples.sample(frac=1, random_state=seed).reset_index(drop=True)
    return examples


# ============================================================
# PROMPTS
# ============================================================

def build_prompt(row, examples_df=None):
    parts = []
    parts.append(GRILLE_BINAIRE_NEUTRE)
    parts.append("\n" + "═" * 79 + "\n")

    if examples_df is not None and len(examples_df) > 0:
        parts.append("                            EXEMPLES\n")
        parts.append("═" * 79 + "\n")

        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            parts.append(f"--- Exemple {i} ---")
            parts.append(f"Article {ex.get('pred_art', 'N/A')}: {str(ex.get('article_text', ''))[:300]}...")
            parts.append(f"Extrait: {str(ex.get('text', ''))[:400]}...")
            parts.append(f"→ Réponse: {label}\n")

        parts.append("═" * 79 + "\n")

    parts.append("                         CAS À ÉVALUER\n")
    parts.append("═" * 79 + "\n")

    pred_art = row.get("pred_art", "N/A")
    article_text = str(row.get("article_text", "")).strip()
    chunk_text = str(row.get("text", "")).strip()

    parts.append(f"Article {pred_art}: {article_text}\n")
    parts.append(f"Extrait: {chunk_text}\n")

    parts.append("═" * 79)
    parts.append("\nRéponse:")

    return "\n".join(parts)


# ============================================================
# RESPONSE PARSING
# ============================================================

def parse_response(response):
    """Return 1 (oui), 0 (non), -1 (invalid)."""
    if response is None:
        return -1

    s = str(response).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s.startswith("oui"):
        return 1
    if s.startswith("non"):
        return 0

    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None

    if has_oui and not has_non:
        return 1
    if has_non and not has_oui:
        return 0

    return -1


# ============================================================
# METRICS
# ============================================================

def evaluate_predictions(y_true, y_pred):
    valid_mask = np.array(y_pred) >= 0
    y_true_valid = np.array(y_true)[valid_mask]
    y_pred_valid = np.array(y_pred)[valid_mask]

    results = {
        "n_total": int(len(y_true)),
        "n_valid": int(valid_mask.sum()),
        "n_invalid": int((~valid_mask).sum()),
        "pct_valid": float(100 * valid_mask.sum() / max(1, len(y_true))),
    }

    if len(y_true_valid) < 10:
        return results

    results["acc"] = float(accuracy_score(y_true_valid, y_pred_valid))
    results["bacc"] = float(balanced_accuracy_score(y_true_valid, y_pred_valid))
    results["f1_macro"] = float(f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0))
    results["f1_oui"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=1, average="binary", zero_division=0))
    results["f1_non"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=0, average="binary", zero_division=0))
    results["mcc"] = float(matthews_corrcoef(y_true_valid, y_pred_valid))

    cm = confusion_matrix(y_true_valid, y_pred_valid)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        results.update({
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "precision_oui": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
            "recall_oui": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
            "precision_non": float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0,
            "recall_non": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        })

    return results


# ============================================================
# GENERATION (batched, chat_template)
# ============================================================

def generate_responses(prompts, model, tokenizer, batch_size=GEN_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    """
    Generate one response per prompt (batch).
    """
    all_responses = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="    Generation"):
        batch_prompts = prompts[i:i+batch_size]

        formatted_prompts = []
        for prompt in batch_prompts:
            messages = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            formatted_prompts.append(formatted)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        for j in range(outputs.shape[0]):
            input_len = inputs["input_ids"][j].shape[0]
            gen_tokens = outputs[j][input_len:]
            generated = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            all_responses.append(generated.strip())

        del inputs, outputs
        _clear_cuda()

    return all_responses


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 80)
    print("5-FOLD OOF FEW-SHOT (NN) — LLaMA-3.1-8B")
    print(f"Folds: {N_FOLDS} | k={K_VALUES}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1) LOAD DATA
    # --------------------------------------------------------
    print("\n[1] Loading data...")
    df = pd.read_csv(EXCEL_PATH)

    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    # --------------------------------------------------------
    # 2) LABELS (gold: Arome)
    # --------------------------------------------------------
    print("\n[2] Extracting labels...")
    df["label_str"] = df["eval_A1"].apply(extract_oui_non)

    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} ({100*(df['label']==1).mean():.1f}%) | Non: {(df['label']==0).sum()}")

    # index embeddings global
    df = df.reset_index(drop=True)
    df["emb_idx"] = np.arange(len(df))

    # --------------------------------------------------------
    # 3) LOAD EMBEDDING MODEL
    # --------------------------------------------------------
    print("\n[3] Chargement LLaMA base (embeddings)...")
    from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

    tokenizer_base = AutoTokenizer.from_pretrained(LLAMA_MODEL_BASE, token=HF_TOKEN, use_fast=True)
    if tokenizer_base.pad_token is None:
        tokenizer_base.pad_token = tokenizer_base.eos_token
    tokenizer_base.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and USE_BF16) else torch.float32

    model_base = AutoModel.from_pretrained(
        LLAMA_MODEL_BASE,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_base.eval()
    print(f"    ✓ Base model loaded: {LLAMA_MODEL_BASE}")

    # --------------------------------------------------------
    # 4) COMPUTE / LOAD EMBEDDINGS (GLOBAL)
    # --------------------------------------------------------
    print("\n[4] Embeddings globaux (1 seule fois)...")
    texts_all = make_text_for_embedding(df)
    embeddings_all = load_or_compute_embeddings(texts_all, model_base, tokenizer_base, "all")
    print(f"    embeddings_all: {embeddings_all.shape}")

    del model_base, tokenizer_base
    _clear_cuda()

    # --------------------------------------------------------
    # 5) LOAD GENERATION MODEL
    # --------------------------------------------------------
    print("\n[5] Loading LLaMA Instruct (generation)...")

    tokenizer_instruct = AutoTokenizer.from_pretrained(LLAMA_MODEL_INSTRUCT, token=HF_TOKEN)
    if tokenizer_instruct.pad_token is None:
        tokenizer_instruct.pad_token = tokenizer_instruct.eos_token
    tokenizer_instruct.padding_side = "left"

    model_instruct = AutoModelForCausalLM.from_pretrained(
        LLAMA_MODEL_INSTRUCT,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_instruct.eval()
    print(f"    ✓ Instruct model loaded: {LLAMA_MODEL_INSTRUCT}")

    # --------------------------------------------------------
    # 6) PREPARE FOLDS (by decision_id)
    # --------------------------------------------------------
    print("\n[6] Preparing folds (by decision_id)...")

    decision_ids = df["decision_id"].dropna().unique()
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(kf.split(decision_ids)):
        train_decisions = set(decision_ids[train_idx])
        test_decisions = set(decision_ids[test_idx])
        folds.append((fold_id, train_decisions, test_decisions))

    print(f"    ✓ {len(folds)} folds ready.")

    # --------------------------------------------------------
    # 7) OOF LOOP
    # --------------------------------------------------------
    print("\n[7] Boucle OOF (folds × k)...")

    all_oof_rows = []
    all_fold_metrics = []

    for fold_id, train_decisions, test_decisions in folds:
        print("\n" + "=" * 80)
        print(f"FOLD {fold_id}/{N_FOLDS-1}")
        print("=" * 80)

        df_pool = df[df["decision_id"].isin(train_decisions)].reset_index(drop=True)
        df_test = df[df["decision_id"].isin(test_decisions)].reset_index(drop=True)

        emb_pool = embeddings_all[df_pool["emb_idx"].values]
        emb_test = embeddings_all[df_test["emb_idx"].values]

        print(f"    Pool: {len(df_pool)} (Oui={int((df_pool['label']==1).sum())} | Non={int((df_pool['label']==0).sum())})")
        print(f"    Test: {len(df_test)} (Oui={int((df_test['label']==1).sum())} | Non={int((df_test['label']==0).sum())})")

        for k in K_VALUES:
            print("\n" + "-" * 70)
            print(f"Fold {fold_id} | k={k}")
            print("-" * 70)

            # 1) build prompts (and store retrieval ids for debug)
            prompts = []
            meta = []  # store (decision_id, chunk_id, label, pred_art)

            if k == 0:
                for i in tqdm(range(len(df_test)), desc="    Prompts (0-shot)"):
                    row = df_test.iloc[i]
                    prompts.append(build_prompt(row, examples_df=None))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))
            else:
                for i in tqdm(range(len(df_test)), desc=f"    Prompts (kNN k={k})"):
                    row = df_test.iloc[i]
                    query_emb = emb_test[i]

                    examples_df = select_nearest_neighbors(
                        df_pool, emb_pool, query_emb, k=k,
                        seed=SEED + 1000 * fold_id + i
                    )

                    prompts.append(build_prompt(row, examples_df=examples_df))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))

            # 2) generate in batches
            print("    Generating responses...")
            responses = generate_responses(
                prompts,
                model=model_instruct,
                tokenizer=tokenizer_instruct,
                batch_size=GEN_BATCH_SIZE,
                max_new_tokens=MAX_NEW_TOKENS
            )

            # 3) parse
            preds = [parse_response(r) for r in responses]

            # 4) evaluation (fold-level)
            y_true = [m[2] for m in meta]
            fold_res = evaluate_predictions(y_true, preds)
            fold_res.update({"fold": fold_id, "k": k})
            all_fold_metrics.append(fold_res)

            print(f"    📊 Foldւ Fold {fold_id} | k={k}")
            print(f"       MCC={fold_res.get('mcc', np.nan):.4f} | BAcc={fold_res.get('bacc', np.nan):.3f} | Acc={fold_res.get('acc', np.nan):.3f}")
            print(f"       F1_macro={fold_res.get('f1_macro', np.nan):.3f} | F1_oui={fold_res.get('f1_oui', np.nan):.3f} | F1_non={fold_res.get('f1_non', np.nan):.3f}")
            print(f"       Valides={fold_res['n_valid']}/{fold_res['n_total']} ({fold_res['pct_valid']:.1f}%)")

            if "tp" in fold_res:
                print(f"       Confusion: TP={fold_res['tp']} FP={fold_res['fp']} TN={fold_res['tn']} FN={fold_res['fn']}")

            # 5) store OOF rows
            for (dec_id, chunk_id, gold, pred_art), resp, pred in zip(meta, responses, preds):
                all_oof_rows.append({
                    "decision_id": dec_id,
                    "chunk_id": chunk_id,
                    "pred_art": pred_art,
                    "fold": fold_id,
                    "k": k,
                    "label": int(gold),
                    "pred": int(pred),
                    "response_raw": resp
                })

    # --------------------------------------------------------
    # 8) GLOBAL OOF METRICS
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("OOF GLOBAL METRICS (par k)")
    print("=" * 80)

    oof_df = pd.DataFrame(all_oof_rows)
    metrics_rows = []
    for k in K_VALUES:
        sub = oof_df[oof_df["k"] == k].copy()
        res = evaluate_predictions(sub["label"].values, sub["pred"].values)
        res["k"] = k
        res["fold"] = "ALL"
        metrics_rows.append(res)

        print(f"\n    k={k}")
        print(f"      MCC={res.get('mcc', np.nan):.4f} | BAcc={res.get('bacc', np.nan):.3f} | Acc={res.get('acc', np.nan):.3f}")
        print(f"      F1_macro={res.get('f1_macro', np.nan):.3f} | F1_oui={res.get('f1_oui', np.nan):.3f} | F1_non={res.get('f1_non', np.nan):.3f}")
        print(f"      Valides={res['n_valid']}/{res['n_total']} ({res['pct_valid']:.1f}%)")

    fold_metrics_df = pd.DataFrame(all_fold_metrics)
    global_metrics_df = pd.DataFrame(metrics_rows)

    # --------------------------------------------------------
    # 9) EXPORTS
    # --------------------------------------------------------
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_oof = os.path.join(OUTPUT_PATH, f"oof_predictions_{ts}.xlsx")
    out_metrics = os.path.join(OUTPUT_PATH, f"metrics_{ts}.xlsx")

    with pd.ExcelWriter(out_metrics) as writer:
        fold_metrics_df.to_excel(writer, sheet_name="fold_metrics", index=False)
        global_metrics_df.to_excel(writer, sheet_name="global_metrics", index=False)

    oof_df.to_excel(out_oof, index=False)

    print("\n" + "=" * 80)
    print("EXPORTS")
    print("=" * 80)
    print(f"    ✓ OOF:     {out_oof}")
    print(f"    ✓ Metrics: {out_metrics}")

    # --------------------------------------------------------
    # 10) PLOTS
    # --------------------------------------------------------
    if DO_PLOTS:
        try:
            plot_df = global_metrics_df.copy().sort_values("k")
            fig = plt.figure(figsize=(11, 5))

            # MCC
            plt.plot(plot_df["k"], plot_df["mcc"], marker="o", linewidth=2, label="MCC")
            # BAcc
            if "bacc" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["bacc"], marker="s", linewidth=2, label="Balanced Acc")
            # F1 macro
            if "f1_macro" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["f1_macro"], marker="^", linewidth=2, label="F1 macro")

            plt.xlabel("k (nb d'exemples few-shot)")
            plt.ylabel("Score")
            plt.title("OOF — Performances en fonction de k (retrieval NN dans 4 folds)")
            plt.xticks(K_VALUES)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()

            fig_path = os.path.join(OUTPUT_PATH, f"metrics_vs_k_{ts}.png")
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            plt.show()

            print(f"    ✓ Figure: {fig_path}")
        except Exception as e:
            print(f"[WARN] plot failed: {e}")

    # Cleanup
    del model_instruct, tokenizer_instruct
    _clear_cuda()

    print("\n" + "=" * 80)
    print("FIN")
    print("=" * 80)

    return oof_df, fold_metrics_df, global_metrics_df


if __name__ == "__main__":
    oof_df, fold_metrics_df, global_metrics_df = main()


# LLaMa few close - A2

In [ ]:
# ============================================================
# SCRIPT — 5-FOLD OOF: 0-SHOT vs FEW-SHOT (Nearest Neighbors)
# Implicit application of the Civil Code (oui/non)
#
# - 5-fold CV split by decision_id (no intra-decision leakage)
# - Example retrieval (k-NN) ONLY within the 4 train folds
# - k ∈ {0, 2, 4, 8} (modifiable)
#
# Embeddings: LLaMA-3.1-8B (mean pooling, last layer)
# Generation: LLaMA-3.1-8B-Instruct
#
# Exports:
# - OOF predictions (par fold, par k)
# - Global metrics table (by k + by fold)
# - Figure MCC/BAcc/F1 vs k
# ============================================================

import os, re, gc, hashlib, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from datetime import datetime

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_fewshot_llama_cv_nn")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# CV
N_FOLDS = 5

# Few-shot
K_VALUES = [0, 2, 4, 8]  # tu peux mettre [0,2,4,8] comme tu dis
MAX_NEW_TOKENS = 20
GEN_BATCH_SIZE = 4

# Embeddings
EMB_MAX_LEN = 512
EMB_BATCH_SIZE = 16
USE_BF16 = True

# Models
LLAMA_MODEL_BASE = "meta-llama/Llama-3.1-8B"             # embeddings
LLAMA_MODEL_INSTRUCT = "meta-llama/Llama-3.1-8B-Instruct" # generation
HF_TOKEN = os.environ["HF_TOKEN"]          # <-- mets ton token

# Plot
DO_PLOTS = True


# ============================================================
# GRILLE (prompt court neutre)
# ============================================================

GRILLE_BINAIRE_NEUTRE = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()


# ============================================================
# HELPERS (cuda / parsing / texte embedding)
# ============================================================

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def extract_oui_non(x):
    """Extract oui/non from the human annotations (robust, conservative)."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


def make_text_for_embedding(df):
    """Embedding text = (article + chunk) pair with stable tags."""
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()
    return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()


def _hash_texts(texts, n=200):
    """Stable hash over a sample + total N (for cache)."""
    if len(texts) == 0:
        return "empty"
    sample = texts[:n] + texts[-n:] if len(texts) > 2 * n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


# ============================================================
# EMBEDDINGS LLaMA (mean pooling last layer) + cache
# ============================================================

def encode_with_llama(texts, model, tokenizer, batch_size=EMB_BATCH_SIZE, max_len=EMB_MAX_LEN, desc="Encoding"):
    device = next(model.parameters()).device
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]

            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16 and torch.cuda.is_available()):
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hidden = out.hidden_states[-1]  # (B, T, D)
            attn_mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)

            emb = (hidden * attn_mask).sum(dim=1) / attn_mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hidden, attn_mask, emb
            _clear_cuda()

    return np.vstack(all_embs)


def load_or_compute_embeddings(texts, model, tokenizer, cache_name):
    texts_hash = _hash_texts(texts)
    cache_path = os.path.join(CACHE_DIR, f"emb_{cache_name}_{texts_hash[:12]}.npy")

    if os.path.exists(cache_path):
        print(f"    ✓ Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"    Computing embeddings ({cache_name})...")
    embeddings = encode_with_llama(texts, model, tokenizer, desc=f"    {cache_name}")
    np.save(cache_path, embeddings)
    print(f"    ✓ Saved: {cache_path}")
    return embeddings


# ============================================================
# NEAREST NEIGHBORS (balanced 50/50 si possible)
# ============================================================

def select_nearest_neighbors(df_pool, embeddings_pool, query_embedding, k, seed=SEED):
    """
    k-NN selection within df_pool (train folds), with class balancing.
    df_pool doit contenir 'label' (0/1).
    """
    if k == 0:
        return None

    rng = np.random.default_rng(seed)

    query_emb = query_embedding.reshape(1, -1)
    similarities = cosine_similarity(embeddings_pool, query_emb).flatten()

    k_per_class = k // 2
    selected_indices = []

    for label in [0, 1]:
        mask = (df_pool["label"].values == label)
        idxs = np.where(mask)[0]
        if len(idxs) == 0:
            continue

        sims_class = similarities[mask]
        top_k = min(k_per_class, len(sims_class))

        # argsort ascending -> take last top_k
        top_local = np.argsort(sims_class)[-top_k:]
        top_global = idxs[top_local]
        selected_indices.extend(top_global.tolist())

    # top up if not enough
    if len(selected_indices) < k:
        remaining = k - len(selected_indices)
        all_indices = np.arange(len(df_pool))
        available = np.setdiff1d(all_indices, np.array(selected_indices, dtype=int), assume_unique=False)

        if len(available) > 0:
            sims_available = similarities[available]
            take = min(remaining, len(available))
            top = np.argsort(sims_available)[-take:]
            selected_indices.extend(available[top].tolist())

    examples = df_pool.iloc[selected_indices].copy()
    # shuffle stable
    examples = examples.sample(frac=1, random_state=seed).reset_index(drop=True)
    return examples


# ============================================================
# PROMPTS
# ============================================================

def build_prompt(row, examples_df=None):
    parts = []
    parts.append(GRILLE_BINAIRE_NEUTRE)
    parts.append("\n" + "═" * 79 + "\n")

    if examples_df is not None and len(examples_df) > 0:
        parts.append("                            EXEMPLES\n")
        parts.append("═" * 79 + "\n")

        for i, (_, ex) in enumerate(examples_df.iterrows(), 1):
            label = "oui" if ex["label"] == 1 else "non"
            parts.append(f"--- Exemple {i} ---")
            parts.append(f"Article {ex.get('pred_art', 'N/A')}: {str(ex.get('article_text', ''))[:300]}...")
            parts.append(f"Extrait: {str(ex.get('text', ''))[:400]}...")
            parts.append(f"→ Réponse: {label}\n")

        parts.append("═" * 79 + "\n")

    parts.append("                         CAS À ÉVALUER\n")
    parts.append("═" * 79 + "\n")

    pred_art = row.get("pred_art", "N/A")
    article_text = str(row.get("article_text", "")).strip()
    chunk_text = str(row.get("text", "")).strip()

    parts.append(f"Article {pred_art}: {article_text}\n")
    parts.append(f"Extrait: {chunk_text}\n")

    parts.append("═" * 79)
    parts.append("\nRéponse:")

    return "\n".join(parts)


# ============================================================
# RESPONSE PARSING
# ============================================================

def parse_response(response):
    """Return 1 (oui), 0 (non), -1 (invalid)."""
    if response is None:
        return -1

    s = str(response).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s.startswith("oui"):
        return 1
    if s.startswith("non"):
        return 0

    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None

    if has_oui and not has_non:
        return 1
    if has_non and not has_oui:
        return 0

    return -1


# ============================================================
# METRICS
# ============================================================

def evaluate_predictions(y_true, y_pred):
    valid_mask = np.array(y_pred) >= 0
    y_true_valid = np.array(y_true)[valid_mask]
    y_pred_valid = np.array(y_pred)[valid_mask]

    results = {
        "n_total": int(len(y_true)),
        "n_valid": int(valid_mask.sum()),
        "n_invalid": int((~valid_mask).sum()),
        "pct_valid": float(100 * valid_mask.sum() / max(1, len(y_true))),
    }

    if len(y_true_valid) < 10:
        return results

    results["acc"] = float(accuracy_score(y_true_valid, y_pred_valid))
    results["bacc"] = float(balanced_accuracy_score(y_true_valid, y_pred_valid))
    results["f1_macro"] = float(f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0))
    results["f1_oui"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=1, average="binary", zero_division=0))
    results["f1_non"] = float(f1_score(y_true_valid, y_pred_valid, pos_label=0, average="binary", zero_division=0))
    results["mcc"] = float(matthews_corrcoef(y_true_valid, y_pred_valid))

    cm = confusion_matrix(y_true_valid, y_pred_valid)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        results.update({
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "precision_oui": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
            "recall_oui": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
            "precision_non": float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0,
            "recall_non": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        })

    return results


# ============================================================
# GENERATION (batched, chat_template)
# ============================================================

def generate_responses(prompts, model, tokenizer, batch_size=GEN_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    """
    Generate one response per prompt (batch).
    """
    all_responses = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="    Generation"):
        batch_prompts = prompts[i:i+batch_size]

        formatted_prompts = []
        for prompt in batch_prompts:
            messages = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            formatted_prompts.append(formatted)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        for j in range(outputs.shape[0]):
            input_len = inputs["input_ids"][j].shape[0]
            gen_tokens = outputs[j][input_len:]
            generated = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            all_responses.append(generated.strip())

        del inputs, outputs
        _clear_cuda()

    return all_responses


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 80)
    print("5-FOLD OOF FEW-SHOT (NN) — LLaMA-3.1-8B")
    print(f"Folds: {N_FOLDS} | k={K_VALUES}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1) LOAD DATA
    # --------------------------------------------------------
    print("\n[1] Loading data...")
    df = pd.read_csv(EXCEL_PATH)

    cols_needed = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A1", "eval_A2", "eval_A3"
    ]
    df = df[[c for c in cols_needed if c in df.columns]].copy()

    # --------------------------------------------------------
    # 2) LABELS (gold: A2)
    # --------------------------------------------------------
    print("\n[2] Extracting labels...")
    df["label_str"] = df["eval_A2"].apply(extract_oui_non)

    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    print(f"    Total: {len(df)} | Oui: {(df['label']==1).sum()} ({100*(df['label']==1).mean():.1f}%) | Non: {(df['label']==0).sum()}")

    # index embeddings global
    df = df.reset_index(drop=True)
    df["emb_idx"] = np.arange(len(df))

    # --------------------------------------------------------
    # 3) LOAD EMBEDDING MODEL
    # --------------------------------------------------------
    print("\n[3] Chargement LLaMA base (embeddings)...")
    from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

    tokenizer_base = AutoTokenizer.from_pretrained(LLAMA_MODEL_BASE, token=HF_TOKEN, use_fast=True)
    if tokenizer_base.pad_token is None:
        tokenizer_base.pad_token = tokenizer_base.eos_token
    tokenizer_base.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and USE_BF16) else torch.float32

    model_base = AutoModel.from_pretrained(
        LLAMA_MODEL_BASE,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_base.eval()
    print(f"    ✓ Base model loaded: {LLAMA_MODEL_BASE}")

    # --------------------------------------------------------
    # 4) COMPUTE / LOAD EMBEDDINGS (GLOBAL)
    # --------------------------------------------------------
    print("\n[4] Embeddings globaux (1 seule fois)...")
    texts_all = make_text_for_embedding(df)
    embeddings_all = load_or_compute_embeddings(texts_all, model_base, tokenizer_base, "all")
    print(f"    embeddings_all: {embeddings_all.shape}")

    del model_base, tokenizer_base
    _clear_cuda()

    # --------------------------------------------------------
    # 5) LOAD GENERATION MODEL
    # --------------------------------------------------------
    print("\n[5] Loading LLaMA Instruct (generation)...")

    tokenizer_instruct = AutoTokenizer.from_pretrained(LLAMA_MODEL_INSTRUCT, token=HF_TOKEN)
    if tokenizer_instruct.pad_token is None:
        tokenizer_instruct.pad_token = tokenizer_instruct.eos_token
    tokenizer_instruct.padding_side = "left"

    model_instruct = AutoModelForCausalLM.from_pretrained(
        LLAMA_MODEL_INSTRUCT,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model_instruct.eval()
    print(f"    ✓ Instruct model loaded: {LLAMA_MODEL_INSTRUCT}")

    # --------------------------------------------------------
    # 6) PREPARE FOLDS (by decision_id)
    # --------------------------------------------------------
    print("\n[6] Preparing folds (by decision_id)...")

    decision_ids = df["decision_id"].dropna().unique()
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(kf.split(decision_ids)):
        train_decisions = set(decision_ids[train_idx])
        test_decisions = set(decision_ids[test_idx])
        folds.append((fold_id, train_decisions, test_decisions))

    print(f"    ✓ {len(folds)} folds ready.")

    # --------------------------------------------------------
    # 7) OOF LOOP
    # --------------------------------------------------------
    print("\n[7] Boucle OOF (folds × k)...")

    all_oof_rows = []
    all_fold_metrics = []

    for fold_id, train_decisions, test_decisions in folds:
        print("\n" + "=" * 80)
        print(f"FOLD {fold_id}/{N_FOLDS-1}")
        print("=" * 80)

        df_pool = df[df["decision_id"].isin(train_decisions)].reset_index(drop=True)
        df_test = df[df["decision_id"].isin(test_decisions)].reset_index(drop=True)

        emb_pool = embeddings_all[df_pool["emb_idx"].values]
        emb_test = embeddings_all[df_test["emb_idx"].values]

        print(f"    Pool: {len(df_pool)} (Oui={int((df_pool['label']==1).sum())} | Non={int((df_pool['label']==0).sum())})")
        print(f"    Test: {len(df_test)} (Oui={int((df_test['label']==1).sum())} | Non={int((df_test['label']==0).sum())})")

        for k in K_VALUES:
            print("\n" + "-" * 70)
            print(f"Fold {fold_id} | k={k}")
            print("-" * 70)

            # 1) build prompts (and store retrieval ids for debug)
            prompts = []
            meta = []  # store (decision_id, chunk_id, label, pred_art)

            if k == 0:
                for i in tqdm(range(len(df_test)), desc="    Prompts (0-shot)"):
                    row = df_test.iloc[i]
                    prompts.append(build_prompt(row, examples_df=None))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))
            else:
                for i in tqdm(range(len(df_test)), desc=f"    Prompts (kNN k={k})"):
                    row = df_test.iloc[i]
                    query_emb = emb_test[i]

                    examples_df = select_nearest_neighbors(
                        df_pool, emb_pool, query_emb, k=k,
                        seed=SEED + 1000 * fold_id + i
                    )

                    prompts.append(build_prompt(row, examples_df=examples_df))
                    meta.append((row["decision_id"], row["chunk_id"], row["label"], row.get("pred_art", None)))

            # 2) generate in batches
            print("    Generating responses...")
            responses = generate_responses(
                prompts,
                model=model_instruct,
                tokenizer=tokenizer_instruct,
                batch_size=GEN_BATCH_SIZE,
                max_new_tokens=MAX_NEW_TOKENS
            )

            # 3) parse
            preds = [parse_response(r) for r in responses]

            # 4) evaluation (fold-level)
            y_true = [m[2] for m in meta]
            fold_res = evaluate_predictions(y_true, preds)
            fold_res.update({"fold": fold_id, "k": k})
            all_fold_metrics.append(fold_res)

            print(f"    📊 Foldւ Fold {fold_id} | k={k}")
            print(f"       MCC={fold_res.get('mcc', np.nan):.4f} | BAcc={fold_res.get('bacc', np.nan):.3f} | Acc={fold_res.get('acc', np.nan):.3f}")
            print(f"       F1_macro={fold_res.get('f1_macro', np.nan):.3f} | F1_oui={fold_res.get('f1_oui', np.nan):.3f} | F1_non={fold_res.get('f1_non', np.nan):.3f}")
            print(f"       Valides={fold_res['n_valid']}/{fold_res['n_total']} ({fold_res['pct_valid']:.1f}%)")

            if "tp" in fold_res:
                print(f"       Confusion: TP={fold_res['tp']} FP={fold_res['fp']} TN={fold_res['tn']} FN={fold_res['fn']}")

            # 5) store OOF rows
            for (dec_id, chunk_id, gold, pred_art), resp, pred in zip(meta, responses, preds):
                all_oof_rows.append({
                    "decision_id": dec_id,
                    "chunk_id": chunk_id,
                    "pred_art": pred_art,
                    "fold": fold_id,
                    "k": k,
                    "label": int(gold),
                    "pred": int(pred),
                    "response_raw": resp
                })

    # --------------------------------------------------------
    # 8) GLOBAL OOF METRICS
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("OOF GLOBAL METRICS (par k)")
    print("=" * 80)

    oof_df = pd.DataFrame(all_oof_rows)
    metrics_rows = []
    for k in K_VALUES:
        sub = oof_df[oof_df["k"] == k].copy()
        res = evaluate_predictions(sub["label"].values, sub["pred"].values)
        res["k"] = k
        res["fold"] = "ALL"
        metrics_rows.append(res)

        print(f"\n    k={k}")
        print(f"      MCC={res.get('mcc', np.nan):.4f} | BAcc={res.get('bacc', np.nan):.3f} | Acc={res.get('acc', np.nan):.3f}")
        print(f"      F1_macro={res.get('f1_macro', np.nan):.3f} | F1_oui={res.get('f1_oui', np.nan):.3f} | F1_non={res.get('f1_non', np.nan):.3f}")
        print(f"      Valides={res['n_valid']}/{res['n_total']} ({res['pct_valid']:.1f}%)")

    fold_metrics_df = pd.DataFrame(all_fold_metrics)
    global_metrics_df = pd.DataFrame(metrics_rows)

    # --------------------------------------------------------
    # 9) EXPORTS
    # --------------------------------------------------------
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_oof = os.path.join(OUTPUT_PATH, f"oof_predictions_{ts}.xlsx")
    out_metrics = os.path.join(OUTPUT_PATH, f"metrics_{ts}.xlsx")

    with pd.ExcelWriter(out_metrics) as writer:
        fold_metrics_df.to_excel(writer, sheet_name="fold_metrics", index=False)
        global_metrics_df.to_excel(writer, sheet_name="global_metrics", index=False)

    oof_df.to_excel(out_oof, index=False)

    print("\n" + "=" * 80)
    print("EXPORTS")
    print("=" * 80)
    print(f"    ✓ OOF:     {out_oof}")
    print(f"    ✓ Metrics: {out_metrics}")

    # --------------------------------------------------------
    # 10) PLOTS
    # --------------------------------------------------------
    if DO_PLOTS:
        try:
            plot_df = global_metrics_df.copy().sort_values("k")
            fig = plt.figure(figsize=(11, 5))

            # MCC
            plt.plot(plot_df["k"], plot_df["mcc"], marker="o", linewidth=2, label="MCC")
            # BAcc
            if "bacc" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["bacc"], marker="s", linewidth=2, label="Balanced Acc")
            # F1 macro
            if "f1_macro" in plot_df.columns:
                plt.plot(plot_df["k"], plot_df["f1_macro"], marker="^", linewidth=2, label="F1 macro")

            plt.xlabel("k (nb d'exemples few-shot)")
            plt.ylabel("Score")
            plt.title("OOF — Performances en fonction de k (retrieval NN dans 4 folds)")
            plt.xticks(K_VALUES)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()

            fig_path = os.path.join(OUTPUT_PATH, f"metrics_vs_k_{ts}.png")
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            plt.show()

            print(f"    ✓ Figure: {fig_path}")
        except Exception as e:
            print(f"[WARN] plot failed: {e}")

    # Cleanup
    del model_instruct, tokenizer_instruct
    _clear_cuda()

    print("\n" + "=" * 80)
    print("FIN")
    print("=" * 80)

    return oof_df, fold_metrics_df, global_metrics_df


if __name__ == "__main__":
    oof_df, fold_metrics_df, global_metrics_df = main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import os

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE_DIR = "artifacts/outputs_fewshot_llama_cv_nn"  # not shipped — see DATA.md
FILES_TO_CHECK = {
    "A2 (09 Jan - 13:17)": f"{BASE_DIR}/oof_predictions_20260109_131756.xlsx",
    "A1 (09 Jan - 13:35)": f"{BASE_DIR}/oof_predictions_20260109_133508.xlsx",
    "Gold   (10 Jan - 09:36)": f"{BASE_DIR}/oof_predictions_20260110_093656.xlsx"
}

K_VALUES = [0, 2, 4, 8]

# ==============================================================================
# 2. ANALYSE
# ==============================================================================
print(f"{'ANNOTATOR':<25} | {'K':<3} | {'RECALL (Capture)':<16} | {'FPR (Noise)':<15} | {'PRECISION':<12}")
print("-" * 85)

results_data = []

for name, full_path in FILES_TO_CHECK.items():

    if not os.path.exists(full_path):
        print(f"❌ {name:<25} : File not found.")
        continue

    try:
        df_all = pd.read_excel(full_path)
    except:
        continue

    # Loop over each K (0, 2, 4, 8)
    for k in K_VALUES:

        # --- FIX IS HERE: ROW-WISE FILTERING ---
        # Keep only rows matching the current 'k'
        df_k = df_all[df_all['k'] == k]

        if df_k.empty:
            continue

        y_true = df_k['label']
        y_pred = df_k['pred']

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        # Compute the business metrics
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0     # ability to find the Oui
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0        # Taux de pollution
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0  # reliability

        print(f"{name:<25} | {k:<3} | {recall:>14.1%} | {fpr:>13.1%} | {precision:>10.1%}")

        results_data.append({
            "Annotator": name,
            "K": k,
            "Recall": recall,
            "FPR": fpr,
            "Precision": precision,
            "TP (Found)": tp,
            "FN (Missed)": fn,
            "FP (Noise)": fp
        })

# ==============================================================================
# 3. SAVE AND CONCLUSION
# ==============================================================================
if results_data:
    df_res = pd.DataFrame(results_data)
    out_path = f"{BASE_DIR}/analyse_finale_recall_fpr.xlsx"
    df_res.to_excel(out_path, index=False)

    print("-" * 85)
    print(f"✅ Analysis saved: {out_path}")

    # Find the winner (Recall > 80% with the least noise possible)
    print("\n🏆 BEST CONFIGURATION (Recall > 80%):")
    candidates = df_res[df_res['Recall'] >= 0.80].sort_values('FPR')

    if not candidates.empty:
        best = candidates.iloc[0]
        print(f"   -> {best['Annotator']} with K={best['K']}")
        print(f"      Recall: {best['Recall']:.1%} | FPR: {best['FPR']:.1%}")
    else:
        print("   No config reaches 80% Recall. See the table for the best trade-off.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import os

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE_DIR = "artifacts/outputs_fewshot_llama_cv_nn"  # not shipped — see DATA.md
FILES_TO_CHECK = {
    "A2 (09 Jan - 13:17)": f"{BASE_DIR}/oof_predictions_20260109_131756.xlsx",
    "A1 (09 Jan - 13:35)": f"{BASE_DIR}/oof_predictions_20260109_133508.xlsx",
    "Gold   (10 Jan - 09:36)": f"{BASE_DIR}/oof_predictions_20260110_093656.xlsx"
}

K_VALUES = [0, 2, 4, 8]

# ==============================================================================
# 2. ANALYSE
# ==============================================================================
# Adjust the header to show volumes
print(f"{'ANNOTATOR':<25} | {'K':<3} | {'RETRIEVED / TOTAL':<18} | {'NUM PRED OUI':<12} | {'RECALL':<8} | {'PRECISION':<10}")
print("-" * 105)

results_data = []

for name, full_path in FILES_TO_CHECK.items():

    if not os.path.exists(full_path):
        print(f"❌ {name:<25} : File not found.")
        continue

    try:
        df_all = pd.read_excel(full_path)
    except:
        continue

    for k in K_VALUES:
        df_k = df_all[df_all['k'] == k]

        if df_k.empty:
            continue

        y_true = df_k['label']
        y_pred = df_k['pred']

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        # --- VOLUME COMPUTATIONS ---
        total_oui_reels = tp + fn      # the actual number of "Oui" in the dataset
        total_pred_oui = tp + fp       # the number of times the model said "Oui"

        # --- METRIC COMPUTATIONS ---
        recall = tp / total_oui_reels if total_oui_reels > 0 else 0.0
        precision = tp / total_pred_oui if total_pred_oui > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        # Build the "45/50" string for display
        ratio_recup = f"{tp}/{total_oui_reels}"

        print(f"{name:<25} | {k:<3} | {ratio_recup:>18} | {total_pred_oui:>12} | {recall:>8.1%} | {precision:>10.1%}")

        results_data.append({
            "Annotator": name,
            "K": k,
            "TP (Retrieved)": tp,
            "Total Actual Oui": total_oui_reels,
            "Num Predicted Oui": total_pred_oui,
            "Recall": recall,
            "Precision": precision,
            "FPR": fpr,
            "FP (Noise)": fp
        })

# ==============================================================================
# 3. SAVE
# ==============================================================================
if results_data:
    df_res = pd.DataFrame(results_data)
    # Reorder the columns for readability in Excel
    cols = ["Annotator", "K", "TP (Retrieved)", "Total Actual Oui", "Num Predicted Oui", "Recall", "Precision", "FP (Noise)", "FPR"]
    df_res = df_res[cols]

    out_path = f"{BASE_DIR}/analyse_volumes_recall.xlsx"
    df_res.to_excel(out_path, index=False)

    print("-" * 105)
    print(f"✅ Detailed analysis saved: {out_path}")

# LlaMa limited pool

In [ ]:
# ============================================================
# SCRIPT: Active Learning - Robustness (Llama 3 edition)
# METHOD: N random splits (150 Train / rest Test)
# STRATEGIES: Centroids (Global) vs Similarity (Contextuel)
# K-SHOTS: 0, 2, 4, 8
# ============================================================

import os
import re
import warnings
import gc
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    matthews_corrcoef,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_llama3_robustesse")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- MODEL CHANGE HERE (Llama 3.1) ---
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# --- EXPERIMENTAL PARAMETERS ---
N_ROUNDS = 5               # Number of random draws
N_ANNOTATED = 100          # Taille du set d'entrainement fixe
K_VALUES = [0, 2, 4, 8]    # The requested k values
STRATEGIES = ["centroid", "similarity"]

MAX_NEW_TOKENS = 10
MAX_PROMPT_LEN = 4096
EMBEDDING_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4

# ============================================================
# PROMPT & UTILS
# ============================================================
# --- BACK TO THE EXACT ORIGINAL GRID ---
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)
    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    if "unique_id" not in df.columns:
        df["unique_id"] = df.index.astype(str)

    print(f"    Total valid rows: {len(df)}")
    return df.reset_index(drop=True)

def create_text_for_embedding(row):
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"

# ============================================================
# EMBEDDINGS (optimized for Llama)
# ============================================================
def get_embeddings(texts, model, tokenizer):
    content_hash = hashlib.md5((str(len(texts)) + texts[0] + texts[-1] + MODEL_NAME).encode()).hexdigest()
    cache_file = os.path.join(CACHE_PATH, f"emb_llama_{content_hash}.npy")

    if os.path.exists(cache_file):
        print("    -> Embeddings loaded from cache.")
        return np.load(cache_file)

    print("    -> Computing embeddings (Llama)...")
    all_embs = []

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE)):
            batch = texts[i : i + EMBEDDING_BATCH_SIZE]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)
            outputs = model(**inputs, output_hidden_states=True)

            hidden = outputs.hidden_states[-1]
            mask = inputs["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
            embeddings = torch.sum(hidden * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

            all_embs.append(embeddings.cpu().numpy())
            _clear_cuda()

    result = np.vstack(all_embs)
    np.save(cache_file, result)
    return result

# ============================================================
# SELECTION LOGIC (CENTROID vs SIMILARITY)
# ============================================================
def get_examples_centroid(train_df, train_embs, k):
    if k == 0: return pd.DataFrame()

    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    centroid_oui = np.mean(train_embs[idx_oui], axis=0).reshape(1, -1)
    centroid_non = np.mean(train_embs[idx_non], axis=0).reshape(1, -1)

    sim_oui = cosine_similarity(train_embs[idx_oui], centroid_oui).flatten()
    sim_non = cosine_similarity(train_embs[idx_non], centroid_non).flatten()

    k_half = k // 2
    top_oui = idx_oui[np.argsort(sim_oui)[::-1][:k_half + (k%2)]]
    top_non = idx_non[np.argsort(sim_non)[::-1][:k_half]]

    indices = []
    for i in range(max(len(top_oui), len(top_non))):
        if i < len(top_oui): indices.append(top_oui[i])
        if i < len(top_non): indices.append(top_non[i])

    return train_df.iloc[indices].copy()

def get_examples_similarity(query_emb, train_df, train_embs, k):
    if k == 0: return pd.DataFrame()
    query_emb = query_emb.reshape(1, -1)

    sims = cosine_similarity(train_embs, query_emb).flatten()

    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    scores_oui = sims[idx_oui]
    scores_non = sims[idx_non]

    k_half = k // 2
    best_local_oui = np.argsort(scores_oui)[::-1][:k_half + (k%2)]
    best_local_non = np.argsort(scores_non)[::-1][:k_half]

    final_indices = np.concatenate([idx_oui[best_local_oui], idx_non[best_local_non]])
    return train_df.iloc[final_indices].copy()

# ============================================================
# PROMPT CONSTRUCTION
# ============================================================
def build_prompt(row, examples_df):
    messages = [{"role": "system", "content": GRILLE_ANNOTATION}]

    # Same structure as the original script: put the examples in the User content
    # Unless a conversational format is preferred; here the reference-examples structure is kept
    user_txt = ""
    if not examples_df.empty:
        user_txt += "Exemples de référence :\n\n"
        for _, ex in examples_df.iterrows():
            lbl = "oui" if ex["label"] == 1 else "non"
            user_txt += f"Extrait: {str(ex['text'])[:300]}...\nArticle: {str(ex['article_text'])[:200]}...\nRéponse: {lbl}\n\n"

    user_txt += f"--- CAS À TRAITER ---\nExtrait: {str(row['text'])}\nArticle: {str(row['article_text'])}\n\nRéponse :"
    messages.append({"role": "user", "content": user_txt})

    return messages

def parse_output(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    if text.startswith("oui"): return 1
    if text.startswith("non"): return 0
    return -1

# ============================================================
# MAIN LOOP
# ============================================================
def main():
    df = load_and_prepare_data(EXCEL_PATH)

    print(f"[2] Loading model {MODEL_NAME}...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN
    )

    print("[3] Computing embeddings (Global)...")
    texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
    embs_all = get_embeddings(texts_all, model, tokenizer)

    detailed_results = []

    print(f"\n[4] Launching {N_ROUNDS} validation rounds...")

    for round_idx in range(N_ROUNDS):
        current_seed = SEED + round_idx
        print(f"\n{'='*60}")
        print(f"ROUND {round_idx + 1}/{N_ROUNDS} (Seed={current_seed})")
        print(f"{'='*60}")

        train_df = df.sample(n=N_ANNOTATED, random_state=current_seed)
        test_df = df.drop(train_df.index)

        train_indices = [df.index.get_loc(i) for i in train_df.index]
        test_indices = [df.index.get_loc(i) for i in test_df.index]
        train_embs = embs_all[train_indices]
        test_embs = embs_all[test_indices]

        for k in K_VALUES:
            for strategy in STRATEGIES:

                if k == 0 and strategy == "similarity":
                    continue

                run_name = f"Round{round_idx+1}_k{k}_{strategy}"
                if k == 0: run_name = f"Round{round_idx+1}_k0_zeroshot"

                print(f"  -> Config: {run_name}")

                fixed_examples = None
                if k > 0 and strategy == "centroid":
                    fixed_examples = get_examples_centroid(train_df, train_embs, k)

                prompts_data = []
                for i in range(len(test_df)):
                    row = test_df.iloc[i]

                    if k == 0: examples = pd.DataFrame()
                    elif strategy == "centroid": examples = fixed_examples
                    else: examples = get_examples_similarity(test_embs[i], train_df, train_embs, k)

                    msgs = build_prompt(row, examples)
                    ex_ids = str(examples.index.tolist()) if not examples.empty else "[]"
                    prompts_data.append({"row": row, "msgs": msgs, "ex_ids": ex_ids})

                texts_gen = [tokenizer.apply_chat_template(p["msgs"], tokenize=False, add_generation_prompt=True) for p in prompts_data]

                for i in tqdm(range(0, len(texts_gen), GENERATION_BATCH_SIZE), desc="    Gen", leave=False):
                    batch = texts_gen[i:i+GENERATION_BATCH_SIZE]

                    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

                    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_PROMPT_LEN).to(model.device)

                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens=MAX_NEW_TOKENS,
                            pad_token_id=tokenizer.pad_token_id,
                            do_sample=False
                        )

                    for idx_b, seq in enumerate(out):
                        input_len = inputs["input_ids"][idx_b].shape[0]
                        gen_txt = tokenizer.decode(seq[input_len:], skip_special_tokens=True)
                        val = parse_output(gen_txt)

                        p_info = prompts_data[i + idx_b]
                        detailed_results.append({
                            "round_id": round_idx + 1,
                            "decision_id": p_info["row"].get("decision_id"),
                            "strategy": strategy if k > 0 else "zero_shot",
                            "k": k,
                            "true_label": p_info["row"]["label"],
                            "pred_label": val,
                            "raw_output": gen_txt,
                            "few_shot_ids": p_info["ex_ids"]
                        })
                _clear_cuda()

    print("\n[5] Analyse et Sauvegarde...")
    res_df = pd.DataFrame(detailed_results)

    round_stats = []
    for (strat, k_val, r_id), group in res_df.groupby(["strategy", "k", "round_id"]):
        valid = group[group["pred_label"] != -1]
        if len(valid) == 0:
            acc, mcc = 0, 0
        else:
            acc = accuracy_score(valid["true_label"], valid["pred_label"])
            mcc = matthews_corrcoef(valid["true_label"], valid["pred_label"])

        round_stats.append({
            "strategy": strat,
            "k": k_val,
            "round": r_id,
            "acc": acc,
            "mcc": mcc
        })

    stats_df = pd.DataFrame(round_stats)

    final_summary = stats_df.groupby(["strategy", "k"]).agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        mcc_mean=("mcc", "mean"),
        mcc_std=("mcc", "std")
    ).reset_index()

    print(f"\n=== MEAN RESULTS (Llama 3 - {N_ROUNDS} Rounds) ===")
    print(final_summary)

    ts = datetime.now().strftime("%Y%m%d_%H%M")
    outfile = os.path.join(OUTPUT_PATH, f"llama3_robustesse_results_{ts}.xlsx")
    with pd.ExcelWriter(outfile) as writer:
        final_summary.to_excel(writer, sheet_name="Global_Summary")
        stats_df.to_excel(writer, sheet_name="Per_Round_Stats")
        res_df.to_excel(writer, sheet_name="All_Predictions", index=False)

    print(f"\nSaved: {outfile}")

if __name__ == "__main__":
    main()

# LLama 100 gold

In [ ]:
# ============================================================
# SCRIPT: Active Learning - Robustness (Llama 3 edition)
# METHOD: N random splits (100 Train / rest Test)
# STRATEGIES: Centroids (Global) vs Similarity (Contextuel)
# K-SHOTS: 0, 2, 4, 8
# VERSION: bf16 (without quantization)
# ============================================================

import os
import re
import warnings
import gc
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.metrics.pairwise import cosine_similarity

# Torch & Transformers
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_llama3_robustesse_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_PATH = os.path.join(OUTPUT_PATH, "_cache")
os.makedirs(CACHE_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- MODELE ---
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# --- EXPERIMENTAL PARAMETERS ---
N_ROUNDS = 5               # Number of random draws
N_ANNOTATED = 100          # Fixed training-set size (chosen at random)
K_VALUES = [0, 2, 4, 8]    # The requested k values
STRATEGIES = ["centroid", "similarity"]
FIXED_SEEDS = [42, 123, 456, 789, 1011]  # Fixed seeds for reproducibility

MAX_NEW_TOKENS = 10
MAX_PROMPT_LEN = 4096
EMBEDDING_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 4

# ============================================================
# PROMPT & UTILS
# ============================================================
GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_and_prepare_data(excel_path):
    print("[1] Loading data...")
    df = pd.read_csv(excel_path)

    def extract_oui_non(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s): return "non"
        return np.nan

    df["a_str"] = df["eval_A1"].apply(extract_oui_non)
    df["t_str"] = df["eval_A2"].apply(extract_oui_non)
    df["s_str"] = df["eval_A3"].apply(extract_oui_non)
    same_mask = (df["a_str"].isin(["oui","non"])) & (df["a_str"] == df["t_str"])
    df["label_str"] = np.where(same_mask, df["a_str"], df["s_str"])
    df = df[df["label_str"].isin(["oui", "non"])].copy()
    df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

    if "unique_id" not in df.columns:
        df["unique_id"] = df.index.astype(str)

    print(f"    Total valid rows: {len(df)}")
    return df.reset_index(drop=True)

def create_text_for_embedding(row):
    return f"Article: {str(row.get('article_text', ''))}\nExtrait: {str(row.get('text', ''))}"

# ============================================================
# EMBEDDINGS (bf16)
# ============================================================
def get_embeddings(texts, model, tokenizer):
    content_hash = hashlib.md5((str(len(texts)) + texts[0] + texts[-1] + MODEL_NAME + "_bf16").encode()).hexdigest()
    cache_file = os.path.join(CACHE_PATH, f"emb_llama_bf16_{content_hash}.npy")

    if os.path.exists(cache_file):
        print("    -> Embeddings loaded from cache.")
        return np.load(cache_file)

    print("    -> Computing embeddings (Llama bf16)...")
    all_embs = []

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE)):
            batch = texts[i : i + EMBEDDING_BATCH_SIZE]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(model.device)
            outputs = model(**inputs, output_hidden_states=True)

            hidden = outputs.hidden_states[-1]
            mask = inputs["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
            embeddings = torch.sum(hidden * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

            all_embs.append(embeddings.cpu().float().numpy())  # Convert to float32 for numpy
            _clear_cuda()

    result = np.vstack(all_embs)
    np.save(cache_file, result)
    return result

# ============================================================
# SELECTION LOGIC (CENTROID vs SIMILARITY)
# ============================================================
def get_examples_centroid(train_df, train_embs, k):
    if k == 0: return pd.DataFrame()

    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    centroid_oui = np.mean(train_embs[idx_oui], axis=0).reshape(1, -1)
    centroid_non = np.mean(train_embs[idx_non], axis=0).reshape(1, -1)

    sim_oui = cosine_similarity(train_embs[idx_oui], centroid_oui).flatten()
    sim_non = cosine_similarity(train_embs[idx_non], centroid_non).flatten()

    k_half = k // 2
    top_oui = idx_oui[np.argsort(sim_oui)[::-1][:k_half + (k%2)]]
    top_non = idx_non[np.argsort(sim_non)[::-1][:k_half]]

    indices = []
    for i in range(max(len(top_oui), len(top_non))):
        if i < len(top_oui): indices.append(top_oui[i])
        if i < len(top_non): indices.append(top_non[i])

    return train_df.iloc[indices].copy()

def get_examples_similarity(query_emb, train_df, train_embs, k):
    if k == 0: return pd.DataFrame()
    query_emb = query_emb.reshape(1, -1)

    sims = cosine_similarity(train_embs, query_emb).flatten()

    idx_oui = np.where(train_df["label"].values == 1)[0]
    idx_non = np.where(train_df["label"].values == 0)[0]

    scores_oui = sims[idx_oui]
    scores_non = sims[idx_non]

    k_half = k // 2
    best_local_oui = np.argsort(scores_oui)[::-1][:k_half + (k%2)]
    best_local_non = np.argsort(scores_non)[::-1][:k_half]

    final_indices = np.concatenate([idx_oui[best_local_oui], idx_non[best_local_non]])
    return train_df.iloc[final_indices].copy()

# ============================================================
# PROMPT CONSTRUCTION
# ============================================================
def build_prompt(row, examples_df):
    messages = [{"role": "system", "content": GRILLE_ANNOTATION}]

    user_txt = ""
    if not examples_df.empty:
        user_txt += "Exemples de référence :\n\n"
        for _, ex in examples_df.iterrows():
            lbl = "oui" if ex["label"] == 1 else "non"
            user_txt += f"Extrait: {str(ex['text'])[:300]}...\nArticle: {str(ex['article_text'])[:200]}...\nRéponse: {lbl}\n\n"

    user_txt += f"--- CAS À TRAITER ---\nExtrait: {str(row['text'])}\nArticle: {str(row['article_text'])}\n\nRéponse :"
    messages.append({"role": "user", "content": user_txt})

    return messages

def parse_output(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    if text.startswith("oui"): return 1
    if text.startswith("non"): return 0
    return -1

# ============================================================
# MAIN LOOP
# ============================================================
def main():
    df = load_and_prepare_data(EXCEL_PATH)

    print(f"[2] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    # --- CHARGEMENT EN BF16 (without quantization) ---
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=HF_TOKEN
    )

    print("[3] Computing embeddings (Global)...")
    texts_all = df.apply(create_text_for_embedding, axis=1).tolist()
    embs_all = get_embeddings(texts_all, model, tokenizer)

    detailed_results = []

    print(f"\n[4] Launching {N_ROUNDS} validation rounds...")

    for round_idx in range(N_ROUNDS):
        current_seed = FIXED_SEEDS[round_idx]  # predefined fixed seed
        print(f"\n{'='*60}")
        print(f"ROUND {round_idx + 1}/{N_ROUNDS} (Seed={current_seed})")
        print(f"{'='*60}")

        # The 100 examples are chosen AT RANDOM here
        train_df = df.sample(n=N_ANNOTATED, random_state=current_seed)
        test_df = df.drop(train_df.index)

        train_indices = [df.index.get_loc(i) for i in train_df.index]
        test_indices = [df.index.get_loc(i) for i in test_df.index]
        train_embs = embs_all[train_indices]
        test_embs = embs_all[test_indices]

        for k in K_VALUES:
            for strategy in STRATEGIES:

                if k == 0 and strategy == "similarity":
                    continue

                run_name = f"Round{round_idx+1}_k{k}_{strategy}"
                if k == 0: run_name = f"Round{round_idx+1}_k0_zeroshot"

                print(f"  -> Config: {run_name}")

                fixed_examples = None
                if k > 0 and strategy == "centroid":
                    fixed_examples = get_examples_centroid(train_df, train_embs, k)

                prompts_data = []
                for i in range(len(test_df)):
                    row = test_df.iloc[i]

                    if k == 0: examples = pd.DataFrame()
                    elif strategy == "centroid": examples = fixed_examples
                    else: examples = get_examples_similarity(test_embs[i], train_df, train_embs, k)

                    msgs = build_prompt(row, examples)
                    ex_ids = str(examples.index.tolist()) if not examples.empty else "[]"
                    prompts_data.append({"row": row, "msgs": msgs, "ex_ids": ex_ids})

                texts_gen = [tokenizer.apply_chat_template(p["msgs"], tokenize=False, add_generation_prompt=True) for p in prompts_data]

                for i in tqdm(range(0, len(texts_gen), GENERATION_BATCH_SIZE), desc="    Gen", leave=False):
                    batch = texts_gen[i:i+GENERATION_BATCH_SIZE]

                    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

                    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_PROMPT_LEN).to(model.device)

                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens=MAX_NEW_TOKENS,
                            pad_token_id=tokenizer.pad_token_id,
                            do_sample=False
                        )

                    for idx_b, seq in enumerate(out):
                        input_len = inputs["input_ids"][idx_b].shape[0]
                        gen_txt = tokenizer.decode(seq[input_len:], skip_special_tokens=True)
                        val = parse_output(gen_txt)

                        p_info = prompts_data[i + idx_b]
                        detailed_results.append({
                            "round_id": round_idx + 1,
                            "decision_id": p_info["row"].get("decision_id"),
                            "strategy": strategy if k > 0 else "zero_shot",
                            "k": k,
                            "true_label": p_info["row"]["label"],
                            "pred_label": val,
                            "raw_output": gen_txt,
                            "few_shot_ids": p_info["ex_ids"]
                        })
                _clear_cuda()

    print("\n[5] Analyse et Sauvegarde...")
    res_df = pd.DataFrame(detailed_results)

    round_stats = []
    for (strat, k_val, r_id), group in res_df.groupby(["strategy", "k", "round_id"]):
        valid = group[group["pred_label"] != -1]
        if len(valid) == 0:
            acc, mcc, prec_oui, rec_oui = 0, 0, 0, 0
            tp, fp, fn, tn = 0, 0, 0, 0
            nb_pred_oui, nb_total = 0, 0
        else:
            y_true = valid["true_label"].values
            y_pred = valid["pred_label"].values

            acc = accuracy_score(y_true, y_pred)
            mcc = matthews_corrcoef(y_true, y_pred)

            # Precision and Recall on "oui" (label=1)
            prec_oui = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
            rec_oui = recall_score(y_true, y_pred, pos_label=1, zero_division=0)

            # Confusion matrix : [[TN, FP], [FN, TP]]
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()

            nb_pred_oui = int((y_pred == 1).sum())
            nb_total = len(y_pred)

        round_stats.append({
            "strategy": strat,
            "k": k_val,
            "round": r_id,
            "acc": acc,
            "mcc": mcc,
            "precision_oui": prec_oui,
            "recall_oui": rec_oui,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "nb_pred_oui": nb_pred_oui,
            "nb_total": nb_total
        })

    stats_df = pd.DataFrame(round_stats)

    final_summary = stats_df.groupby(["strategy", "k"]).agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        mcc_mean=("mcc", "mean"),
        mcc_std=("mcc", "std"),
        precision_oui_mean=("precision_oui", "mean"),
        precision_oui_std=("precision_oui", "std"),
        recall_oui_mean=("recall_oui", "mean"),
        recall_oui_std=("recall_oui", "std"),
        TP_total=("TP", "sum"),
        FP_total=("FP", "sum"),
        FN_total=("FN", "sum"),
        TN_total=("TN", "sum"),
        nb_pred_oui_total=("nb_pred_oui", "sum"),
        nb_total_total=("nb_total", "sum")
    ).reset_index()

    print(f"\n=== MEAN RESULTS (Llama 3 bf16 - {N_ROUNDS} Rounds) ===")
    print(final_summary[["strategy", "k", "acc_mean", "acc_std", "mcc_mean", "mcc_std",
                         "precision_oui_mean", "recall_oui_mean", "FP_total", "nb_pred_oui_total"]])

    # Show the aggregated confusion matrix per config
    print("\n=== AGGREGATED CONFUSION MATRICES ===")
    for _, row in final_summary.iterrows():
        print(f"\n{row['strategy']} k={row['k']}:")
        print(f"  [[TN={row['TN_total']}, FP={row['FP_total']}], [FN={row['FN_total']}, TP={row['TP_total']}]]")
        print(f"  Pred Oui: {row['nb_pred_oui_total']}/{row['nb_total_total']}")

    ts = datetime.now().strftime("%Y%m%d_%H%M")

    # Save to Excel
    outfile = os.path.join(OUTPUT_PATH, f"llama3_bf16_robustesse_results_{ts}.xlsx")
    with pd.ExcelWriter(outfile) as writer:
        final_summary.to_excel(writer, sheet_name="Global_Summary", index=False)
        stats_df.to_excel(writer, sheet_name="Per_Round_Stats", index=False)
        res_df.to_excel(writer, sheet_name="All_Predictions", index=False)

    print(f"\nSaved Excel: {outfile}")

    # Save OOF (Out-Of-Fold) as pickle for reuse
    oof_data = {
        "predictions": res_df,
        "per_round_stats": stats_df,
        "summary": final_summary,
        "config": {
            "model": MODEL_NAME,
            "n_rounds": N_ROUNDS,
            "n_annotated": N_ANNOTATED,
            "k_values": K_VALUES,
            "strategies": STRATEGIES,
            "fixed_seeds": FIXED_SEEDS
        }
    }
    oof_file = os.path.join(OUTPUT_PATH, f"llama3_bf16_OOF_{ts}.pkl")
    pd.to_pickle(oof_data, oof_file)
    print(f"Saved OOF: {oof_file}")

if __name__ == "__main__":
    main()

# LLaMa 0 shot 4 bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Llama 3.1 8B (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
# Change here: Llama-specific output folder
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_llama_0shot_analysis")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Change here: Llama 3.1 model ID
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Set your token here (Llama 3 is gated; accept the license on HF)
HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 8
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Llama 3.1 handles system/user roles well via apply_chat_template
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Llama 3.1) on {len(df)} rows...")
    all_preds, all_raw = [], []

    # apply_chat_template automatically handles Llama-specific formatting (<|begin_of_text|>, <|start_header_id|>, etc.)
    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        # Llama sometimes needs a pad_token set explicitly if not already done
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id # Important pour Llama
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    # Change here: column renaming for Llama
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Llama_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME}...")

    # Quantization config (recommended for a 16 GB T4/L4 GPU)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    # Llama 3 sometimes has a specific pad token, but it often has to be set manually for batching
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    # Change here: columns stored in the main DF
    df["Llama_Pred"] = preds
    df["Llama_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Llama 3.1) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_llama_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_llama_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# LLama 0 shot ATS 16bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Llama 3.1 8B (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_llama_0shot_analysis_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

HF_TOKEN = os.environ["HF_TOKEN"]

# Aligned with doc 1
BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Llama 3.1 bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Llama_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bf16 (comme doc 1)
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Llama_Pred"] = preds
    df["Llama_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Llama 3.1 bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_llama_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_llama_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Mistral Nemo 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Mistral-Nemo 12B (0-shot) — Metrics + "OOF" saves
# VERSION fp32/bf16 selon GPU disponible (A100)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_mistral_0shot_analysis")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mistral-Nemo 12B Instruct (Mistral's "14B")
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"
HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

# Precision choice: "fp32" for A100-80GB, "bf16" for A100-40GB
PRECISION = "bf16"  # Change en "fp32" si tu as 80 Go

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def get_gpu_memory():
    if torch.cuda.is_available():
        return torch.cuda.get_device_properties(0).total_memory / 1e9
    return 0

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Mistral {PRECISION}) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Mistral_Pred", "Mistral_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Mistral_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    # Automatic GPU detection
    gpu_mem = get_gpu_memory()
    print(f"\n[GPU] VRAM detected: {gpu_mem:.1f} Go")

    # dtype selection
    if PRECISION == "fp32":
        dtype = torch.float32
        precision_str = "fp32"
        print(f"[Init] Chargement en float32 (~48 Go VRAM)")
    else:
        dtype = torch.bfloat16
        precision_str = "bf16"
        print(f"[Init] Chargement in bfloat16 (~24 Go VRAM)")

    print(f"[Init] Loading model {MODEL_NAME}...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    print(f"    ✓ Model loaded in {precision_str}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Mistral_Pred"] = preds
    df["Mistral_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print(f"\n=== RESULTS (Mistral {precision_str}) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_mistral_{precision_str}_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Mistral_Pred", "Mistral_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_mistral_{precision_str}_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Magistral 0 shot

In [ ]:
import shutil
shutil.rmtree("/root/.cache/huggingface/hub", ignore_errors=True)

# 2nd try magistral

In [ ]:
# ============================================================
# SCRIPT: Evaluation Magistral-Small 24B (VERSION FINALE)
# With a toggleable REASONING option
# ============================================================
import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import subprocess
import torch
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

# === MAIN PARAMETER: REASONING MODE ===
ENABLE_REASONING = True  # False = direct oui/non answer (FAST)
                          # True = the model reasons before answering (SLOW)

# Chemins
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_magistral_final")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Model
MODEL_NAME = "mistralai/Magistral-Small-2509"
HF_TOKEN = os.environ["HF_TOKEN"]

# Generation parameters per the Mistral docs
SEED = 42
BATCH_SIZE = 1  # 1 to avoid padding issues with this model

# Tokens to generate
if ENABLE_REASONING:
    MAX_NEW_TOKENS = 2048  # Reasoning = needs more tokens
else:
    MAX_NEW_TOKENS = 30    # No reasoning = short answer

np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# PROMPTS
# ------------------------------------------------------------

# Prompt WITHOUT reasoning (direct answer)
SYSTEM_PROMPT_SIMPLE = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# Prompt WITH reasoning (the model reasons)
SYSTEM_PROMPT_REASONING = """Tu es un expert en droit civil français.

Tâche: Déterminer si l'article de loi fourni est implicitement appliqué dans l'extrait de décision.

Instructions:
1. Analyse d'abord le texte et l'article
2. Réfléchis à la correspondance juridique
3. Conclus par une ligne finale contenant UNIQUEMENT "oui" ou "non"

Règles de décision:
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique).""".strip()

SYSTEM_PROMPT = SYSTEM_PROMPT_REASONING if ENABLE_REASONING else SYSTEM_PROMPT_SIMPLE

# ------------------------------------------------------------
# 1. DEPENDENCY INSTALLATION
# ------------------------------------------------------------
def install_deps():
    print("[0] Checking dependencies...")
    try:
        import mistral_common
        import transformers
        print(f"    ✓ mistral_common: {mistral_common.__version__}")
        print(f"    ✓ transformers: {transformers.__version__}")
    except ImportError:
        print("    Installing dependencies...")
        subprocess.run([
            "pip", "install", "-q", "--upgrade",
            "transformers", "mistral-common>=1.8.5", "accelerate",
            "--break-system-packages"
        ], check=False)

# ------------------------------------------------------------
# 2. UTILITY FUNCTIONS
# ------------------------------------------------------------
def clean_label(x):
    """Convert an annotation to 0/1/NaN"""
    if pd.isna(x):
        return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]:
        return 1
    if s in ["non", "0", "no"]:
        return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s):
        return 1
    if re.search(r"\bnon\b", s):
        return 0
    return np.nan

def load_data(path):
    """
    Load the data and correctly build the Gold Standard:
    - If A2 and A1 agree → their value
    - If they disagree → A3 decides (gold standard)
    - Otherwise → take A1 or A2 if available
    """
    print(f"[1] Loading data: {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(path)

    # Check the required columns
    for c in ["text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    # Convert the annotations to numeric labels
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col in df.columns:
            df[f"lbl_{col.split('_')[1]}"] = df[col].apply(clean_label)
        else:
            df[f"lbl_{col.split('_')[1]}"] = np.nan

    def get_gold(row):
        """
        Calcul du Gold Standard:
        1. If A2 AND A1 agree → their common value
        2. If A2 and A1 disagree → A3 decides
        3. Si une seule annotation disponible → on la prend
        """
        t = row.get("lbl_A2", np.nan)
        a = row.get("lbl_A1", np.nan)
        s = row.get("lbl_A3", np.nan)

        t_valid = pd.notna(t)
        a_valid = pd.notna(a)
        s_valid = pd.notna(s)

        # Case 1: the two main annotators agree
        if t_valid and a_valid and t == a:
            return int(t)

        # Case 2: disagreement → A3 decides
        if t_valid and a_valid and t != a:
            if s_valid:
                return int(s)  # A3 = gold standard in case of disagreement
            else:
                return np.nan  # Pas de gold sans arbitrage

        # Case 3: only one main annotation available
        if t_valid and not a_valid:
            return int(t)
        if a_valid and not t_valid:
            return int(a)

        # Cas 4: Seulement A3 disponible
        if s_valid:
            return int(s)

        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # Gold stats
    n_total = len(df)
    n_gold = df["lbl_Gold"].notna().sum()
    n_agree = ((df["lbl_A2"] == df["lbl_A1"]) & df["lbl_A2"].notna()).sum()
    n_disagree = ((df["lbl_A2"] != df["lbl_A1"]) & df["lbl_A2"].notna() & df["lbl_A1"].notna()).sum()

    print(f"    Total rows: {n_total}")
    print(f"    Accords A2-A1: {n_agree}")
    print(f"    Disagreements (A3 decides): {n_disagree}")
    print(f"    Gold labels valides: {n_gold}")

    # Keep only rows with text
    df = df.dropna(subset=["text", "article_text"]).reset_index(drop=True)
    print(f"    Rows with valid text: {len(df)}")

    return df

def parse_response(response_text):
    """Extract 0 (non) or 1 (oui) from the model's response"""
    text = str(response_text).strip()

    # 1. Strip thinking tags if present
    for tag in ["[/think]", "</think>", "[/THINK]", "</THINK>"]:
        if tag.lower() in text.lower():
            parts = re.split(tag, text, flags=re.IGNORECASE)
            text = parts[-1].strip()

    text_lower = text.lower()
    text_clean = re.sub(r"[^\w\sàâäéèêëïîôùûüç]", " ", text_lower).strip()

    # 2. Match exact
    if text_clean in ["oui", "yes"]:
        return 1
    if text_clean in ["non", "no"]:
        return 0

    # 3. Starts with oui/non
    if text_clean.startswith("oui"):
        return 1
    if text_clean.startswith("non"):
        return 0

    # 4. Look in the last 100 characters (the conclusion)
    end_text = text_lower[-100:] if len(text_lower) > 100 else text_lower

    has_oui = bool(re.search(r"\boui\b", end_text))
    has_non = bool(re.search(r"\bnon\b", end_text))

    if has_non and not has_oui:
        return 0
    if has_oui and not has_non:
        return 1

    # 5. Last occurrence in the whole text
    matches = list(re.finditer(r"\b(oui|non)\b", text_lower))
    if matches:
        last_match = matches[-1].group(1)
        return 1 if last_match == "oui" else 0

    return -1  # Non classifiable

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. INFERENCE
# ------------------------------------------------------------
def run_inference(df, model, tokenizer):
    """Run inference over all rows"""
    preds_text = []

    mode_str = "WITH reasoning" if ENABLE_REASONING else "WITHOUT reasoning (direct answer)"
    print(f"\n[2] Inference over {len(df)} rows - Mode: {mode_str}")
    print(f"    Max tokens: {MAX_NEW_TOKENS}")

    for i, (idx, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Generation")):

        # Build the message
        user_content = f"Article: {row['article_text']}\n\nExtrait de décision: {row['text']}"

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]

        # Tokenization with apply_chat_template
        # Note: use tokenize=True to avoid the warning
        try:
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(model.device)
        except Exception as e:
            # Fallback on error
            prompt = tokenizer.apply_chat_template(messages, tokenize=False)
            inputs = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

        prompt_len = inputs.shape[1]

        # Generation
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,    # Mistral-recommended parameter
                top_p=0.95,         # Mistral-recommended parameter
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decode only the new tokens
        generated_ids = outputs[0][prompt_len:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        preds_text.append(response)

        # Show the first examples
        if i < 3:
            pred = parse_response(response)
            pred_label = {-1: "❌ INVALID", 0: "NON", 1: "OUI"}.get(pred)
            print(f"\n{'='*50}")
            print(f"EXEMPLE {i+1}")
            print(f"{'='*50}")
            print(f"📄 Article: {row['article_text'][:100]}...")
            print(f"📝 Decision: {row['text'][:100]}...")
            print(f"🤖 Response: {response[:200]}{'...' if len(response) > 200 else ''}")
            print(f"✅ Parsed: {pred_label}")

        # Periodic memory cleanup
        if i % 50 == 0:
            _clear_cuda()

    return preds_text

# ------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------
def calculate_metrics(y_true, y_pred, name=""):
    """Compute the classification metrics"""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    # Mask for the valid predictions
    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    result = {
        "Target": name,
        "N_Total": len(y_pred),
        "N_Valid": len(yt),
        "N_Invalid": int(np.sum(y_pred == -1)),
        "Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "F1": np.nan,
        "MCC": np.nan
    }

    if len(yt) == 0:
        return result, np.array([[0,0],[0,0]])

    cm = confusion_matrix(yt, yp, labels=[0, 1])

    result.update({
        "Accuracy": round(accuracy_score(yt, yp), 4),
        "Precision": round(precision_score(yt, yp, pos_label=1, zero_division=0), 4),
        "Recall": round(recall_score(yt, yp, pos_label=1, zero_division=0), 4),
        "F1": round(f1_score(yt, yp, pos_label=1, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0, 4)
    })

    return result, cm

def print_confusion_matrix(cm, name):
    print(f"\n--- Confusion Matrix ({name}) ---")
    print(f"              Pred_NON  Pred_OUI")
    print(f"  True_NON    {cm[0,0]:6d}    {cm[0,1]:6d}")
    print(f"  True_OUI    {cm[1,0]:6d}    {cm[1,1]:6d}")

# ------------------------------------------------------------
# 5. MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    print("=" * 60)
    print("MAGISTRAL-SMALL EVALUATION")
    print(f"Mode: {'WITH' if ENABLE_REASONING else 'WITHOUT'} reasoning")
    print("=" * 60)

    # A. Dependencies
    install_deps()

    # B. Load the data
    df = load_data(EXCEL_PATH)

    # C. Load the model
    print("\n[Init] Loading model...")

    from transformers import AutoTokenizer

    # Import the right model class
    try:
        from transformers import Mistral3ForConditionalGeneration
        ModelClass = Mistral3ForConditionalGeneration
        print("    Utilisation de Mistral3ForConditionalGeneration")
    except ImportError:
        from transformers import AutoModelForCausalLM
        ModelClass = AutoModelForCausalLM
        print("    Using AutoModelForCausalLM (fallback)")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = ModelClass.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    print(f"    ✓ Model loaded: {MODEL_NAME}")
    print(f"    ✓ Vocab size: {tokenizer.vocab_size}")

    # D. Inference
    raw_responses = run_inference(df, model, tokenizer)

    # E. Post-processing
    print("\n[3] Analyzing results...")

    parsed_preds = [parse_response(txt) for txt in raw_responses]
    df["Magistral_Raw"] = raw_responses
    df["Magistral_Pred"] = parsed_preds

    # F. Metrics
    results = []
    cms = {}

    for col, name in [("lbl_Gold", "GOLD"), ("lbl_A2", "A2"), ("lbl_A1", "A1")]:
        if col in df.columns:
            r, cm = calculate_metrics(df[col].values, np.array(parsed_preds), name=name)
            results.append(r)
            cms[name] = cm

    # Display
    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)

    results_df = pd.DataFrame(results)
    print(results_df[["Target", "N_Valid", "N_Invalid", "Accuracy", "Precision", "Recall", "F1", "MCC"]].to_string(index=False))

    for name, cm in cms.items():
        print_confusion_matrix(cm, name)

    # Stats on invalid responses
    n_invalid = sum(1 for p in parsed_preds if p == -1)
    if n_invalid > 0:
        print(f"\n⚠️ {n_invalid} non-classifiable responses ({100*n_invalid/len(parsed_preds):.1f}%)")
        print("Examples of invalid responses:")
        for i, (pred, raw) in enumerate(zip(parsed_preds, raw_responses)):
            if pred == -1 and i < 5:
                print(f"  [{i}] '{raw[:100]}...'")

    # G. Save
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    mode_suffix = "reasoning" if ENABLE_REASONING else "direct"

    # Full results file
    output_file = os.path.join(OUTPUT_PATH, f"magistral_{mode_suffix}_{timestamp}.xlsx")
    df.to_excel(output_file, index=False)

    # Metrics file
    metrics_file = os.path.join(OUTPUT_PATH, f"metrics_{mode_suffix}_{timestamp}.xlsx")
    results_df.to_excel(metrics_file, index=False)

    print(f"\n✅ Done!")
    print(f"   Results: {output_file}")
    print(f"   Metrics: {metrics_file}")

# Qwen 32 B

In [ ]:
# ============================================================
# SCRIPT: Evaluation Qwen 2.5 32B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_qwen32B_0shot_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for a 32B in bfloat16 (~64 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (32-bit) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Qwen_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~64 GB of VRAM for a 32B model in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Qwen_Pred"] = preds
    df["Qwen_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Qwen_Pred", "Qwen_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Gemma 27 B

In [ ]:
# ============================================================
# SCRIPT: Evaluation Gemma 27B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_gemma27B_0shot_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "google/gemma-2-27b-it"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for a 27B in bfloat16 (~54 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Gemma 2 uses a chat format with <start_of_turn> and <end_of_turn>
    # But with apply_chat_template, we use the standard format
    messages = [
        {"role": "user", "content": f"""{GRILLE_ANNOTATION}

Article: {str(row['article_text'])}
Extrait: {str(row['text'])}

Réponse (oui/non) :"""}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Gemma_Pred", "Gemma_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["Gemma_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~54 GB of VRAM for a 27B model in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["Gemma_Pred"] = preds
    df["Gemma_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Gemma_Pred", "Gemma_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Command R 35B 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Command R (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_commandr_0shot_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "CohereForAI/c4ai-command-r-v01"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for Command R 35B in bfloat16 (~70 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Command R supports the standard format with system + user
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "CommandR_Pred", "CommandR_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["CommandR_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~70 GB of VRAM for Command R 35B in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["CommandR_Pred"] = preds
    df["CommandR_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "CommandR_Pred", "CommandR_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Aya 32B 0 shot

In [ ]:
# ============================================================
# SCRIPT: Evaluation Aya Expanse 32B (0-shot) — Metrics + "OOF" saves
# VERSION: bfloat16 (half precision, no quantization)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_aya_expanse_32B_0shot_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "CohereForAI/aya-expanse-32b"
HF_TOKEN = os.environ["HF_TOKEN"]

# Batch size for Aya Expanse 32B in bfloat16 (~64 GB VRAM)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 64
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    # regex robuste
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    # minimal columns
    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    # annotation columns (if missing -> NaN)
    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    # GoldDerived = agreement A2/A1 else A3
    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)

    # filtre textes
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    # Aya Expanse uses the Cohere format with system + user
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (bfloat16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode per item (slice the generated part)
        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    # ignore invalid preds + missing labels
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()
    # y_true_col already contains labels 0/1/NaN
    # keep only key cols
    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "AyaExpanse_Pred", "AyaExpanse_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    # error type vs target
    def get_status(row):
        g, p = row["y_true_target"], row["AyaExpanse_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bfloat16...")
    print("       ~64 GB of VRAM for Aya Expanse 32B in BF16")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bfloat16 (half precision, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,  # bfloat16 precision
        device_map="auto",
        trust_remote_code=True
    )

    print(f"       Model loaded. dtype: {model.dtype}")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)
    df["AyaExpanse_Pred"] = preds
    df["AyaExpanse_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    # Print summary + confusion matrices
    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (bfloat16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "AyaExpanse_Pred", "AyaExpanse_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target (3 files)
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# LlaMa 70B 4bits

In [ ]:
# ============================================================
# SCRIPT: Evaluation Llama 3.1 70B (0-shot) — 4-bit quantization
# Uses bitsandbytes for 4-bit quantization (QLoRA-style)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_llama70b_0shot_analysis_4bit")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-70B-Instruct"

HF_TOKEN = os.environ["HF_TOKEN"]

# Reduced batch size for the 70B (even in 4-bit, the model is heavier)
BATCH_SIZE = 2
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    messages = [
        {"role": "system", "content": GRILLE_ANNOTATION},
        {"role": "user", "content": f"Article: {str(row['article_text'])}\nExtrait: {str(row['text'])}\n\nRéponse (oui/non) :"}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Llama 3.1 70B 4-bit) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Llama_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in 4-bit (bitsandbytes)...")

    # bitsandbytes configuration for 4-bit quantization (NF4)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",              # NormalFloat4 - better quality
        bnb_4bit_compute_dtype=torch.bfloat16,  # Calculs in bf16 pour la vitesse
        bnb_4bit_use_double_quant=True,         # Double quantization to save memory
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    # Show GPU memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Llama_Pred"] = preds
    df["Llama_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Llama 3.1 70B 4-bit) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_llama70b_4bit_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Llama_Pred", "Llama_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_llama70b_4bit_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# Saul 7B fp16

In [ ]:
# ============================================================
# SCRIPT: Evaluation Saul 7B Instruct (0-shot) — Metrics + "OOF" saves
# (Here "OOF" = row-by-row predictions over the whole dataset, without folds)
#
# Targets evaluated:
#   - A1
#   - A2
#   - GoldDerived = (A1==A2 ? A1 : eval_A3)
#
# Sauvegardes:
#   - metrics_extended_*.xlsx
#   - OOF_0shot_A1_*.xlsx
#   - OOF_0shot_A2_*.xlsx
#   - OOF_0shot_Gold_*.xlsx
#   - predictions_full_*.xlsx (tout-en-un)
# ============================================================

import os
import re
import warnings
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs_saul_0shot_analysis_bf16")
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Saul-7B-Instruct (legal model based on Mistral)
MODEL_NAME = "Equall/Saul-7B-Instruct-v1"

HF_TOKEN = os.environ["HF_TOKEN"]

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20
MAX_PROMPT_LEN = 4096

GRILLE_ANNOTATION = """Tu es un expert en droit civil français. Ta tâche est de déterminer si un article du Code civil est implicitement appliqué dans un extrait de décision de justice.

IMPORTANT:
- Réponds UNIQUEMENT par "oui" ou "non"
- "oui" = l'article est implicitement appliqué (le raisonnement juridique utilise cet article sans le citer)
- "non" = l'article n'est pas appliqué (simple mention des faits, ou autre régime juridique)

Ne donne aucune explication, juste "oui" ou "non".
""".strip()

# ============================================================
# GPU helpers
# ============================================================
def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# ============================================================
# LABELS + GOLD
# ============================================================
def clean_label(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in ["oui", "1", "yes"]: return 1
    if s in ["non", "0", "no"]: return 0
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return 1
    if re.search(r"\bnon\b", s): return 0
    return np.nan

def load_and_prep_data(path):
    print(f"[1] Loading and building the Gold Standard from {path}...")
    df = pd.read_csv(path)

    for c in ["decision_id", "text", "article_text"]:
        if c not in df.columns:
            raise ValueError(f"Colonne manquante dans l'Excel: {c}")

    for col in ["eval_A2", "eval_A1", "eval_A3"]:
        if col not in df.columns:
            df[col] = np.nan

    df["lbl_A2"] = df["eval_A2"].apply(clean_label)
    df["lbl_A1"] = df["eval_A1"].apply(clean_label)
    df["lbl_A3"] = df["eval_A3"].apply(clean_label)

    def get_gold(row):
        t, a, s = row["lbl_A2"], row["lbl_A1"], row["lbl_A3"]
        if (not pd.isna(t)) and (not pd.isna(a)) and (t == a):
            return t
        if not pd.isna(s):
            return s
        return np.nan

    df["lbl_Gold"] = df.apply(get_gold, axis=1)
    df = df.dropna(subset=["text", "article_text"]).copy().reset_index(drop=True)

    print(f"    Total rows: {len(df)}")
    print(f"    A2 valides: {df['lbl_A2'].notna().sum()}")
    print(f"    A1 valides: {df['lbl_A1'].notna().sum()}")
    print(f"    Gold valides  : {df['lbl_Gold'].notna().sum()}")
    return df

# ============================================================
# PROMPT / PARSE / INFERENCE
# ============================================================
def build_prompt_0shot(row):
    """
    Saul-7B-Instruct (basé sur Mistral) n'accepte PAS les messages "system".
    So the system instructions are folded directly into the user message.
    """
    user_content = f"""{GRILLE_ANNOTATION}

Article: {str(row['article_text'])}
Extrait: {str(row['text'])}

Réponse (oui/non) :"""

    messages = [
        {"role": "user", "content": user_content}
    ]
    return messages

def parse_response(response_text):
    text = str(response_text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)

    if text.startswith("oui") or text == "oui": return 1
    if text.startswith("non") or text == "non": return 0

    words = text.split()
    if "oui" in words and "non" not in words: return 1
    if "non" in words and "oui" not in words: return 0
    return -1

def run_inference(df, model, tokenizer):
    print(f"[2] 0-shot inference (Saul-7B-Instruct bf16) on {len(df)} rows...")
    all_preds, all_raw = [], []

    prompts_formatted = [
        tokenizer.apply_chat_template(build_prompt_0shot(row), tokenize=False, add_generation_prompt=True)
        for _, row in df.iterrows()
    ]

    for i in tqdm(range(0, len(prompts_formatted), BATCH_SIZE), desc="Generation"):
        batch_texts = prompts_formatted[i:i+BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prompt_len = inputs["input_ids"].shape[1]
        for j in range(outputs.shape[0]):
            response = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True)
            all_raw.append(response)
            all_preds.append(parse_response(response))

        _clear_cuda()

    return np.array(all_preds, dtype=int), all_raw

# ============================================================
# METRICS + CONFUSION MATRIX
# ============================================================
def print_confusion(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    print("Confusion matrix (rows=true, cols=pred) [0=non, 1=oui]")
    print(f"          pred_non   pred_oui")
    print(f"true_non    {tn:6d}   {fp:6d}")
    print(f"true_oui    {fn:6d}   {tp:6d}")

def calculate_metrics(y_true, y_pred, name=""):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=int)

    mask = (~np.isnan(y_true)) & (y_pred != -1)
    yt = y_true[mask].astype(int)
    yp = y_pred[mask].astype(int)

    out = {
        "Target": name,
        "N_Valid": int(len(yt)),
        "N_InvalidPred": int(np.sum(y_pred == -1)),
    }

    if len(yt) == 0:
        return out, np.array([[0, 0], [0, 0]])

    acc = accuracy_score(yt, yp)
    mcc = matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else 0.0
    prec_oui = precision_score(yt, yp, pos_label=1, zero_division=0)
    rec_oui = recall_score(yt, yp, pos_label=1, zero_division=0)
    f1_oui = f1_score(yt, yp, pos_label=1, zero_division=0)

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    out.update({
        "Accuracy_Global": round(acc, 4),
        "Precision_Oui": round(prec_oui, 4),
        "Recall_Oui": round(rec_oui, 4),
        "F1_Oui": round(f1_oui, 4),
        "FPR_pct": round(100.0 * fpr, 2),
        "MCC": round(float(mcc), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Recup_vs_Positifs": f"{tp}/{tp+fn}" if (tp+fn) > 0 else "0/0"
    })

    return out, cm

# ============================================================
# SAVE "OOF" PER TARGET
# ============================================================
def save_target_oof(df, y_true_col, target_name, timestamp):
    out_df = df.copy()

    cols_order = [
        "decision_id", "chunk_id", "pred_art",
        "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Saul_Pred", "Saul_Raw"
    ]
    out_df = out_df[[c for c in cols_order if c in out_df.columns]].copy()
    out_df["Target"] = target_name
    out_df["y_true_target"] = df[y_true_col]

    def get_status(row):
        g, p = row["y_true_target"], row["Saul_Pred"]
        if pd.isna(g) or p == -1: return "Ignore"
        if g == 1 and p == 1: return "TP"
        if g == 0 and p == 1: return "FP"
        if g == 1 and p == 0: return "FN"
        if g == 0 and p == 0: return "TN"
        return "Error"

    out_df["Error_Type_vs_Target"] = out_df.apply(get_status, axis=1)

    path = os.path.join(OUTPUT_PATH, f"OOF_0shot_{target_name}_{timestamp}.xlsx")
    out_df.to_excel(path, index=False)
    return path

# ============================================================
# MAIN
# ============================================================
def main():
    df = load_and_prep_data(EXCEL_PATH)

    print(f"[Init] Loading model {MODEL_NAME} in bf16...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chargement in bf16
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()

    # 0-shot inference
    preds, raw_texts = run_inference(df, model, tokenizer)

    df["Saul_Pred"] = preds
    df["Saul_Raw"] = raw_texts

    # Evaluation on each target
    print("\n[3] Computing metrics (A2 / A1 / Gold)...")
    results = []
    cms = {}

    r_t, cm_t = calculate_metrics(df["lbl_A2"].values, preds, name="A2")
    results.append(r_t); cms["A2"] = cm_t

    r_a, cm_a = calculate_metrics(df["lbl_A1"].values, preds, name="A1")
    results.append(r_a); cms["A1"] = cm_a

    r_g, cm_g = calculate_metrics(df["lbl_Gold"].values, preds, name="GOLD_STANDARD")
    results.append(r_g); cms["GOLD_STANDARD"] = cm_g

    metrics_df = pd.DataFrame(results)

    cols_view = ["Target", "N_Valid", "Precision_Oui", "Recall_Oui", "FPR_pct", "MCC", "Recup_vs_Positifs"]
    print("\n=== RESULTS (Saul-7B-Instruct bf16) ===")
    print(metrics_df[cols_view])

    for name in ["A2", "A1", "GOLD_STANDARD"]:
        print(f"\n--- Matrice de confusion: {name} ---")
        print_confusion(cms[name])

    # Save metrics
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    metrics_path = os.path.join(OUTPUT_PATH, f"metrics_extended_saul_bf16_{timestamp}.xlsx")
    metrics_df.to_excel(metrics_path, index=False)

    # Save full predictions
    cols_full = [
        "decision_id", "chunk_id", "pred_art", "text", "article_text",
        "eval_A2", "eval_A1", "eval_A3",
        "lbl_A2", "lbl_A1", "lbl_Gold",
        "Saul_Pred", "Saul_Raw"
    ]
    final_df = df[[c for c in cols_full if c in df.columns]].copy()
    preds_path = os.path.join(OUTPUT_PATH, f"predictions_full_saul_bf16_{timestamp}.xlsx")
    final_df.to_excel(preds_path, index=False)

    # Save "OOF" per target
    oof_t_path = save_target_oof(df, "lbl_A2", "A2", timestamp)
    oof_a_path = save_target_oof(df, "lbl_A1", "A1", timestamp)
    oof_g_path = save_target_oof(df, "lbl_Gold", "Gold", timestamp)

    print("\n[Fini] Sauvegardes:")
    print(f"- Metrics: {metrics_path}")
    print(f"- Predictions full: {preds_path}")
    print(f"- OOF A2: {oof_t_path}")
    print(f"- OOF A1: {oof_a_path}")
    print(f"- OOF Gold: {oof_g_path}")
    print(f"\nDossier: {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

# bitsandbytes

In [ ]:
import bitsandbytes as bnb
print(f"Installed version: {bnb.__version__}")

# SetFit

In [ ]:
# ============================================================
# SETFIT PIPELINE (Contrastive Learning + Classification Head)
# ============================================================

import os, re, warnings, gc
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from datasets import Dataset
from setfit import SetFitModel, SetFitTrainer
from sentence_transformers.losses import CosineSimilarityLoss
warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

# ============================================================
# FLAG: choose the annotator for the gold label
# Options: "A1" or "A2"
# ============================================================
GOLD_ANNOTATOR = "A1"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, f"outputs_setfit_gold_{GOLD_ANNOTATOR}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Recommended model for French with SetFit
SETFIT_MODEL_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# SetFit hyperparameters
BATCH_SIZE = 16
NUM_EPOCHS = 1
NUM_ITERATIONS = 20  # Number of pairs generated per sentence (the key to performance!)

THR_GRID = np.linspace(0.05, 0.95, 181)

# -------------------------
# Helpers
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -------------------------
# Text configs (same logic as the reference script)
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " ; " + chunk).tolist() # [SEP] is not needed for SetFit; a natural separator is enough
    if cfg_id == "cfg2":
        return ("Article: " + art + " Extrait: " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("Article " + pred_art + ": " + art + " Extrait: " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD DATA
# ============================================================
print("="*80)
print("LOADING DATA")
print(f"GOLD ANNOTATOR: {GOLD_ANNOTATOR}")
print("="*80)

df0 = pd.read_csv(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)

if GOLD_ANNOTATOR == "A1":
    df0["label_str"] = df0["a"]
elif GOLD_ANNOTATOR == "A2":
    df0["label_str"] = df0["t"]
else:
    raise ValueError("GOLD_ANNOTATOR inconnu")

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

# Build the folds
df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# TRAIN LOOP (SetFit Training per Fold)
# ============================================================
rows = []
best_global = None

# Iterate over the text configs (as before)
for cfg_id in ["cfg1", "cfg2", "cfg3"]:

    # Prepare the text for the whole column
    df_cv["text_input"] = make_text_inputs(df_cv, cfg_id)

    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    print("\n" + "="*60)
    print(f"Running SetFit | Config: {cfg_id}")
    print("="*60)

    # Cross-Validation Loop
    for fold in range(N_SPLITS):
        print(f"  > Fold {fold+1}/{N_SPLITS}...")

        # 1. Split Data
        train_df = df_cv[df_cv["fold"] != fold][["text_input", "label"]]
        test_df = df_cv[df_cv["fold"] == fold][["text_input", "label"]]

        # Convert to HuggingFace Dataset
        # SetFit expects "text" and "label" columns
        ds_train = Dataset.from_pandas(train_df.rename(columns={"text_input": "text"}))
        # ds_test = Dataset.from_pandas(test_df.rename(columns={"text_input": "text"}))

        # 2. Init Model (Important: Reload from scratch at each fold to avoid data leakage)
        model = SetFitModel.from_pretrained(SETFIT_MODEL_ID)

        # 3. Trainer
        trainer = SetFitTrainer(
            model=model,
            train_dataset=ds_train,
            loss_class=CosineSimilarityLoss, # <--- LA CORRECTION EST ICI (au lieu de None)
            metric="accuracy",
            batch_size=BATCH_SIZE,
            num_iterations=NUM_ITERATIONS,
            num_epochs=NUM_EPOCHS,
            column_mapping={"text": "text", "label": "label"}
        )
        # 4. Train
        trainer.train()

        # 5. Predict Probabilities on Test Fold
        # SetFit predict_proba renvoie [prob_0, prob_1]
        test_texts = test_df["text_input"].tolist()
        preds_proba = model.predict_proba(test_texts)[:, 1] # On prend la proba de la classe 1

        # Fill OOF
        test_indices = df_cv[df_cv["fold"] == fold].index
        proba_oof[test_indices] = preds_proba

        # Cleanup
        del model, trainer, ds_train
        _clear_cuda()

    # --- Metrics calculation for this Config ---
    # threshold 0.5
    pred_05 = (proba_oof >= 0.5).astype(int)
    met_05 = compute_metrics_from_pred(y_true, pred_05)

    # best MCC threshold
    best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
    pred_best = (proba_oof >= best_thr).astype(int)
    met_best = compute_metrics_from_pred(y_true, pred_best)

    row = {
        "Model": "SetFit",
        "Base_Model": SETFIT_MODEL_ID,
        "Gold_Annotator": GOLD_ANNOTATOR,
        "TextCfg": cfg_id,
        "Num_Iterations": NUM_ITERATIONS,

        "Thr(best)": best_thr,
        "MCC(best)": met_best["MCC"],
        "Acc(best)": met_best["Accuracy"],
        "BAcc(best)": met_best["Balanced Acc"],
        "BF1(best)": met_best["Balanced F1"],

        "MCC@0.5": met_05["MCC"],
        "Acc@0.5": met_05["Accuracy"],
        "BAcc@0.5": met_05["Balanced Acc"],
        "BF1@0.5": met_05["Balanced F1"],

        "ΔMCC": met_best["MCC"] - met_05["MCC"],
        "proba_oof_path": None
    }

    # Save OOF
    run_id = f"setfit_{cfg_id}"
    proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id}.npy")
    np.save(proba_path, proba_oof)
    row["proba_oof_path"] = proba_path

    rows.append(row)

    print(f"Result {cfg_id}: MCC(best)={met_best['MCC']:.4f} at thr={best_thr}")

# ============================================================
# SAVE RESULTS
# ============================================================
df_res = pd.DataFrame(rows).sort_values("MCC(best)", ascending=False)

out_file = os.path.join(OUTPUT_PATH, f"SETFIT_RESULTS_gold_{GOLD_ANNOTATOR}.xlsx")
df_res.to_excel(out_file, index=False)

print("\n" + "="*80)
print(f"✅ DONE. Results saved to: {out_file}")
print("Top Config:")
print(df_res.iloc[0][["TextCfg", "MCC(best)", "BAcc(best)"]])
print("="*80)

In [ ]:
# ============================================================
# SAVE OOF (per cfg) — put this at the end of each cfg loop
# ============================================================

run_id = f"setfit_{cfg_id}_gold_{GOLD_ANNOTATOR}"

# 1) Save raw OOF probs (fast reload)
proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id}.npy")
np.save(proba_path, proba_oof)

# 2) Save a tabular OOF file (easy debug / analysis)
oof_df = df_cv[["decision_id", "chunk_id", "fold", "label"]].copy()
oof_df["text_cfg"] = cfg_id
oof_df["model"] = "SetFit"
oof_df["base_model"] = SETFIT_MODEL_ID
oof_df["gold_annotator"] = GOLD_ANNOTATOR
oof_df["num_iterations"] = NUM_ITERATIONS
oof_df["proba_oui"] = proba_oof

# optional: add hard preds with your best threshold
oof_df["pred_thr05"] = (oof_df["proba_oui"] >= 0.5).astype(int)
oof_df["pred_best"]  = (oof_df["proba_oui"] >= best_thr).astype(int)
oof_df["thr_best"]   = float(best_thr)

csv_path = os.path.join(OUTPUT_PATH, f"oof_{run_id}.csv")
oof_df.to_csv(csv_path, index=False)

xlsx_path = os.path.join(OUTPUT_PATH, f"oof_{run_id}.xlsx")
oof_df.to_excel(xlsx_path, index=False)

print(f"[OOF saved] {cfg_id}")
print(f"  - {proba_path}")
print(f"  - {csv_path}")
print(f"  - {xlsx_path}")

# and keep it in your metrics row if you want
row["proba_oof_path"] = proba_path
row["oof_csv_path"] = csv_path
row["oof_xlsx_path"] = xlsx_path
